# Part 1 Risk Scoring

### Pipeline to parse all drugs under ATC group N06

In [ ]:
"""
Pipeline to parse all drugs under ATC group N06A (Antidepressants)
from the WHO ATC/DDD Index (https://atcddd.fhi.no/atc_ddd_index/).

Strategy:
    1. Fetch the N06A index page.
    2. Extract the child subgroup codes
       (N06AA, N06AB, N06AF, N06AG, N06AX).
    3. Visit each subgroup page and parse the drug-level rows
       (ATC code + name, and where available DDD / unit / route / note).
    4. Return a flat list of drug dicts.
    5. Save the results to a CSV file.

Notes:
    - N06AA covers non-selective monoamine reuptake inhibitors (e.g. the
      classic tricyclics such as amitriptyline and imipramine).
    - N06AB covers the SSRIs (e.g. fluoxetine, sertraline, citalopram).
    - N06AF and N06AG cover the monoamine oxidase inhibitors
      (non-selective, and MAO-A selective, respectively).
    - Lithium is NOT in N06A; it is classified under N05AN, so it will
      not appear in these results.
    - Esketamine for nasal administration lives in N06AX ("Other
      antidepressants"), while esketamine injections are in N01AX and
      will not appear here.
"""

import csv
import time
import requests
from bs4 import BeautifulSoup

BASE_URL = "https://atcddd.fhi.no/atc_ddd_index/"
HEADERS = {"User-Agent": "Mozilla/5.0 (ATC-DDD-parser; research use)"}

# Column order used for the CSV output
CSV_FIELDS = ["atc_code", "name", "ddd", "unit", "route", "note", "subgroup"]


# ----------------------------------------------------------------------
# Low-level fetch
# ----------------------------------------------------------------------
def fetch(code: str, session: requests.Session) -> BeautifulSoup:
    """Fetch a single ATC code page and return parsed soup."""
    resp = session.get(BASE_URL, params={"code": code}, headers=HEADERS, timeout=30)
    resp.raise_for_status()
    return BeautifulSoup(resp.text, "html.parser")


# ----------------------------------------------------------------------
# Discover subgroup codes under a parent code
# ----------------------------------------------------------------------
def get_child_codes(soup: BeautifulSoup, parent_code: str) -> list[str]:
    """
    Extract immediate child ATC codes from a page.
    Children appear as links of the form '?code=N06AA'.
    """
    codes = []
    for a in soup.select("a[href*='code=']"):
        href = a["href"]
        code = href.split("code=")[-1].split("&")[0].strip()
        # keep codes that extend the parent (e.g. N06AA under N06A)
        if (
            code.startswith(parent_code)
            and code != parent_code
            and len(code) == len(parent_code) + 1   # one level deeper
            and code not in codes
        ):
            codes.append(code)
    return codes


# ----------------------------------------------------------------------
# Parse drug-level rows from a subgroup page
# ----------------------------------------------------------------------
def parse_drugs(soup: BeautifulSoup, subgroup_code: str) -> list[dict]:
    """
    On a subgroup (chemical subgroup) page the substances are listed in a
    table whose rows look like:
        ATC code | Name | DDD | Unit | Adm.R | Note
    The 7-character codes (e.g. N06AB03) identify chemical substances.
    """
    drugs = []
    for row in soup.select("table tr"):
        cells = [c.get_text(strip=True) for c in row.find_all("td")]
        if not cells:
            continue

        atc_code = cells[0]
        # A substance-level code is the subgroup code + 2 digits, e.g. N06AB03
        if not (atc_code.startswith(subgroup_code) and len(atc_code) == len(subgroup_code) + 2):
            continue

        drug = {
            "atc_code": atc_code,
            "name":     cells[1] if len(cells) > 1 else None,
            "ddd":      cells[2] if len(cells) > 2 and cells[2] else None,
            "unit":     cells[3] if len(cells) > 3 and cells[3] else None,
            "route":    cells[4] if len(cells) > 4 and cells[4] else None,
            "note":     cells[5] if len(cells) > 5 and cells[5] else None,
            "subgroup": subgroup_code,
        }
        # Some substances span multiple DDD/route rows; the name cell is empty
        # on continuation rows, so carry the previous name forward.
        if not drug["name"] and drugs:
            drug["name"] = drugs[-1]["name"]
        drugs.append(drug)
    return drugs


# ----------------------------------------------------------------------
# Orchestration
# ----------------------------------------------------------------------
def scrape_atc(parent_code: str = "N06A", polite_delay: float = 0.5) -> list[dict]:
    with requests.Session() as session:
        root = fetch(parent_code, session)
        subgroups = get_child_codes(root, parent_code)

        all_drugs = []
        for sub in subgroups:
            sub_soup = fetch(sub, session)
            all_drugs.extend(parse_drugs(sub_soup, sub))
            time.sleep(polite_delay)   # be courteous to the server
        return all_drugs


# ----------------------------------------------------------------------
# CSV export
# ----------------------------------------------------------------------
def save_to_csv(drugs: list[dict], filename: str = "atc_n06a_drugs.csv") -> None:
    """
    Write the parsed drug entries to a CSV file.

    Each dict in `drugs` is written as one row using the column order in
    CSV_FIELDS. Missing values (None) are written as empty cells.
    """
    with open(filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=CSV_FIELDS, extrasaction="ignore")
        writer.writeheader()
        for d in drugs:
            # Replace None with "" so empty cells are clean in the CSV
            writer.writerow({k: (d.get(k) if d.get(k) is not None else "") for k in CSV_FIELDS})

    print(f"Saved {len(drugs)} rows to '{filename}'")


# ----------------------------------------------------------------------
if __name__ == "__main__":
    drugs = scrape_atc("N06A")
    print(f"Found {len(drugs)} drug entries under N06A (Antidepressants)\n")
    for d in drugs:
        ddd = f"{d['ddd']} {d['unit']} ({d['route']})" if d["ddd"] else "no DDD"
        print(f"{d['atc_code']:<9} {d['name']:<30} {ddd}")

    # If you only want the plain list of drug names:
    names = sorted({d["name"] for d in drugs if d["name"]})
    print("\nUnique drug names:")
    print(names)

    # Save the full result set to CSV
    save_to_csv(drugs, "atc_n06a_drugs.csv")

Found 68 drug entries under N06A (Antidepressants)

N06AA01   desipramine                    0.1 g (O)
N06AA02   imipramine                     0.1 g (O)
N06AA03   imipramine oxide               0.1 g (O)
N06AA04   clomipramine                   0.1 g (O)
N06AA05   opipramol                      0.15 g (O)
N06AA06   trimipramine                   0.15 g (O)
N06AA07   lofepramine                    0.105 g (O)
N06AA08   dibenzepin                     0.3 g (O)
N06AA09   amitriptyline                  75 mg (O)
N06AA10   nortriptyline                  75 mg (O)
N06AA11   protriptyline                  30 mg (O)
N06AA12   doxepin                        0.1 g (O)
N06AA13   iprindole                      90 mg (O)
N06AA14   melitracen                     75 mg (O)
N06AA15   butriptyline                   75 mg (O)
N06AA16   dosulepin                      0.15 g (O)
N06AA17   amoxapine                      0.15 g (O)
N06AA18   dimetacrine                    0.15 g (O)
N06AA19   amineptine   

In [ ]:
!cp "/content/atc_n06a_drugs.csv" "/content/drive/MyDrive/Dr Uccello/00_Studies/Z_database/atc_n06a_drugs.csv"

### Ki Db

In [ ]:
!cp "/content/drive/MyDrive/Dr Uccello/00_Studies/Z_database" /content/ -r

### Loading twas

In [ ]:
!cp "/content/drive/MyDrive/Dr Uccello/00_Studies/Z_proximity/proximity_cvs/zhou" /content/ -r

In [ ]:
!rm "/content/zhou/zhou_KSD" -r

In [ ]:
!pip install rapidfuzz tqdm openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 33.6 MB/s eta 0:00:00


# The pipeline

In [ ]:
# =============================================================================
# 080KvD-v3  —  N06A ANTIDEPRESSANTS : Ki -> DDD -> TWAS  (JvH "logp_expo" method)
# -----------------------------------------------------------------------------
# WHAT CHANGED vs the original 080KvD (V2):
#   * TWAS is now injected PER RECEPTOR, BEFORE aggregation:
#         twas_scale = exp(LOGP_BETA * -log10(p)),  -log10(p) capped at MAX_NEGLOG10_P
#         contribution = log1p(affinity_dose_score) * receptor_weight * twas_scale
#     (instead of a drug-level 1 + factor*max|z| boost applied after summing)
#   * Fairness imputation for missing metabolic-gene p-values uses mean -log10(p)
#   * Tiny optional secondary drug-level boost  (1 + 0.02 * max|z|)
#   * Reference drug stays amitriptyline = 100
#   * DUAL RUN: identical pipeline with (a) literature METABOLIC_WEIGHTS
#                                       (b) uniform weights = 1.0
#   * Post-hoc chronic 5-HT adaptation (SERT-high drugs) re-scored with the SAME
#     logp_expo machinery  -> a third "method" for the comparative analyses
#   * Full Stage-3 transdiagnostic synthesis (S1..S10) + MASTER_SUMMARY.txt
# Run as ONE cell.
# =============================================================================

!pip -q install rapidfuzz tqdm openpyxl beautifulsoup4 requests 2>/dev/null

import os, re, csv, glob, json, time, math, textwrap, datetime, itertools, warnings, shutil
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

from rapidfuzz import process, fuzz
from tqdm import tqdm

try:
    from scipy.stats import (spearmanr, kendalltau, wilcoxon, mannwhitneyu,
                             hypergeom, chi2)
    from scipy.cluster.hierarchy import linkage, dendrogram, fcluster, cophenet
    from scipy.spatial.distance import squareform
    HAVE_SCIPY = True
except Exception as _e:
    HAVE_SCIPY = False
    print(f"[warn] scipy unavailable ({_e}) — inferential statistics limited.")

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    HAVE_MPL = True
except Exception as _e:
    HAVE_MPL = False
    print(f"[warn] matplotlib unavailable ({_e}) — figures skipped.")


# =============================================================================
# CONFIG  ——  EDIT PATHS HERE
# =============================================================================
KI_DB_PATH     = "/content/Z_database/KiDatabase_2026-06-22.csv"
DRUGS_CSV_PATH = "/content/Z_database/atc_n06a_drugs.csv"   # auto-scraped if absent

TWAS_DIRS = [
    "/content/zhou/zhou_CKD",
    "/content/zhou/zhou_ESSHP",
    "/content/zhou/zhou_T2D",
    "/content/zhou/zhou_obesity",
]

OUTPUT_ROOT   = "/content/pipeline_output/n06a_logp_expo"
STAGE3_DIR    = os.path.join(OUTPUT_ROOT, "stage3_dual")
MASTER_TXT    = os.path.join(OUTPUT_ROOT, "MASTER_SUMMARY.txt")

COPY_TO_DRIVE = False
DRIVE_DEST    = "/content/drive/MyDrive/Dr Uccello/00_Studies/080_GHS_CVS"

# ---- matching / QC ----------------------------------------------------------
FUZZY_THRESHOLD      = 80
KI_AGGREGATION       = "min"        # min | median | mean
STRONG_BINDER_KI_NM  = 10.0
TWAS_P_SIGNIFICANT   = 0.05
KI_IMPUTE_STRATEGY   = "mean"       # none | mean | median
TWAS_IMPUTE_STRATEGY = "mean"       # none | mean | median
CSV_ENCODINGS        = ["utf-8", "utf-8-sig", "latin-1", "cp1252"]

# ---- JvH-style TWAS scaling (mild exponential on -log10 p) ------------------
LOGP_BETA           = 0.045    # tune 0.03 - 0.06
MAX_NEGLOG10_P      = 25.0     # cap so ultra-small p cannot explode the score
SECONDARY_BOOST_FAC = 0.02     # tiny secondary drug-level boost on max|z|
TWAS_SIG_BONUS      = 0.10     # set 0.0 for pure logp scaling
REFERENCE_DRUG      = "amitriptyline"
TWAS_SCALING        = "logp_expo"   # logp_expo | logp_linear | z_expo | z_linear | none

# ---- post-hoc chronic 5-HT adaptation (antidepressant-specific) -------------
RUN_5HT_ADAPTATION = True
SERT_THRESHOLD  = 0.1      # affinity_dose_score on SLC6A4 above which adaptation fires
HTR2C_INCREASE  = 0.18
HTR2A_INCREASE  = 0.12
HTR1A_REDUCTION = 0.05
HTR1A_FLOOR     = 0.01

# ---- Stage-3 settings -------------------------------------------------------
TOP_K        = 10
ALPHA        = 0.05
RANDOM_SEED  = 42
KEY_MOVERS_N = 12
np.random.seed(RANDOM_SEED)

SSRI_SNRI_REGEX = (
    "sertraline|fluoxetine|paroxetine|citalopram|escitalopram|fluvoxamine|"
    "venlafaxine|desvenlafaxine|duloxetine|milnacipran|levomilnacipran|"
    "clomipramine|vortioxetine|vilazodone|imipramine"
)

# ---- External ORDINAL literature weight-gain / metabolic liability ----------
# 1 = highest liability. Orthogonal sanity reference ONLY; not used by the model.
# Compiled from Serretti & Mandelli 2010, Gafoor 2018, Salvi 2016, Cipriani 2018.
# EDIT FREELY — incomplete lists are fine (unmatched drugs are simply skipped).
ENABLE_EXTERNAL_VALIDATION = True
LITERATURE_WEIGHT_GAIN_RANK = {
    "amitriptyline": 1, "mirtazapine": 2, "doxepin": 3, "imipramine": 4,
    "trimipramine": 5, "nortriptyline": 6, "clomipramine": 7, "paroxetine": 8,
    "mianserin": 9, "amoxapine": 10, "maprotiline": 11, "dosulepin": 5,
    "desipramine": 9, "phenelzine": 8, "tranylcypromine": 14, "isocarboxazid": 12,
    "citalopram": 12, "escitalopram": 13, "sertraline": 14, "fluvoxamine": 15,
    "venlafaxine": 16, "duloxetine": 17, "milnacipran": 18, "vortioxetine": 18,
    "fluoxetine": 19, "bupropion": 21, "amfebutamone": 21, "agomelatine": 20,
    "reboxetine": 22, "moclobemide": 20, "trazodone": 11, "nefazodone": 15,
    "tianeptine": 16, "vilazodone": 17, "lofepramine": 8, "opipramol": 7,
    "protriptyline": 10, "melitracen": 6,
}

os.makedirs(OUTPUT_ROOT, exist_ok=True)
os.makedirs(STAGE3_DIR, exist_ok=True)
S3_TABLES = os.path.join(STAGE3_DIR, "tables"); os.makedirs(S3_TABLES, exist_ok=True)
S3_FIGS   = os.path.join(STAGE3_DIR, "figures"); os.makedirs(S3_FIGS, exist_ok=True)


# =============================================================================
# Receptor (Ki DB label) -> official HGNC gene symbol
# =============================================================================
RECEPTOR_TO_GENE = {
    "5-HT1A": "HTR1A", "5HT1A": "HTR1A", "5-HT1B": "HTR1B", "5HT1B": "HTR1B",
    "5-HT1D": "HTR1D", "5-HT1E": "HTR1E", "5-HT1F": "HTR1F",
    "5-HT2A": "HTR2A", "5HT2A": "HTR2A", "5-HT2B": "HTR2B",
    "5-HT2C": "HTR2C", "5HT2C": "HTR2C", "5-HT3": "HTR3A",
    "5-HT5A": "HTR5A", "5-HT6": "HTR6", "5-HT7": "HTR7",
    "D1": "DRD1", "D2": "DRD2", "D3": "DRD3", "D4": "DRD4", "D5": "DRD5",
    "DRD1": "DRD1", "DRD2": "DRD2", "DRD3": "DRD3", "DRD4": "DRD4", "DRD5": "DRD5",
    "alpha1A": "ADRA1A", "alpha1B": "ADRA1B", "alpha1D": "ADRA1D",
    "alpha2A": "ADRA2A", "alpha2B": "ADRA2B", "alpha2C": "ADRA2C",
    "alpha1": "ADRA1A", "alpha2": "ADRA2A",
    "beta1": "ADRB1", "beta2": "ADRB2", "beta3": "ADRB3",
    "H1": "HRH1", "H2": "HRH2", "H3": "HRH3", "H4": "HRH4",
    "M1": "CHRM1", "M2": "CHRM2", "M3": "CHRM3", "M4": "CHRM4", "M5": "CHRM5",
    "Muscarinic": "CHRM1", "Sigma1": "SIGMAR1", "Sigma 1": "SIGMAR1",
    "SERT": "SLC6A4", "DAT": "SLC6A3", "NET": "SLC6A2",
}
RECEPTOR_TO_GENE = {k.strip().lower(): v for k, v in RECEPTOR_TO_GENE.items()}

# =============================================================================
# ANTIDEPRESSANT METABOLIC WEIGHTS  (unchanged from 080KvD V2 — see that file
# for the full per-receptor evidence annotations: Kroeze 2003, Salvi 2016,
# Kim 2007, Johnson 2005, Tecott 1995, Gray & Roth 2001/2011, Albert 2011.)
# =============================================================================
METABOLIC_WEIGHTS = {
    "HRH1": 1.00, "HTR2C": 0.72, "CHRM3": 0.68,
    "HTR2A": 0.45, "ADRA1A": 0.42, "ADRA1B": 0.40, "HTR6": 0.38,
    "ADRA2A": 0.28, "ADRA2B": 0.26, "ADRA2C": 0.26,
    "CHRM1": 0.25, "CHRM4": 0.20, "CHRM5": 0.18,
    "HTR7": 0.22, "DRD3": 0.15,
    "DRD2": 0.10, "DRD4": 0.08, "HTR1A": 0.07, "ADRB1": 0.07, "ADRB2": 0.06,
    "SIGMAR1": 0.04,
    "SLC6A4": 0.08, "SLC6A2": 0.05, "SLC6A3": 0.04,
}

WEIGHTING_SCHEMES = {                      # <-- the dual run
    "weighted": dict(METABOLIC_WEIGHTS),   # literature prior  (clinical proxy)
    "uniform":  None,                      # None => uniform 1.0 for every gene
}
KEY_RECEPTORS = ["HRH1", "HTR2C", "CHRM3", "HTR2A", "ADRA1A", "HTR6",
                 "SLC6A4", "SLC6A2", "SLC6A3"]

TWAS_SYMBOL_COL_CANDIDATES = ["gene_name","genename","gene_symbol","symbol","hgnc_symbol"]
TWAS_ENSG_COL_CANDIDATES   = ["gene","ensembl_gene_id","gene_id","geneid","ensembl","id"]
TWAS_Z_COL_CANDIDATES      = ["twas.z","twas_z","zscore","z_score","zstat","z"]
TWAS_P_COL_CANDIDATES      = ["twas.p","twas_p","pvalue","p_value","pval","p"]


# =============================================================================
# GENERIC HELPERS
# =============================================================================
SUMMARY, SIG, FILES_OUT = [], [], []
BAR, SUB = "=" * 100, "-" * 100
def S(x=""):   SUMMARY.append(str(x))
def SEC(t):    S(""); S(BAR); S(str(t).upper()); S(BAR)
def SUBSEC(t): S(""); S(SUB); S(str(t)); S(SUB)

def stars(p):
    if p is None or (isinstance(p, float) and np.isnan(p)): return "n/a"
    return "****" if p < 1e-4 else "***" if p < 1e-3 else "**" if p < 1e-2 else "*" if p < .05 else "ns"

def fmt(x, nd=3):
    try:
        if x is None or (isinstance(x, float) and np.isnan(x)): return "NA"
        if isinstance(x, (bool, np.bool_)): return str(bool(x))
        if isinstance(x, (int, np.integer)): return f"{int(x):,}"
        ax = abs(float(x))
        if ax != 0 and (ax < 1e-3 or ax >= 1e6): return f"{float(x):.3e}"
        return f"{float(x):.{nd}f}"
    except Exception:
        return str(x)

def save_table(df, name, note="", root=S3_TABLES):
    p = os.path.join(root, name); df.to_csv(p, index=False)
    FILES_OUT.append((p, note)); return p

def save_fig(fig, name, note=""):
    if not HAVE_MPL: return None
    p = os.path.join(S3_FIGS, name); fig.savefig(p, dpi=150, bbox_inches="tight")
    plt.close(fig); FILES_OUT.append((p, note)); return p

def _is_nan(x):
    if x is None: return True
    try: return bool(np.isnan(x))
    except (TypeError, ValueError): return False

def read_csv_robust(path, **kw):
    kw.setdefault("low_memory", False); last = None
    for enc in CSV_ENCODINGS:
        try: return pd.read_csv(path, encoding=enc, **kw)
        except UnicodeDecodeError as e: last = e; continue
    try:
        return pd.read_csv(path, encoding="latin-1", encoding_errors="replace", **kw)
    except Exception:
        raise last if last else RuntimeError(f"Could not read {path}")

def read_csv_safe(path):
    if not path or not os.path.exists(path): return None
    for enc in CSV_ENCODINGS:
        try: return pd.read_csv(path, encoding=enc, low_memory=False)
        except Exception: continue
    return None

def normalise_name(s):
    if not isinstance(s, str): return ""
    return re.sub(r"\s+", " ", s.strip().lower())
norm_drug = normalise_name

def receptor_to_gene(receptor):
    if not isinstance(receptor, str): return None
    k = receptor.strip().lower()
    if k in RECEPTOR_TO_GENE: return RECEPTOR_TO_GENE[k]
    k2 = re.sub(r"[\s\-]", "", k)
    for kk, v in RECEPTOR_TO_GENE.items():
        if re.sub(r"[\s\-]", "", kk) == k2: return v
    return None

def parse_num(value):
    if value is None: return np.nan
    if isinstance(value, (int, float)) and not isinstance(value, bool): return float(value)
    s = str(value).strip()
    if s == "" or s.lower() in {"na","nan","nd","n/a","-"}: return np.nan
    s = s.replace(",", ""); s = re.sub(r"^[><=~]+", "", s)
    m = re.search(r"-?\d+\.?\d*(?:[eE][-+]?\d+)?", s)
    return float(m.group()) if m else np.nan
parse_ki = parse_num

def ddd_to_mg(ddd, unit):
    v = parse_num(ddd)
    if _is_nan(v): return np.nan
    u = (str(unit).strip().lower() if unit is not None else "")
    return v * {"g":1000.0,"mg":1.0,"mcg":0.001,"µg":0.001,"ug":0.001}.get(u, 1.0)

def aggregate_ki(values, how):
    v = values.dropna()
    if v.empty: return np.nan
    return float({"min": v.min(), "median": v.median(), "mean": v.mean()}.get(how, v.min()))

def find_col(df, cands):
    lm = {str(c).strip().lower(): c for c in df.columns}
    for c in cands:
        if c.lower() in lm: return lm[c.lower()]
    return None

def _strip_ensg_version(s): return re.sub(r"\.\d+$", "", str(s).strip().upper())
def _safe_label(p):
    b = os.path.basename(os.path.normpath(p)) or re.sub(r"[^A-Za-z0-9_.-]+","_",p)
    return re.sub(r"[^A-Za-z0-9_.-]+","_",b).strip("_") or "twas"

def bh_fdr(pvals):
    p = np.asarray(pvals, float); ok = ~np.isnan(p)
    q = np.full_like(p, np.nan, float); pv = p[ok]; n = pv.size
    if n == 0: return q
    o = np.argsort(pv); r = pv[o]
    adj = np.clip(np.minimum.accumulate((r * n / (np.arange(n)+1))[::-1])[::-1], 0, 1)
    out = np.empty(n); out[o] = adj; q[ok] = out
    return q

def kendalls_w(R):
    R = np.asarray(R, float); n, m = R.shape
    if n < 3 or m < 2: return (np.nan,)*4
    Ri = R.sum(axis=1); Sd = ((Ri - Ri.mean())**2).sum()
    W = 12*Sd/(m**2*(n**3-n)); chi = m*(n-1)*W; df = n-1
    p = float(1-chi2.cdf(chi, df)) if HAVE_SCIPY else np.nan
    return float(W), float(chi), int(df), p

def binom_p(k, n, p=0.5):
    if n == 0: return np.nan
    try:
        from scipy.stats import binomtest; return float(binomtest(int(k), int(n), p).pvalue)
    except Exception:
        try:
            from scipy.stats import binom_test; return float(binom_test(int(k), int(n), p))
        except Exception: return np.nan

def zscore(s):
    s = pd.Series(s, dtype=float); sd = s.std(ddof=0)
    return (s - s.mean())/sd if sd and sd > 0 else s*0.0

def boot_ci(x, y, n=1000):
    if not HAVE_SCIPY or len(x) < 5: return (np.nan, np.nan)
    x = np.asarray(x, float); y = np.asarray(y, float); idx = np.arange(len(x)); vals = []
    for _ in range(n):
        b = np.random.choice(idx, len(idx), replace=True)
        if len(np.unique(x[b])) < 3 or len(np.unique(y[b])) < 3: continue
        r = spearmanr(x[b], y[b])[0]
        if not np.isnan(r): vals.append(r)
    return (float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))) if vals else (np.nan, np.nan)


# =============================================================================
# 0. Ensure the N06A drug list exists (scrape WHO ATC/DDD if missing)
# =============================================================================
def scrape_n06a(out_csv):
    import requests
    from bs4 import BeautifulSoup
    BASE = "https://atcddd.fhi.no/atc_ddd_index/"
    HDR = {"User-Agent": "Mozilla/5.0 (ATC-DDD-parser; research use)"}
    FIELDS = ["atc_code","name","ddd","unit","route","note","subgroup"]

    def fetch(code, sess):
        r = sess.get(BASE, params={"code": code}, headers=HDR, timeout=30)
        r.raise_for_status(); return BeautifulSoup(r.text, "html.parser")

    def children(soup, parent):
        out = []
        for a in soup.select("a[href*='code=']"):
            c = a["href"].split("code=")[-1].split("&")[0].strip()
            if c.startswith(parent) and c != parent and len(c) == len(parent)+1 and c not in out:
                out.append(c)
        return out

    def parse(soup, sub):
        rows = []
        for tr in soup.select("table tr"):
            cells = [c.get_text(strip=True) for c in tr.find_all("td")]
            if not cells: continue
            code = cells[0]
            if not (code.startswith(sub) and len(code) == len(sub)+2): continue
            d = {"atc_code": code,
                 "name":  cells[1] if len(cells) > 1 else None,
                 "ddd":   cells[2] if len(cells) > 2 and cells[2] else None,
                 "unit":  cells[3] if len(cells) > 3 and cells[3] else None,
                 "route": cells[4] if len(cells) > 4 and cells[4] else None,
                 "note":  cells[5] if len(cells) > 5 and cells[5] else None,
                 "subgroup": sub}
            if not d["name"] and rows: d["name"] = rows[-1]["name"]
            rows.append(d)
        return rows

    with requests.Session() as sess:
        root = fetch("N06A", sess); allrows = []
        for sub in children(root, "N06A"):
            allrows += parse(fetch(sub, sess), sub); time.sleep(0.5)
    os.makedirs(os.path.dirname(out_csv) or ".", exist_ok=True)
    with open(out_csv, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=FIELDS, extrasaction="ignore"); w.writeheader()
        for d in allrows:
            w.writerow({k: (d.get(k) if d.get(k) is not None else "") for k in FIELDS})
    print(f"  scraped {len(allrows)} N06A rows -> {out_csv}")

if not os.path.exists(DRUGS_CSV_PATH):
    print(f"[info] {DRUGS_CSV_PATH} not found — scraping WHO ATC/DDD for N06A ...")
    try:
        scrape_n06a(DRUGS_CSV_PATH)
    except Exception as e:
        raise FileNotFoundError(f"Drug list missing and scraping failed: {e}")


# =============================================================================
# 1. TWAS LOOKUP
# =============================================================================
def _read_twas_file(fpath):
    for sep in [",", "\t", None]:
        for enc in CSV_ENCODINGS:
            try:
                df = pd.read_csv(fpath, sep=sep, encoding=enc, engine="python",
                                 low_memory=False, on_bad_lines="skip")
                if df is not None and not df.empty and df.shape[1] >= 2: return df
            except Exception: continue
    try:
        df = pd.read_csv(fpath, engine="python", on_bad_lines="skip")
        if df is not None and not df.empty: return df
    except Exception: pass
    return None

def build_twas_lookup(twas_dir):
    lookup, skipped, ok_files = {}, [], 0
    files = glob.glob(os.path.join(twas_dir, "*.*"))
    if not files:
        print(f"  [warn] no files in {twas_dir}"); return lookup
    for fp in files:
        try:
            df = _read_twas_file(fp)
            if df is None or df.empty:
                skipped.append((os.path.basename(fp), "unreadable/empty")); continue
            sym = find_col(df, TWAS_SYMBOL_COL_CANDIDATES)
            ens = find_col(df, TWAS_ENSG_COL_CANDIDATES)
            zc  = find_col(df, TWAS_Z_COL_CANDIDATES)
            pc  = find_col(df, TWAS_P_COL_CANDIDATES)
            if ens is not None and ens == sym: ens = None
            if zc is None or (sym is None and ens is None):
                skipped.append((os.path.basename(fp), f"missing cols z={zc} sym={sym} ensg={ens}")); continue
            n = len(df)
            sv = df[sym].astype(str).str.strip().str.upper() if sym else pd.Series([""]*n, index=df.index)
            ev = df[ens].map(_strip_ensg_version) if ens else pd.Series([""]*n, index=df.index)
            zv = df[zc].map(parse_num)
            pv = df[pc].map(parse_num) if pc else pd.Series([np.nan]*n, index=df.index)
            bad = {"NAN","NA","NONE",""}
            for i in df.index:
                g = sv[i]
                if g in bad: g = ev[i]
                if g in bad: continue
                z, p = zv[i], pv[i]
                if _is_nan(z) and _is_nan(p): continue
                prev = lookup.get(g)
                if prev is None:
                    lookup[g] = {"twas_z": z, "twas_p": p}
                else:
                    pp = prev.get("twas_p", np.nan)
                    if not _is_nan(p) and (_is_nan(pp) or p < pp):
                        lookup[g] = {"twas_z": z, "twas_p": p}
            ok_files += 1
        except Exception as e:
            skipped.append((os.path.basename(fp), f"error: {e}")); continue
    print(f"  TWAS genes loaded: {len(lookup):,} from {ok_files}/{len(files)} file(s)")
    for nm, why in skipped[:6]: print(f"    - skipped {nm}: {why}")
    return lookup

def get_twas_for_gene(gene, lookup):
    if gene is None or lookup is None: return None
    return lookup.get(str(gene).strip().upper())


# =============================================================================
# 2. LONG TABLE (drug x receptor) — weight-independent, built ONCE per trait
# =============================================================================
def _recompute_affinity_columns(res):
    ki, ddd = res["Ki_nM"], res["DDD_mg"]
    res["inv_Ki"] = np.where(ki.notna() & (ki > 0), 1.0/ki, np.nan)
    has_inv = res["inv_Ki"].notna(); has_ddd = ddd.notna() & (ddd > 0)
    log_ddd = np.log(ddd.where(has_ddd) + 1.0)
    res["affinity_dose_score"] = np.where(has_inv & has_ddd, res["inv_Ki"]*log_ddd,
                                   np.where(has_inv, res["inv_Ki"], np.nan))
    res["strong_binder"] = ki.notna() & (ki < STRONG_BINDER_KI_NM)
    return res

def drop_zero_score_rows(res):
    if res.empty or "affinity_dose_score" not in res.columns: return res, 0
    z = res["affinity_dose_score"] == 0; n = int(z.sum())
    if n: res = res.loc[~z].reset_index(drop=True)
    print(f"  [drop-zero] removed {n} row(s) with affinity_dose_score == 0.")
    return res, n

def impute_missing_ki(res, strategy="none"):
    if res.empty: return res
    if "ki_imputed" not in res.columns: res["ki_imputed"] = False
    if strategy == "none": return res
    valid = res.loc[res["Ki_nM"].notna(), ["receptor","Ki_nM"]]
    if valid.empty:
        print("  [ki-impute] no reference values; skipped."); return res
    if strategy == "mean":
        per = valid.groupby("receptor")["Ki_nM"].mean(); glob = float(valid["Ki_nM"].mean())
    else:
        per = valid.groupby("receptor")["Ki_nM"].median(); glob = float(valid["Ki_nM"].median())
    need = res["Ki_nM"].isna() & res["receptor"].notna(); n = int(need.sum())
    if n == 0:
        print(f"  [ki-impute] '{strategy}': nothing to fill."); return res
    res.loc[need, "Ki_nM"] = res.loc[need, "receptor"].map(per).fillna(glob).values
    res.loc[need, "ki_imputed"] = True
    res = _recompute_affinity_columns(res)
    print(f"  [ki-impute] '{strategy}': filled {n} Ki value(s) (global fallback {glob:.3g} nM).")
    return res

def impute_missing_twas(res, strategy="none"):
    if res.empty: return res
    if "twas_imputed" not in res.columns: res["twas_imputed"] = False
    if strategy == "none": return res
    vz = res.loc[res["twas_z"].notna(), "twas_z"]; vp = res.loc[res["twas_p"].notna(), "twas_p"]
    if vz.empty and vp.empty:
        print("  [twas-impute] no reference values; skipped."); return res
    if strategy == "mean":
        fz = float(vz.mean()) if not vz.empty else np.nan
        fp = float(vp.mean()) if not vp.empty else np.nan
    else:
        fz = float(vz.median()) if not vz.empty else np.nan
        fp = float(vp.median()) if not vp.empty else np.nan
    nz = res["gene"].notna() & res["twas_z"].isna()
    npz = res["gene"].notna() & res["twas_p"].isna()
    n = int((nz | npz).sum())
    if n == 0:
        print(f"  [twas-impute] '{strategy}': nothing to fill."); return res
    if not _is_nan(fz): res.loc[nz, "twas_z"] = fz
    if not _is_nan(fp): res.loc[npz, "twas_p"] = fp
    res.loc[nz | npz, "twas_imputed"] = True
    print(f"  [twas-impute] '{strategy}': filled {n} row(s) (z={fz:.3f}, p={fp:.3g}); kept NON-significant.")
    return res

def build_long_table(twas_dir):
    """Ki x DDD x TWAS long table for one trait (independent of receptor weights)."""
    stats = {"twas_dir": twas_dir}
    print(f"Building TWAS lookup for: {twas_dir}")
    lookup = build_twas_lookup(twas_dir)
    stats["n_genes_twas"] = len(lookup)
    for g in ("HTR2C","HRH1","DRD2","CHRM1","CHRM3","HTR2A","SLC6A4"):
        print(f"    {g} present? {g in lookup}")

    ki = read_csv_robust(KI_DB_PATH);      print(f"  Ki rows: {len(ki):,}")
    drugs = read_csv_robust(DRUGS_CSV_PATH); print(f"  Drugs: {len(drugs):,}")
    stats["n_drugs_input"] = len(drugs)

    lig_c = find_col(ki, ["Test Ligands","Test Ligand","Ligand","ligand"])
    rec_c = find_col(ki, ["Receptor","receptor","Target"])
    kiv_c = find_col(ki, ["Ki Value","Ki","Ki_nM","Ki (nM)"])
    if not all([lig_c, rec_c, kiv_c]):
        raise ValueError(f"Ki columns not found (ligand={lig_c}, receptor={rec_c}, ki={kiv_c}); "
                         f"available: {list(ki.columns)}")
    ki["_ligand_norm"] = ki[lig_c].map(normalise_name)
    ki["_ki_nM"] = ki[kiv_c].map(parse_num)
    vocab = sorted(set(ki["_ligand_norm"]) - {""})

    rows, missing = [], set()
    print("\nMatching drugs and building drug x receptor table ...")
    for _, drug in tqdm(drugs.iterrows(), total=len(drugs)):
        name = drug.get("name"); norm = normalise_name(name)
        ddd_mg = ddd_to_mg(drug.get("ddd"), drug.get("unit"))
        inv_ddd = (1.0/ddd_mg) if (not _is_nan(ddd_mg) and ddd_mg > 0) else np.nan
        if not norm: continue
        m = process.extractOne(norm, vocab, scorer=fuzz.WRatio)
        if m is None or m[1] < FUZZY_THRESHOLD:
            rows.append({"atc_code": drug.get("atc_code"), "drug": name,
                         "matched_ligand": None, "match_score": (m[1] if m else np.nan),
                         "receptor": None, "gene": None, "Ki_nM": np.nan, "inv_Ki": np.nan,
                         "DDD_mg": ddd_mg, "inv_DDD": inv_ddd, "affinity_dose_score": np.nan,
                         "n_measurements": 0, "twas_z": np.nan, "twas_p": np.nan,
                         "strong_binder": False, "twas_significant": False,
                         "ki_imputed": False, "twas_imputed": False})
            continue
        lig, score, _ = m
        sub = ki[ki["_ligand_norm"] == lig]
        for receptor, grp in sub.groupby(rec_c):
            ki_nM = aggregate_ki(grp["_ki_nM"], KI_AGGREGATION)
            n_meas = int(grp["_ki_nM"].notna().sum())
            inv_ki = (1.0/ki_nM) if (not _is_nan(ki_nM) and ki_nM > 0) else np.nan
            gene = receptor_to_gene(receptor)
            if not _is_nan(inv_ki) and not _is_nan(ddd_mg) and ddd_mg > 0:
                sc = inv_ki * np.log(ddd_mg + 1.0)
            elif not _is_nan(inv_ki): sc = inv_ki
            else: sc = np.nan
            tw = get_twas_for_gene(gene, lookup) if gene else None
            if gene and tw is None: missing.add(gene)
            tz = tw.get("twas_z", np.nan) if tw else np.nan
            tp = tw.get("twas_p", np.nan) if tw else np.nan
            rows.append({"atc_code": drug.get("atc_code"), "drug": name,
                         "matched_ligand": lig, "match_score": round(score,1),
                         "receptor": receptor, "gene": gene, "Ki_nM": ki_nM, "inv_Ki": inv_ki,
                         "DDD_mg": ddd_mg, "inv_DDD": inv_ddd, "affinity_dose_score": sc,
                         "n_measurements": n_meas, "twas_z": tz, "twas_p": tp,
                         "strong_binder": (not _is_nan(ki_nM)) and ki_nM < STRONG_BINDER_KI_NM,
                         "twas_significant": (not _is_nan(tp)) and tp < TWAS_P_SIGNIFICANT,
                         "ki_imputed": False, "twas_imputed": False})

    res = pd.DataFrame(rows)
    stats["n_missing_twas_genes"] = len(missing)
    stats["missing_twas_sample"] = sorted(missing)[:12]
    if missing:
        print(f"\n[TWAS] {len(missing)} receptor gene(s) had no TWAS data "
              f"(e.g. {', '.join(sorted(missing)[:8])})")

    if not res.empty:
        n0 = res["drug"].nunique()
        keep = res[res["DDD_mg"].notna()]["drug"].unique()
        res = res[res["drug"].isin(keep)].reset_index(drop=True)
        n1 = res["drug"].nunique(); stats["n_drugs_ddd_dropped"] = int(n0-n1)
        print(f"\n[DDD filter] kept {n1}/{n0} drug(s) with a valid DDD ({n0-n1} dropped).")
        print("\nImputation step ...")
        res = impute_missing_ki(res, KI_IMPUTE_STRATEGY)
        res = impute_missing_twas(res, TWAS_IMPUTE_STRATEGY)

    res, nz = drop_zero_score_rows(res); stats["n_zero_score_dropped"] = nz

    if not res.empty:
        res["rank_in_drug"] = res.groupby("drug")["affinity_dose_score"].rank(ascending=False, method="min")
        res = res.sort_values(["drug","affinity_dose_score"], ascending=[True, False]).reset_index(drop=True)
        stats.update({
            "n_drugs_matched": int(res.loc[res["receptor"].notna(),"drug"].nunique()),
            "n_pairs": int(res["receptor"].notna().sum()),
            "n_ki_imputed": int(res["ki_imputed"].sum()),
            "n_twas_imputed": int(res["twas_imputed"].sum()),
            "n_strong_binders": int(res["strong_binder"].sum()),
            "n_twas_significant": int(res["twas_significant"].sum())})
    else:
        stats.update({"n_drugs_matched":0,"n_pairs":0,"n_ki_imputed":0,
                      "n_twas_imputed":0,"n_strong_binders":0,"n_twas_significant":0})
    return res, stats


# =============================================================================
# 3. RISK SCORE  ——  JvH METHOD: per-receptor exp(beta * -log10 p) BEFORE summing
# =============================================================================
def calculate_metabolic_risk_score(
    df, weights=None, use_metabolic_weights=True, per_drug_weights=None,
    score_transform="log1p", twas_boost=True, twas_scaling=TWAS_SCALING,
    beta=LOGP_BETA, gamma=0.12, twas_sig_bonus=TWAS_SIG_BONUS, normalize=True,
    reference_drug=REFERENCE_DRUG, missing_gene_strategy="mean_abs_z",
    max_neglog10_p=MAX_NEGLOG10_P, secondary_boost_factor=SECONDARY_BOOST_FAC,
    return_detail=False, verbose=True):
    """
    Per-drug metabolic risk for antidepressants.

    TWAS INTEGRATION (JvH / Version-3 style):
        twas_scale = exp(beta * -log10(p)),  -log10(p) capped at max_neglog10_p
        contribution_i = log1p(affinity_dose_score_i) * weight_i * twas_scale_i
        base_score     = sum_i contribution_i          <-- scaled BEFORE summing
        risk           = base_score
                         * (1 + secondary_boost_factor * max|twas_z|)
                         * (1 + twas_sig_bonus * n_twas_sig_metabolic)
        normalised so that `reference_drug` = 100.

    twas_scaling: "logp_expo" (default) | "logp_linear" | "z_expo" | "z_linear" | "none"
    per_drug_weights: optional {drug: {gene: w}} — used by the 5-HT adaptation.
    """
    if weights is None: weights = METABOLIC_WEIGHTS
    d = df.copy()
    mask = d["gene"].notna() & d["affinity_dose_score"].notna()
    if int(mask.sum()) == 0:
        empty = pd.DataFrame(columns=["risk_rank","drug","metabolic_risk_score"])
        return (empty, pd.DataFrame()) if return_detail else empty

    # ---- weight vector -------------------------------------------------
    if per_drug_weights is not None:
        active_keys = set(weights.keys())
        wv = [float(per_drug_weights.get(dr, weights).get(g, 0.0))
              for dr, g in zip(d.loc[mask,"drug"], d.loc[mask,"gene"])]
        d.loc[mask, "receptor_weight"] = wv
        if verbose: print("  [weights] per-drug ADAPTED weights (5-HT remodelling)")
    elif use_metabolic_weights:
        active_keys = set(weights.keys())
        d.loc[mask, "receptor_weight"] = d.loc[mask,"gene"].map(weights).fillna(0.0)
        if verbose: print("  [weights] literature METABOLIC_WEIGHTS (clinical proxy)")
    else:
        active_keys = set(d.loc[mask,"gene"].dropna().unique())
        d.loc[mask, "receptor_weight"] = 1.0
        if verbose: print("  [weights] UNIFORM 1.0 for every recognised receptor (genomics-driven)")

    metab_mask = mask & d["gene"].isin(active_keys)
    uses_p = twas_scaling in ("logp_expo","logp_linear")
    cov_col = "twas_p" if uses_p else "twas_z"
    n_metab = int(metab_mask.sum())
    n_cov = int((metab_mask & d[cov_col].notna()).sum())
    if twas_boost and n_metab > 0 and verbose:
        cov = n_cov/n_metab
        print(f"  [note] TWAS coverage: {n_cov}/{n_metab} ({cov:.0%}) scored-gene rows have {cov_col}.")

    if score_transform == "log1p":
        affinity = np.log1p(d.loc[mask,"affinity_dose_score"].clip(lower=0))
    elif score_transform == "none":
        affinity = d.loc[mask,"affinity_dose_score"]
    else:
        raise ValueError(f"Unknown score_transform: {score_transform!r}")

    d.loc[mask,"affinity_log1p"] = affinity
    d.loc[mask,"weighted_contribution"] = affinity * d.loc[mask,"receptor_weight"]
    d.loc[mask,"contribution_no_twas"]  = d.loc[mask,"weighted_contribution"]

    drug_scores = (d.groupby("drug", dropna=False)["weighted_contribution"].sum()
                    .reset_index().rename(columns={"weighted_contribution":"base_score"}))

    df_filled = d.copy()
    if twas_boost and twas_scaling != "none":
        # ---- fairness imputation for scored genes lacking TWAS ----------
        if missing_gene_strategy == "mean_abs_z":
            if uses_p:
                ap = df_filled.loc[metab_mask & df_filled["twas_p"].notna(),"twas_p"].clip(lower=1e-300)
                mean_nlp = float((-np.log10(ap)).mean()) if len(ap) else 0.0
                im = mask & df_filled["twas_p"].isna() & df_filled["gene"].isin(active_keys)
                if int(im.sum()):
                    df_filled.loc[im,"twas_p"] = 10.0 ** (-mean_nlp)
                    if verbose: print(f"  [fairness] imputed {int(im.sum())} p-value(s) with mean -log10(p)={mean_nlp:.3f}")
            else:
                az = df_filled.loc[metab_mask & df_filled["twas_z"].notna(),"twas_z"].abs()
                mz = float(az.mean()) if len(az) else 0.0
                im = mask & df_filled["twas_z"].isna() & df_filled["gene"].isin(active_keys)
                if int(im.sum()):
                    df_filled.loc[im,"twas_z"] = mz
                    if verbose: print(f"  [fairness] imputed {int(im.sum())} row(s) with mean |z|={mz:.3f}")
        elif missing_gene_strategy != "zero":
            raise ValueError(f"Unknown missing_gene_strategy: {missing_gene_strategy!r}")

        # ---- per-receptor TWAS scale -----------------------------------
        if twas_scaling == "logp_expo":
            ps = df_filled.loc[mask,"twas_p"].clip(lower=1e-300)
            nlp = (-np.log10(ps)).clip(upper=max_neglog10_p).fillna(0.0)
            df_filled.loc[mask,"twas_scale"] = np.exp(beta * nlp)
            if verbose: print(f"  [TWAS-logp] exp scaling on -log10(p) (beta={beta}, cap={max_neglog10_p}); "
                              f"secondary boost {secondary_boost_factor}")
        elif twas_scaling == "logp_linear":
            ps = df_filled.loc[mask,"twas_p"].clip(lower=1e-300)
            nlp = (-np.log10(ps)).clip(upper=max_neglog10_p).fillna(0.0)
            df_filled.loc[mask,"twas_scale"] = 1.0 + gamma * nlp
        elif twas_scaling == "z_expo":
            df_filled.loc[mask,"twas_scale"] = np.exp(beta * df_filled.loc[mask,"twas_z"].abs().fillna(0.0))
        elif twas_scaling == "z_linear":
            df_filled.loc[mask,"twas_scale"] = 1.0 + beta * df_filled.loc[mask,"twas_z"].abs().fillna(0.0)
        else:
            raise ValueError(f"Unknown twas_scaling: {twas_scaling!r}")

        df_filled.loc[mask,"weighted_contribution"] = (
            affinity * df_filled.loc[mask,"receptor_weight"] * df_filled.loc[mask,"twas_scale"])

        drug_scores = (df_filled.groupby("drug", dropna=False)["weighted_contribution"].sum()
                        .reset_index().rename(columns={"weighted_contribution":"base_score"}))

        tsig = (df_filled.groupby("drug")["twas_z"]
                 .apply(lambda x: x.abs().max() if x.notna().any() else 0.0)
                 .reset_index().rename(columns={"twas_z":"max_abs_twas_z"}))
        drug_scores = drug_scores.merge(tsig, on="drug", how="left")
        drug_scores["max_abs_twas_z"] = drug_scores["max_abs_twas_z"].fillna(0.0)
        boost = 1 + secondary_boost_factor * drug_scores["max_abs_twas_z"]
    else:
        df_filled.loc[mask,"twas_scale"] = 1.0
        boost = 1.0

    if twas_sig_bonus and "twas_significant" in d.columns:
        sig = d[mask & d["twas_significant"].fillna(False) & (d["receptor_weight"] > 0)]
        nsig = sig.groupby("drug").size().reset_index(name="n_twas_sig_metabolic")
        drug_scores = drug_scores.merge(nsig, on="drug", how="left")
        drug_scores["n_twas_sig_metabolic"] = drug_scores["n_twas_sig_metabolic"].fillna(0).astype(int)
        sig_mult = 1 + twas_sig_bonus * drug_scores["n_twas_sig_metabolic"]
    else:
        drug_scores["n_twas_sig_metabolic"] = 0
        sig_mult = 1.0

    drug_scores["metabolic_risk_score"] = drug_scores["base_score"] * boost * sig_mult

    if normalize:
        rm = drug_scores["drug"].astype(str).str.lower() == reference_drug.lower()
        rs = drug_scores.loc[rm, "metabolic_risk_score"]
        if not rs.empty and rs.values[0] > 0:
            ref = rs.values[0]
        else:
            if verbose: print(f"  [warn] reference '{reference_drug}' absent/zero; normalising to max.")
            ref = drug_scores["metabolic_risk_score"].max()
        if ref and ref > 0:
            drug_scores["metabolic_risk_score"] = drug_scores["metabolic_risk_score"]/ref*100
            drug_scores["risk_vs_reference"] = drug_scores["metabolic_risk_score"] - 100

    nz = int((drug_scores["metabolic_risk_score"] == 0).sum())
    if nz:
        drug_scores = drug_scores[drug_scores["metabolic_risk_score"] != 0]
        if verbose: print(f"  [drop-zero] removed {nz} drug(s) with risk == 0.")

    drug_scores["risk_rank"] = drug_scores["metabolic_risk_score"].rank(ascending=False, method="min").astype("Int64")
    drug_scores = drug_scores.sort_values("metabolic_risk_score", ascending=False).reset_index(drop=True)
    pref = ["risk_rank","drug","metabolic_risk_score","base_score","max_abs_twas_z",
            "n_twas_sig_metabolic","risk_vs_reference"]
    out = drug_scores[[c for c in pref if c in drug_scores.columns]]

    if not return_detail: return out
    det = df_filled.loc[mask, ["drug","gene","receptor","Ki_nM","DDD_mg","affinity_dose_score",
                               "affinity_log1p","twas_z","twas_p","twas_significant","strong_binder",
                               "ki_imputed","twas_imputed","receptor_weight","twas_scale",
                               "weighted_contribution","contribution_no_twas"]].copy()
    det = det.rename(columns={"weighted_contribution":"contribution"})
    tot = det.groupby("drug")["contribution"].transform("sum")
    det["share_pct"] = np.where(tot > 0, 100*det["contribution"]/tot, np.nan)
    return out, det


# =============================================================================
# 4. 5-HT ADAPTATION (antidepressant-specific post-hoc, re-scored with logp_expo)
# =============================================================================
def sert_affinity_map(long_df):
    sub = long_df[long_df["gene"] == "SLC6A4"][["drug","affinity_dose_score"]]
    return sub.groupby("drug")["affinity_dose_score"].max().to_dict() if not sub.empty else {}

def adapted_weights_for(sert_aff, base):
    w = dict(base)
    if sert_aff is not None and sert_aff > SERT_THRESHOLD:
        if "HTR2C" in w: w["HTR2C"] += HTR2C_INCREASE
        if "HTR2A" in w: w["HTR2A"] += HTR2A_INCREASE
        if "HTR1A" in w: w["HTR1A"] = max(HTR1A_FLOOR, w["HTR1A"] - HTR1A_REDUCTION)
    return w


# =============================================================================
# 5. OUTPUT WRITERS
# =============================================================================
def write_run_outputs(res, det, risk, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    paths = {}
    p = os.path.join(out_dir, "n06a_ki_twas_full_results.csv"); res.to_csv(p, index=False); paths["long"] = p
    p = os.path.join(out_dir, "metabolic_risk_score_per_drug.csv"); risk.to_csv(p, index=False); paths["risk"] = p
    if det is not None and not det.empty:
        p = os.path.join(out_dir, "receptor_contributions.csv"); det.to_csv(p, index=False); paths["detail"] = p
    matched = res[res["receptor"].notna()]
    if not matched.empty:
        summ = (matched.groupby(["atc_code","drug"]).agg(
                    n_receptors=("receptor","nunique"), n_strong_binders=("strong_binder","sum"),
                    n_twas_sig=("twas_significant","sum"), n_ki_imputed=("ki_imputed","sum"),
                    n_twas_imputed=("twas_imputed","sum"), best_Ki_nM=("Ki_nM","min"),
                    top_score=("affinity_dose_score","max"), DDD_mg=("DDD_mg","first"))
                .reset_index().sort_values("top_score", ascending=False))
        p = os.path.join(out_dir, "drug_summary.csv"); summ.to_csv(p, index=False); paths["summary"] = p
        try:
            xp = os.path.join(out_dir, "n06a_ki_twas_full_results.xlsx")
            with pd.ExcelWriter(xp, engine="openpyxl") as xw:
                res.to_excel(xw, sheet_name="All", index=False)
                matched[matched["rank_in_drug"] <= 5].sort_values(["drug","rank_in_drug"]) \
                    .to_excel(xw, sheet_name="Top5_per_drug", index=False)
                matched[matched["twas_significant"]].sort_values("affinity_dose_score", ascending=False) \
                    .to_excel(xw, sheet_name="TWAS_significant", index=False)
                summ.to_excel(xw, sheet_name="Drug_summary", index=False)
                risk.to_excel(xw, sheet_name="Risk", index=False)
            paths["xlsx"] = xp
        except Exception as e:
            print(f"  (skipped Excel export: {e})")
    for k, v in paths.items(): FILES_OUT.append((v, f"run output [{k}]"))
    return paths


# =============================================================================
# 6. MAIN PIPELINE — dual run (+ optional 5-HT adapted third method)
# =============================================================================
print("\n" + "#"*100)
print("# PART 1 — SCORING (JvH logp_expo TWAS integration, dual weighting)")
print("#"*100)

RISK, DETAIL, RUNSTATS, LONGTBL = {}, {}, {}, {}
ADAPT_TABLES = {}
METHOD_ORDER = ["weighted", "uniform"] + (["weighted_5ht_adapted"] if RUN_5HT_ADAPTATION else [])

for twas_dir in TWAS_DIRS:
    trait = _safe_label(twas_dir)
    print("\n" + "#"*100)
    print(f"# TRAIT: {trait}   ({twas_dir})")
    print("#"*100)
    try:
        long_df, base_stats = build_long_table(twas_dir)
    except Exception as e:
        print(f"  [ERROR] long-table build failed for {trait}: {e}")
        RUNSTATS[("*", trait)] = {"error": str(e), "trait": trait, "method": "*"}
        continue
    if long_df.empty:
        print(f"  [warn] empty result table for {trait}"); continue
    LONGTBL[trait] = long_df

    # ---------------- dual weighting -------------------------------------
    for scheme in ["weighted", "uniform"]:
        out_dir = os.path.join(OUTPUT_ROOT, scheme, trait)
        print(f"\n--- scoring [{scheme}] {trait} ---")
        risk, det = calculate_metabolic_risk_score(
            long_df, weights=METABOLIC_WEIGHTS,
            use_metabolic_weights=(scheme == "weighted"),
            score_transform="log1p", twas_boost=True, twas_scaling=TWAS_SCALING,
            beta=LOGP_BETA, twas_sig_bonus=TWAS_SIG_BONUS, normalize=True,
            reference_drug=REFERENCE_DRUG, missing_gene_strategy="mean_abs_z",
            max_neglog10_p=MAX_NEGLOG10_P, secondary_boost_factor=SECONDARY_BOOST_FAC,
            return_detail=True)
        write_run_outputs(long_df, det, risk, out_dir)
        RISK[(scheme, trait)] = risk; DETAIL[(scheme, trait)] = det
        st = dict(base_stats); st.update({"method": scheme, "trait": trait, "out_dir": out_dir,
                                          "n_drugs_scored": int(len(risk))})
        RUNSTATS[(scheme, trait)] = st
        print(risk.head(10).to_string(index=False))

    # ---------------- 5-HT adaptation (weighted base) ---------------------
    if RUN_5HT_ADAPTATION:
        scheme = "weighted_5ht_adapted"
        out_dir = os.path.join(OUTPUT_ROOT, scheme, trait)
        print(f"\n--- scoring [{scheme}] {trait} ---")
        smap = sert_affinity_map(long_df)
        pdw = {d: adapted_weights_for(smap.get(d, 0.0), METABOLIC_WEIGHTS)
               for d in long_df["drug"].dropna().unique()}
        risk_a, det_a = calculate_metabolic_risk_score(
            long_df, weights=METABOLIC_WEIGHTS, per_drug_weights=pdw,
            score_transform="log1p", twas_boost=True, twas_scaling=TWAS_SCALING,
            beta=LOGP_BETA, twas_sig_bonus=TWAS_SIG_BONUS, normalize=True,
            reference_drug=REFERENCE_DRUG, missing_gene_strategy="mean_abs_z",
            max_neglog10_p=MAX_NEGLOG10_P, secondary_boost_factor=SECONDARY_BOOST_FAC,
            return_detail=True)
        write_run_outputs(long_df, det_a, risk_a, out_dir)
        RISK[(scheme, trait)] = risk_a; DETAIL[(scheme, trait)] = det_a
        st = dict(base_stats); st.update({"method": scheme, "trait": trait, "out_dir": out_dir,
                                          "n_drugs_scored": int(len(risk_a))})
        RUNSTATS[(scheme, trait)] = st

        base_r = RISK[("weighted", trait)][["drug","metabolic_risk_score","risk_rank"]] \
                    .rename(columns={"metabolic_risk_score":"base_risk","risk_rank":"base_rank"})
        adp = risk_a[["drug","metabolic_risk_score","risk_rank"]] \
                    .rename(columns={"metabolic_risk_score":"adapted_risk","risk_rank":"adapted_rank"})
        cmp_ = base_r.merge(adp, on="drug", how="outer")
        cmp_["sert_affinity"] = cmp_["drug"].map(smap).fillna(0.0)
        cmp_["adapted"] = cmp_["sert_affinity"] > SERT_THRESHOLD
        cmp_["adaptation_delta"] = cmp_["adapted_risk"] - cmp_["base_risk"]
        cmp_["rank_delta"] = cmp_["base_rank"].astype(float) - cmp_["adapted_rank"].astype(float)
        cmp_["is_ssri_snri"] = cmp_["drug"].str.contains(SSRI_SNRI_REGEX, case=False, na=False)
        cmp_ = cmp_.sort_values("adaptation_delta", ascending=False).reset_index(drop=True)
        p = os.path.join(out_dir, "metabolic_risk_5ht_adapted_vs_base.csv")
        cmp_.to_csv(p, index=False); FILES_OUT.append((p, "5-HT adaptation vs base"))
        ADAPT_TABLES[trait] = cmp_
        print(cmp_.head(10).to_string(index=False))

if not RISK:
    raise RuntimeError("No successful runs — check KI_DB_PATH / DRUGS_CSV_PATH / TWAS_DIRS.")


# =============================================================================
# 7. LONG RANK TABLE ACROSS METHODS x TRAITS
# =============================================================================
rows = []
for (m, t), df in RISK.items():
    for _, r in df.iterrows():
        rows.append({"method": m,
                     "method_kind": ("uniform" if m == "uniform" else "weighted"),
                     "trait": t, "drug": r["drug"], "drug_key": norm_drug(r["drug"]),
                     "risk_rank": float(r["risk_rank"]) if pd.notna(r["risk_rank"]) else np.nan,
                     "risk_score": r.get("metabolic_risk_score", np.nan),
                     "base_score": r.get("base_score", np.nan),
                     "max_abs_twas_z": r.get("max_abs_twas_z", np.nan),
                     "n_twas_sig": r.get("n_twas_sig_metabolic", np.nan)})
LONG = pd.DataFrame(rows)
save_table(LONG, "00_master_long_risk_table.csv", "All runs: method x trait x drug")

METHODS = [m for m in METHOD_ORDER if m in set(LONG["method"])]
TRAITS  = sorted(LONG["trait"].unique())
DRUG_LABEL = LONG.drop_duplicates("drug_key").set_index("drug_key")["drug"].to_dict()
WMETHOD, UMETHOD = "weighted", "uniform"
AMETHOD = "weighted_5ht_adapted" if "weighted_5ht_adapted" in METHODS else None
HAVE_PAIR = WMETHOD in METHODS and UMETHOD in METHODS

RANK_WIDE  = LONG.pivot_table(index="drug_key", columns=["method","trait"], values="risk_rank")
SCORE_WIDE = LONG.pivot_table(index="drug_key", columns=["method","trait"], values="risk_score")
COMPLETE   = RANK_WIDE.dropna(how="any")
N_RUNS     = RANK_WIDE.shape[1]

ws = SCORE_WIDE.copy(); ws.columns = [f"{a}|{b}" for a, b in ws.columns]
ws.insert(0, "drug", [DRUG_LABEL[i] for i in ws.index])
save_table(ws.reset_index(drop=True), "01_wide_scores_all_runs.csv", "Drug x (method|trait) scores")
wr = RANK_WIDE.copy(); wr.columns = [f"{a}|{b}" for a, b in wr.columns]
wr.insert(0, "drug", [DRUG_LABEL[i] for i in wr.index])
save_table(wr.reset_index(drop=True), "02_wide_ranks_all_runs.csv", "Drug x (method|trait) ranks")


# =============================================================================
# 8. SUMMARY — HEADER + PIPELINE SECTIONS
# =============================================================================
S(BAR)
S("N06A ANTIDEPRESSANTS — Ki -> DDD -> TWAS METABOLIC-RISK PIPELINE (080KvD-v3)")
S("TWAS INTEGRATION: JvH METHOD — per-receptor exp(beta * -log10 p), applied BEFORE aggregation")
S("DUAL RUN: weighted (clinical proxy) vs uniform (genomics-influenced)"
  + (" + weighted_5ht_adapted (chronic SERT remodelling)" if AMETHOD else ""))
S(BAR)
S(f"Generated              : {datetime.datetime.now():%Y-%m-%d %H:%M:%S}")
S(f"Ki database            : {KI_DB_PATH}")
S(f"Drug list              : {DRUGS_CSV_PATH}")
S(f"Output root            : {OUTPUT_ROOT}")
S(f"Stage-3 dir            : {STAGE3_DIR}")
S(f"Score formula          : affinity_dose_score = inv_Ki * log(DDD_mg + 1)")
S(f"Contribution           : log1p(affinity_dose_score) * receptor_weight * exp(beta*-log10 p)")
S(f"Drug risk              : sum(contributions) * (1+{SECONDARY_BOOST_FAC}*max|z|) "
  f"* (1+{TWAS_SIG_BONUS}*n_twas_sig)")
S(f"Normalisation          : {REFERENCE_DRUG} = 100 (within each run)")
S(f"Methods                : {', '.join(METHODS)}")
S(f"Traits                 : {', '.join(TRAITS)}")
S(f"Runs (method x trait)  : {N_RUNS}   drugs(union)={RANK_WIDE.shape[0]}  complete={COMPLETE.shape[0]}")
S("")
S("Configuration:")
for k, v in [("FUZZY_THRESHOLD",FUZZY_THRESHOLD),("KI_AGGREGATION",KI_AGGREGATION),
             ("STRONG_BINDER_KI_NM",STRONG_BINDER_KI_NM),("TWAS_P_SIGNIFICANT",TWAS_P_SIGNIFICANT),
             ("KI_IMPUTE_STRATEGY",KI_IMPUTE_STRATEGY),("TWAS_IMPUTE_STRATEGY",TWAS_IMPUTE_STRATEGY),
             ("TWAS_SCALING",TWAS_SCALING),("LOGP_BETA",LOGP_BETA),("MAX_NEGLOG10_P",MAX_NEGLOG10_P),
             ("SECONDARY_BOOST_FAC",SECONDARY_BOOST_FAC),("TWAS_SIG_BONUS",TWAS_SIG_BONUS),
             ("SERT_THRESHOLD",SERT_THRESHOLD),("HTR2C_INCREASE",HTR2C_INCREASE),
             ("HTR2A_INCREASE",HTR2A_INCREASE),("HTR1A_REDUCTION",HTR1A_REDUCTION),
             ("TOP_K",TOP_K),("ALPHA",ALPHA),("scipy",HAVE_SCIPY),("matplotlib",HAVE_MPL)]:
    S(f"  {k:<22}= {v}")

SEC("A. RUN INVENTORY / PROVENANCE")
S(f"{'method':<24}{'trait':<16}{'twas genes':>12}{'drugs in':>10}{'matched':>9}{'pairs':>8}"
  f"{'ki_imp':>8}{'tw_imp':>8}{'tw_sig':>8}{'scored':>8}")
S(SUB)
for (m, t) in sorted(RUNSTATS.keys()):
    s = RUNSTATS[(m, t)]
    if s.get("error"):
        S(f"{m:<24}{t:<16}  ERROR: {s['error']}"); continue
    S(f"{m:<24}{t:<16}{s.get('n_genes_twas',0):>12,}{s.get('n_drugs_input',0):>10,}"
      f"{s.get('n_drugs_matched',0):>9}{s.get('n_pairs',0):>8}{s.get('n_ki_imputed',0):>8}"
      f"{s.get('n_twas_imputed',0):>8}{s.get('n_twas_significant',0):>8}{s.get('n_drugs_scored',0):>8}")
for t in TRAITS:
    k = ("weighted", t)
    if k in RUNSTATS and RUNSTATS[k].get("missing_twas_sample"):
        S(f"  [{t}] receptor genes without TWAS ({RUNSTATS[k].get('n_missing_twas_genes',0)}): "
          f"{', '.join(RUNSTATS[k]['missing_twas_sample'])}")

SEC("B. PER-RUN DESCRIPTIVES AND TOP-RISK TABLES")
desc = []
for (m, t), df in sorted(RISK.items()):
    sc = df["metabolic_risk_score"].dropna()
    desc.append({"method": m, "trait": t, "n": len(df),
                 "min": sc.min(), "q25": sc.quantile(.25), "median": sc.median(),
                 "q75": sc.quantile(.75), "max": sc.max(), "mean": sc.mean(), "sd": sc.std(),
                 "n_above_ref": int((sc > 100).sum()),
                 "top_drug": df.iloc[0]["drug"] if len(df) else "NA",
                 "top_score": df.iloc[0]["metabolic_risk_score"] if len(df) else np.nan})
DESC = pd.DataFrame(desc); save_table(DESC, "03_per_run_descriptives.csv", "Score distributions")
S(f"{'method':<24}{'trait':<16}{'n':>5}{'median':>10}{'IQR':>10}{'max':>10}{'>100':>7}  top drug")
S(SUB)
for _, r in DESC.iterrows():
    S(f"{r['method']:<24}{r['trait']:<16}{int(r['n']):>5}{fmt(r['median'],2):>10}"
      f"{fmt(r['q75']-r['q25'],2):>10}{fmt(r['max'],2):>10}{int(r['n_above_ref']):>7}  "
      f"{r['top_drug']} ({fmt(r['top_score'],1)})")

for (m, t), df in sorted(RISK.items()):
    SUBSEC(f"TOP {TOP_K} — {m} | {t}   ({REFERENCE_DRUG} = 100)")
    S(f"{'rank':>5}  {'drug':<30}{'risk':>10}{'base':>12}{'max|z|':>9}{'n_sig':>7}{'vs ref':>10}")
    for _, r in df.head(TOP_K).iterrows():
        S(f"{str(r.get('risk_rank','')):>5}  {str(r['drug'])[:29]:<30}"
          f"{fmt(r.get('metabolic_risk_score'),2):>10}{fmt(r.get('base_score'),4):>12}"
          f"{fmt(r.get('max_abs_twas_z'),3):>9}{fmt(r.get('n_twas_sig_metabolic'),0):>7}"
          f"{fmt(r.get('risk_vs_reference'),2):>10}")
    tail = df.tail(5).iloc[::-1]
    S("  lowest-risk 5: " + ", ".join(f"{r['drug']} ({fmt(r['metabolic_risk_score'],1)})"
                                      for _, r in tail.iterrows()))

# ---- receptor drivers per run ----------------------------------------------
SEC("C. RECEPTOR-DRIVER DECOMPOSITION PER RUN")
S("contribution = log1p(affinity_dose_score) * receptor_weight * exp(beta * -log10 p)")
drv_all = []
for (m, t), det in sorted(DETAIL.items()):
    if det is None or det.empty: continue
    d = det.copy(); d["method"] = m; d["trait"] = t; drv_all.append(d)
    g = d.groupby("gene")["contribution"].sum().reset_index()
    g["pct"] = 100*g["contribution"]/g["contribution"].sum()
    g = g.sort_values("contribution", ascending=False)
    S("")
    S(f"[{m} | {t}] " + " | ".join(f"{r['gene']} {fmt(r['pct'],1)}%" for _, r in g.head(8).iterrows()))
    if len(g):
        SIG.append(f"[C] {m} | {t}: dominant receptor driver = {g.iloc[0]['gene']} "
                   f"({fmt(g.iloc[0]['pct'],1)}% of total signal); runner-up = "
                   f"{g.iloc[1]['gene'] if len(g) > 1 else 'NA'}.")
if drv_all:
    DRV = pd.concat(drv_all, ignore_index=True)
    DRV["drug_key"] = DRV["drug"].map(norm_drug)
    DRV["method_kind"] = np.where(DRV["method"] == "uniform", "uniform", "weighted")
    save_table(DRV, "04_receptor_contributions_all_runs.csv", "Per drug x receptor contribution")
else:
    DRV = None

SUBSEC("DRIVER BREAKDOWN FOR THE TOP 6 DRUGS OF EACH RUN")
if DRV is not None:
    for (m, t), rdf in sorted(RISK.items()):
        sub = DRV[(DRV.method == m) & (DRV.trait == t)]
        if sub.empty: continue
        S(""); S(f"### {m} | {t}")
        for _, dr in rdf.head(6).iterrows():
            dd = sub[sub["drug_key"] == norm_drug(dr["drug"])].sort_values("contribution", ascending=False)
            if dd.empty: continue
            S(f"  {dr['drug']}  (risk {fmt(dr['metabolic_risk_score'],2)}, rank {int(dr['risk_rank'])}, "
              f"DDD {fmt(dd.iloc[0]['DDD_mg'],1)} mg)")
            S(f"    {'gene':<9}{'Ki nM':>10}{'w':>7}{'twas_z':>9}{'twas_p':>11}{'scale':>8}{'share %':>9}{'flags':>18}")
            for _, r in dd.head(6).iterrows():
                fl = []
                if bool(r.get("strong_binder", False)): fl.append("strong")
                if bool(r.get("twas_significant", False)): fl.append("twasSig")
                if bool(r.get("ki_imputed", False)): fl.append("KiImp")
                if bool(r.get("twas_imputed", False)): fl.append("TwImp")
                S(f"    {str(r['gene'])[:8]:<9}{fmt(r['Ki_nM'],2):>10}{fmt(r['receptor_weight'],2):>7}"
                  f"{fmt(r['twas_z']):>9}{fmt(r['twas_p']):>11}{fmt(r['twas_scale'],4):>8}"
                  f"{fmt(r['share_pct'],1):>9}{','.join(fl)[:17]:>18}")
            top = dd.iloc[0]
            if pd.notna(top["share_pct"]) and top["share_pct"] > 50:
                SIG.append(f"[C][CONCENTRATED] {m} | {t}: {dr['drug']} risk is dominated by a single "
                           f"receptor ({top['gene']}, {fmt(top['share_pct'],1)}%).")

    SUBSEC("EFFECT OF THE TWAS LAYER ON ORDERING (with vs without exp scaling)")
    eff = []
    for (m, t), grp in DRV.groupby(["method","trait"]):
        agg = grp.groupby("drug").agg(with_twas=("contribution","sum"),
                                      without_twas=("contribution_no_twas","sum")).reset_index()
        agg = agg[agg["without_twas"] > 0]
        if len(agg) < 4: continue
        rho, p = (spearmanr(agg["with_twas"].rank(ascending=False),
                            agg["without_twas"].rank(ascending=False)) if HAVE_SCIPY else (np.nan, np.nan))
        agg["d"] = (agg["without_twas"].rank(ascending=False, method="min")
                    - agg["with_twas"].rank(ascending=False, method="min"))
        nmov = int((agg["d"] != 0).sum())
        eff.append({"method": m, "trait": t, "n": len(agg), "rho": rho, "p": p,
                    "n_reranked": nmov, "max_abs_rank_change": int(agg["d"].abs().max()),
                    "median_amplification": float((agg["with_twas"]/agg["without_twas"]).median())})
        S(f"  {m:<24}{t:<16} rho={fmt(rho)} {stars(p)}  re-ranked={nmov}/{len(agg)}  "
          f"max|dRank|={int(agg['d'].abs().max())}  median amplification x"
          f"{fmt((agg['with_twas']/agg['without_twas']).median(),4)}")
        if nmov:
            SIG.append(f"[C] {m} | {t}: the exponential TWAS layer re-ranks {nmov}/{len(agg)} drugs "
                       f"(max |Δrank|={int(agg['d'].abs().max())}, rho={fmt(rho)}).")
    if eff: save_table(pd.DataFrame(eff), "05_twas_layer_effect_on_ordering.csv",
                       "Ordering with vs without the TWAS exponential scale")


# =============================================================================
# 9. STAGE 3 — TRANSDIAGNOSTIC SYNTHESIS
# =============================================================================
print("\n" + "#"*100); print("# STAGE 3 — TRANSDIAGNOSTIC SYNTHESIS"); print("#"*100)

# ---------------- S1 amplification index -------------------------------------
SEC("S1. TRANSDIAGNOSTIC GENOMIC-AMPLIFICATION INDEX (weighted -> uniform)")
S("rank_delta(trait) = rank_weighted - rank_uniform  (POSITIVE = drug RISES when the")
S("literature prior is removed, i.e. it is amplified by the genomic/affinity layer).")
AMP = None; DELTA_LONG = None
if HAVE_PAIR:
    parts = []
    for t in TRAITS:
        w = LONG[(LONG.method == WMETHOD) & (LONG.trait == t)][["drug_key","drug","risk_rank","risk_score"]]
        u = LONG[(LONG.method == UMETHOD) & (LONG.trait == t)][["drug_key","risk_rank","risk_score"]]
        mg = w.merge(u, on="drug_key", suffixes=("_w","_u"))
        if mg.empty: continue
        mg["trait"] = t
        mg["rank_delta"] = mg["risk_rank_w"] - mg["risk_rank_u"]
        mg["log2_score_ratio"] = np.log2((mg["risk_score_u"]/mg["risk_score_w"]).replace(0, np.nan))
        parts.append(mg)
    if parts:
        DELTA_LONG = pd.concat(parts, ignore_index=True)
        save_table(DELTA_LONG, "S1a_rank_delta_long.csv", "Per drug x trait rank delta")
        g = DELTA_LONG.groupby(["drug_key","drug"])
        AMP = g["rank_delta"].agg(mean_delta="mean", sd_delta="std", min_delta="min",
                                  max_delta="max", n_traits="count").reset_index()
        AMP["median_log2_score_ratio"] = g["log2_score_ratio"].median().values
        AMP["n_pos"] = g["rank_delta"].apply(lambda x: int((x > 0).sum())).values
        AMP["n_neg"] = g["rank_delta"].apply(lambda x: int((x < 0).sum())).values
        AMP["sign_test_p"] = [binom_p(max(r.n_pos, r.n_neg), r.n_pos + r.n_neg) for r in AMP.itertuples()]
        AMP["consistent_amplifier"] = (AMP["min_delta"] > 0) & (AMP["n_traits"] == len(TRAITS))
        AMP["consistent_dominant"]  = (AMP["max_delta"] < 0) & (AMP["n_traits"] == len(TRAITS))
        AMP["direction"] = np.where(AMP["consistent_amplifier"], "GENOMICALLY AMPLIFIED (all traits)",
                            np.where(AMP["consistent_dominant"], "PRIOR-DOMINANT (all traits)",
                            np.where(AMP["mean_delta"] > 0, "amplified on average",
                            np.where(AMP["mean_delta"] < 0, "prior-dominant on average", "stable"))))
        AMP = AMP.sort_values("mean_delta", ascending=False).reset_index(drop=True)
        save_table(AMP, "S1b_genomic_amplification_index.csv", "Transdiagnostic amplification index")
        S("")
        S(f"{'drug':<28}{'mean d':>9}{'sd':>8}{'min':>7}{'max':>7}{'+/-':>8}{'signP':>10}{'sig':>6}  direction")
        S(SUB)
        for _, r in AMP.iterrows():
            S(f"{str(r['drug'])[:27]:<28}{fmt(r['mean_delta'],2):>9}{fmt(r['sd_delta'],2):>8}"
              f"{fmt(r['min_delta'],0):>7}{fmt(r['max_delta'],0):>7}"
              f"{str(int(r['n_pos']))+'/'+str(int(r['n_neg'])):>8}{fmt(r['sign_test_p']):>10}"
              f"{stars(r['sign_test_p']):>6}  {r['direction']}")
        ca, cd = AMP[AMP.consistent_amplifier], AMP[AMP.consistent_dominant]
        S(""); S("Consistent amplifiers (rise in ALL traits): " +
                 (", ".join(f"{r['drug']} (+{fmt(r['mean_delta'],1)})" for _, r in ca.iterrows()) or "none"))
        S("Consistent prior-dominants (fall in ALL traits): " +
          (", ".join(f"{r['drug']} ({fmt(r['mean_delta'],1)})" for _, r in cd.iterrows()) or "none"))
        for _, r in ca.iterrows():
            SIG.append(f"[S1] {r['drug']} is TRANSDIAGNOSTICALLY GENOMICS-AMPLIFIED: rises in all "
                       f"{int(r['n_traits'])} traits without the prior (mean Δrank +{fmt(r['mean_delta'],2)}, "
                       f"sign-test p={fmt(r['sign_test_p'])}).")
        for _, r in cd.iterrows():
            SIG.append(f"[S1] {r['drug']} is PRIOR-DEPENDENT: falls in all {int(r['n_traits'])} traits "
                       f"without the literature weights (mean Δrank {fmt(r['mean_delta'],2)}).")
        for _, r in AMP[(AMP.n_traits == len(TRAITS)) & (AMP.sd_delta >= 2)].iterrows():
            SIG.append(f"[S1][HETEROGENEOUS] {r['drug']} shows trait-dependent amplification "
                       f"(mean {fmt(r['mean_delta'],2)}, SD {fmt(r['sd_delta'],2)}).")
        SUBSEC("MAGNITUDE OF THE WEIGHTED -> UNIFORM SHIFT, BY TRAIT")
        S(f"{'trait':<16}{'n':>5}{'mean|d|':>10}{'median|d|':>11}{'max|d|':>9}{'n moved':>9}")
        for t, grp in DELTA_LONG.groupby("trait"):
            S(f"{t:<16}{len(grp):>5}{fmt(grp['rank_delta'].abs().mean(),2):>10}"
              f"{fmt(grp['rank_delta'].abs().median(),2):>11}{fmt(grp['rank_delta'].abs().max(),0):>9}"
              f"{int((grp['rank_delta']!=0).sum()):>9}")
        worst = DELTA_LONG.groupby("trait")["rank_delta"].apply(lambda x: x.abs().mean()).sort_values()
        if len(worst):
            SIG.append(f"[S1] Weighting matters LEAST for {worst.index[0]} (mean |Δrank|="
                       f"{fmt(worst.iloc[0],2)}) and MOST for {worst.index[-1]} ({fmt(worst.iloc[-1],2)}).")
else:
    S("Both a weighted and a uniform run are required — section skipped.")

# ---------------- S1b 5-HT adaptation shift ----------------------------------
if AMETHOD:
    SEC("S1b. CHRONIC 5-HT ADAPTATION: RANK SHIFT (weighted -> weighted_5ht_adapted)")
    S(f"Adaptation fires when SERT affinity_dose_score > {SERT_THRESHOLD}: "
      f"HTR2C +{HTR2C_INCREASE}, HTR2A +{HTR2A_INCREASE}, HTR1A -{HTR1A_REDUCTION} (floor {HTR1A_FLOOR}).")
    frames = []
    for t, cmp_ in ADAPT_TABLES.items():
        c = cmp_.copy(); c["trait"] = t; frames.append(c)
    if frames:
        ADAPT_LONG = pd.concat(frames, ignore_index=True)
        save_table(ADAPT_LONG, "S1c_5ht_adaptation_long.csv", "5-HT adaptation vs base, all traits")
        agg = (ADAPT_LONG.groupby("drug")
               .agg(mean_delta=("adaptation_delta","mean"), sd_delta=("adaptation_delta","std"),
                    mean_rank_gain=("rank_delta","mean"), adapted=("adapted","max"),
                    ssri=("is_ssri_snri","max"), n=("trait","nunique"))
               .reset_index().sort_values("mean_delta", ascending=False))
        save_table(agg, "S1d_5ht_adaptation_summary.csv", "Mean 5-HT adaptation effect per drug")
        S(f"{'drug':<28}{'mean Δrisk':>12}{'sd':>8}{'mean rank gain':>16}{'adapted':>9}{'SSRI/SNRI':>11}")
        S(SUB)
        for _, r in agg.iterrows():
            S(f"{str(r['drug'])[:27]:<28}{fmt(r['mean_delta'],2):>12}{fmt(r['sd_delta'],2):>8}"
              f"{fmt(r['mean_rank_gain'],2):>16}{str(bool(r['adapted'])):>9}{str(bool(r['ssri'])):>11}")
        up = agg[(agg["mean_rank_gain"] > 0) & (agg["adapted"] == True)]
        if len(up):
            SIG.append("[S1b] Chronic 5-HT adaptation raises the metabolic rank of: " +
                       ", ".join(f"{r['drug']} (+{fmt(r['mean_rank_gain'],1)} ranks, "
                                 f"Δrisk {fmt(r['mean_delta'],1)})" for _, r in up.head(8).iterrows()) + ".")
        ss = agg[agg["ssri"] == True]
        if len(ss):
            SIG.append(f"[S1b] Mean adaptation Δrisk among SSRIs/SNRIs = "
                       f"{fmt(ss['mean_delta'].mean(),2)} points (n={len(ss)}), i.e. acute Ki-only "
                       f"scoring UNDER-estimates their long-term metabolic liability.")

# ---------------- S2 cross-trait stability -----------------------------------
SEC("S2. CROSS-TRAIT STABILITY WITHIN EACH METHOD")
STAB = {}
for m in METHODS:
    piv = LONG[LONG.method == m].pivot_table(index="drug_key", columns="trait", values="risk_rank").dropna(how="any")
    if piv.shape[0] < 3 or piv.shape[1] < 2: continue
    W, chi, dfree, pW = kendalls_w(piv.values)
    tab = pd.DataFrame({"drug_key": piv.index, "drug": [DRUG_LABEL[i] for i in piv.index],
                        "mean_rank": piv.mean(axis=1).values, "sd_rank": piv.std(axis=1).values,
                        "rank_range": (piv.max(axis=1)-piv.min(axis=1)).values,
                        "most_implicated_trait": piv.idxmin(axis=1).values,
                        "least_implicated_trait": piv.idxmax(axis=1).values}
                       ).sort_values("mean_rank").reset_index(drop=True)
    STAB[m] = {"tab": tab, "piv": piv, "W": W, "chi2": chi, "df": dfree, "p": pW}
    save_table(tab, f"S2_stability_{re.sub(r'[^A-Za-z0-9]+','_',m)}.csv", f"Cross-trait stability {m}")
S(f"{'method':<26}{'drugs':>7}{'Kendall W':>11}{'chi2':>10}{'df':>5}{'p':>12}{'sig':>6}{'mean rank SD':>14}")
S(SUB)
for m, d in STAB.items():
    S(f"{m:<26}{d['tab'].shape[0]:>7}{fmt(d['W']):>11}{fmt(d['chi2'],1):>10}{d['df']:>5}"
      f"{fmt(d['p']):>12}{stars(d['p']):>6}{fmt(d['tab']['sd_rank'].mean(),3):>14}")
    if not np.isnan(d["p"]) and d["p"] < ALPHA:
        SIG.append(f"[S2] {m}: the {len(TRAITS)} trait-specific rankings are significantly concordant "
                   f"(Kendall's W={fmt(d['W'])}, p={fmt(d['p'])} {stars(d['p'])}) -> largely transdiagnostic.")
    S("  pairwise trait rank correlations:")
    for t1, t2 in itertools.combinations(d["piv"].columns, 2):
        rho, p = (spearmanr(d["piv"][t1], d["piv"][t2]) if HAVE_SCIPY else (np.nan, np.nan))
        S(f"    {t1:<16}{t2:<16}rho={fmt(rho):>8}  p={fmt(p):>11} {stars(p)}")
        if not np.isnan(p) and p < ALPHA and rho < 0.8:
            SIG.append(f"[S2] {m}: {t1} vs {t2} only moderately correlated (rho={fmt(rho)}) "
                       f"-> trait-specific genomic structure.")

if HAVE_PAIR and WMETHOD in STAB and UMETHOD in STAB:
    a = STAB[WMETHOD]["tab"][["drug_key","drug","sd_rank","mean_rank"]]
    b = STAB[UMETHOD]["tab"][["drug_key","sd_rank","mean_rank"]]
    cm = a.merge(b, on="drug_key", suffixes=("_weighted","_uniform"))
    cm["sd_diff"] = cm["sd_rank_uniform"] - cm["sd_rank_weighted"]
    cm["more_stable_under"] = np.where(cm["sd_diff"] > 0, "weighted",
                               np.where(cm["sd_diff"] < 0, "uniform", "tie"))
    save_table(cm, "S2_stability_method_comparison.csv", "Per-drug rank SD: weighted vs uniform")
    pw = np.nan
    if HAVE_SCIPY and cm["sd_diff"].abs().sum() > 0:
        try: pw = float(wilcoxon(cm["sd_rank_weighted"], cm["sd_rank_uniform"])[1])
        except Exception: pw = np.nan
    SUBSEC("PAIRED COMPARISON OF CROSS-TRAIT RANK VARIABILITY")
    S(f"Mean rank SD weighted (clinical proxy)  : {fmt(cm['sd_rank_weighted'].mean(),3)}")
    S(f"Mean rank SD uniform (genomics-driven)  : {fmt(cm['sd_rank_uniform'].mean(),3)}")
    S(f"Drugs more stable under weighted        : {int((cm['more_stable_under']=='weighted').sum())}/{len(cm)}")
    S(f"Wilcoxon signed-rank p                  : {fmt(pw)} {stars(pw)}")
    if not np.isnan(pw) and pw < ALPHA:
        direction = ("the LITERATURE PRIOR STABILISES ranks across diseases"
                     if cm['sd_rank_weighted'].mean() < cm['sd_rank_uniform'].mean()
                     else "the GENOMICS-INFLUENCED ranking is the more stable one")
        SIG.append(f"[S2] Cross-trait rank variability differs significantly between methods "
                   f"(Wilcoxon p={fmt(pw)} {stars(pw)}) -> {direction}.")
    else:
        SIG.append(f"[S2] No significant difference in cross-trait rank stability between the "
                   f"clinical-proxy and genomics-influenced rankings (Wilcoxon p={fmt(pw)}).")
    S(""); S(f"{'drug':<28}{'SD weighted':>13}{'SD uniform':>12}{'diff':>9}  more stable under")
    for _, r in cm.sort_values("sd_diff", ascending=False).iterrows():
        S(f"{str(r['drug'])[:27]:<28}{fmt(r['sd_rank_weighted'],2):>13}"
          f"{fmt(r['sd_rank_uniform'],2):>12}{fmt(r['sd_diff'],2):>9}  {r['more_stable_under']}")

# ---------------- S3 transdiagnostic top-k sets -------------------------------
SEC("S3. TRANSDIAGNOSTIC TOP-K SETS (robust vs amplified vs prior-driven)")
SETS = {}
for m in METHODS:
    piv = LONG[LONG.method == m].pivot_table(index="drug_key", columns="trait", values="risk_rank").dropna(how="any")
    if piv.empty: continue
    mr = piv.mean(axis=1).sort_values(); SETS[m] = list(mr.index[:TOP_K])
    SUBSEC(f"TRANSDIAGNOSTIC TOP {TOP_K} — {m}")
    S(f"{'#':>3}  {'drug':<30}{'mean rank across traits':>26}")
    for i, k in enumerate(mr.index[:TOP_K], 1):
        S(f"{i:>3}  {DRUG_LABEL[k][:29]:<30}{fmt(mr[k],2):>26}")
if HAVE_PAIR and WMETHOD in SETS and UMETHOD in SETS:
    A, B = set(SETS[WMETHOD]), set(SETS[UMETHOD])
    inter, only_u, only_w = A & B, B - A, A - B
    N = len(set(RANK_WIDE.index))
    p_over = float(hypergeom.sf(len(inter)-1, N, TOP_K, TOP_K)) if (HAVE_SCIPY and N >= TOP_K) else np.nan
    jac = len(inter)/len(A | B) if (A | B) else np.nan
    SUBSEC("SET ALGEBRA OF THE TWO TRANSDIAGNOSTIC TOP-K LISTS")
    S(f"Overlap {len(inter)}/{TOP_K}   Jaccard={fmt(jac,2)}   hypergeometric p={fmt(p_over)} {stars(p_over)}")
    S("ROBUST DUAL-EVIDENCE SET  : " + ", ".join(sorted(DRUG_LABEL[k] for k in inter)))
    S("GENOMICALLY AMPLIFIED SET : " + (", ".join(sorted(DRUG_LABEL[k] for k in only_u)) or "none"))
    S("PRIOR-DEPENDENT SET       : " + (", ".join(sorted(DRUG_LABEL[k] for k in only_w)) or "none"))
    save_table(pd.DataFrame({"drug": [DRUG_LABEL[k] for k in sorted(A | B)],
                             "in_weighted_topk": [k in A for k in sorted(A | B)],
                             "in_uniform_topk": [k in B for k in sorted(A | B)],
                             "category": ["robust (both)" if (k in A and k in B) else
                                          "genomically amplified (uniform only)" if k in B else
                                          "prior-dependent (weighted only)" for k in sorted(A | B)]}),
               "S3_transdiagnostic_topk_sets.csv", "Top-k set membership")
    SIG.append(f"[S3] Robust transdiagnostic high-risk set (top-{TOP_K} under BOTH rankings, "
               f"overlap {len(inter)}/{TOP_K}, hypergeom p={fmt(p_over)}): "
               + ", ".join(sorted(DRUG_LABEL[k] for k in inter)) + ".")
    if only_u: SIG.append("[S3] Genomically amplified candidates (top-k only without the prior): "
                          + ", ".join(sorted(DRUG_LABEL[k] for k in only_u)) + ".")
    if only_w: SIG.append("[S3] Prior-dependent candidates (top-k only with the literature weights): "
                          + ", ".join(sorted(DRUG_LABEL[k] for k in only_w)) + ".")

# ---------------- S4 receptor driver contrast ---------------------------------
SEC("S4. RECEPTOR-DRIVER CONTRAST: WHICH GENES GAIN SHARE WITHOUT THE PRIOR?")
GENE_DELTA = None
if DRV is not None and HAVE_PAIR:
    pooled = DRV[DRV.method.isin([WMETHOD, UMETHOD])] \
                .groupby(["method","trait","gene"])["contribution"].sum().reset_index()
    pooled["share"] = pooled.groupby(["method","trait"])["contribution"].transform(lambda x: 100*x/x.sum())
    pw_ = pooled.pivot_table(index=["gene","trait"], columns="method", values="share").reset_index().fillna(0)
    if {WMETHOD, UMETHOD}.issubset(pw_.columns):
        pw_["delta_share"] = pw_[UMETHOD] - pw_[WMETHOD]
        GENE_DELTA = (pw_.groupby("gene").agg(mean_weighted=(WMETHOD,"mean"),
                        mean_uniform=(UMETHOD,"mean"), mean_delta=("delta_share","mean"),
                        sd_delta=("delta_share","std"), min_delta=("delta_share","min"),
                        max_delta=("delta_share","max"), n_traits=("trait","nunique")).reset_index())
        GENE_DELTA["metabolic_weight"] = GENE_DELTA["gene"].map(METABOLIC_WEIGHTS)
        GENE_DELTA["consistent_gain"] = GENE_DELTA["min_delta"] > 0
        GENE_DELTA["consistent_loss"] = GENE_DELTA["max_delta"] < 0
        GENE_DELTA = GENE_DELTA.sort_values("mean_delta", ascending=False).reset_index(drop=True)
        save_table(GENE_DELTA, "S4a_gene_share_delta.csv", "Share change when the prior is removed")
        save_table(pw_, "S4b_gene_share_by_trait.csv", "Gene share per trait and method")
        S("delta = uniform - weighted (positive = the gene GAINS influence without the prior).")
        S("")
        S(f"{'gene':<10}{'w share%':>10}{'u share%':>10}{'delta':>9}{'sd':>8}{'all traits':>12}{'lit w':>8}")
        S(SUB)
        for _, r in GENE_DELTA.iterrows():
            flag = "GAIN(all)" if r["consistent_gain"] else ("LOSS(all)" if r["consistent_loss"] else "")
            S(f"{str(r['gene'])[:9]:<10}{fmt(r['mean_weighted'],2):>10}{fmt(r['mean_uniform'],2):>10}"
              f"{fmt(r['mean_delta'],2):>9}{fmt(r['sd_delta'],2):>8}{flag:>12}{fmt(r['metabolic_weight'],2):>8}")
        for _, r in GENE_DELTA[GENE_DELTA.consistent_gain].head(6).iterrows():
            SIG.append(f"[S4] {r['gene']} gains influence in EVERY trait without the prior "
                       f"(+{fmt(r['mean_delta'],2)} share points; literature weight "
                       f"{fmt(r['metabolic_weight'],2)}) -> genomics-driven driver.")
        for _, r in GENE_DELTA[GENE_DELTA.consistent_loss].tail(6).iterrows():
            SIG.append(f"[S4] {r['gene']} loses influence in EVERY trait without the prior "
                       f"({fmt(r['mean_delta'],2)} share points) -> prior-driven driver.")
        if len(GENE_DELTA):
            SIG.append(f"[S4] Dominant driver switches from "
                       f"{GENE_DELTA.sort_values('mean_weighted', ascending=False).iloc[0]['gene']} "
                       f"(clinical proxy) to "
                       f"{GENE_DELTA.sort_values('mean_uniform', ascending=False).iloc[0]['gene']} "
                       f"(genomics-influenced).")
    if AMP is not None:
        movers = list(AMP.reindex(AMP["mean_delta"].abs().sort_values(ascending=False).index)
                        ["drug_key"].head(KEY_MOVERS_N))
        SUBSEC("PER-DRUG RECEPTOR-SHARE CONTRAST FOR THE LARGEST MOVERS")
        mrows = []
        for dk in movers:
            sub = DRV[(DRV.drug_key == dk) & (DRV.method.isin([WMETHOD, UMETHOD]))]
            if sub.empty: continue
            piv = sub.pivot_table(index="gene", columns="method", values="share_pct", aggfunc="mean").fillna(0)
            if not {WMETHOD, UMETHOD}.issubset(piv.columns): continue
            piv["delta"] = piv[UMETHOD] - piv[WMETHOD]; piv = piv.sort_values("delta", ascending=False)
            ar = AMP[AMP.drug_key == dk]
            S(""); S(f"  {DRUG_LABEL[dk]}   mean Δrank = "
                     f"{fmt(ar.iloc[0]['mean_delta'],2) if len(ar) else 'NA'}  "
                     f"({ar.iloc[0]['direction'] if len(ar) else ''})")
            S(f"    {'gene':<10}{'weighted %':>12}{'uniform %':>11}{'delta':>9}")
            for gname, r in piv.head(6).iterrows():
                S(f"    {str(gname)[:9]:<10}{fmt(r[WMETHOD],1):>12}{fmt(r[UMETHOD],1):>11}{fmt(r['delta'],1):>9}")
            for gname, r in piv.tail(3).iterrows():
                S(f"    {str(gname)[:9]:<10}{fmt(r[WMETHOD],1):>12}{fmt(r[UMETHOD],1):>11}{fmt(r['delta'],1):>9}")
            mrows.append({"drug": DRUG_LABEL[dk], "top_gaining_gene": piv.index[0],
                          "gain_delta": piv.iloc[0]["delta"], "top_losing_gene": piv.index[-1],
                          "loss_delta": piv.iloc[-1]["delta"]})
            SIG.append(f"[S4][DRUG] {DRUG_LABEL[dk]}: share shifts toward {piv.index[0]} "
                       f"(+{fmt(piv.iloc[0]['delta'],1)} pts) and away from {piv.index[-1]} "
                       f"({fmt(piv.iloc[-1]['delta'],1)} pts) without the prior.")
        if mrows: save_table(pd.DataFrame(mrows), "S4c_mover_driver_contrast.csv",
                             "Top gaining / losing receptor per major mover")
else:
    S("Contribution data or the method pair unavailable — section skipped.")

# ---------------- S5 residuals vs literature ----------------------------------
SEC("S5. RESIDUAL ANALYSIS AGAINST AN ORDINAL LITERATURE WEIGHT-GAIN ORDERING")
S("residual = model_rank - literature_rank, both re-ranked within the matched subset.")
S("NEGATIVE residual = the model calls the drug HIGHER risk than the literature does.")
RESID_SUM, resid_rows, VAL = {}, [], None
if ENABLE_EXTERNAL_VALIDATION and HAVE_SCIPY:
    val_rows = []
    for m in METHODS:
        pts = []
        for t in TRAITS:
            d = LONG[(LONG.method == m) & (LONG.trait == t)].copy()
            d["lit"] = d["drug_key"].map(LITERATURE_WEIGHT_GAIN_RANK)
            d = d.dropna(subset=["lit","risk_rank"])
            if len(d) < 5: continue
            d["model_r"] = d["risk_rank"].rank(method="average")
            d["lit_r"] = d["lit"].rank(method="average")
            d["residual"] = d["model_r"] - d["lit_r"]
            rho, p = spearmanr(d["model_r"], d["lit_r"])
            lo, hi = boot_ci(d["model_r"], d["lit_r"])
            d["rho_trait"], d["p_trait"] = rho, p
            pts.append(d[["method","trait","drug","drug_key","model_r","lit_r","residual","rho_trait","p_trait"]])
            val_rows.append({"method": m, "trait": t, "n_matched": len(d),
                             "spearman_rho": rho, "p": p, "CI_lo": lo, "CI_hi": hi})
        if not pts: continue
        R = pd.concat(pts, ignore_index=True); resid_rows.append(R)
        summ = (R.groupby(["drug_key","drug"])["residual"]
                  .agg(mean_resid="mean", sd_resid="std", n="count").reset_index())
        summ["mean_abs_resid"] = R.groupby(["drug_key","drug"])["residual"].apply(lambda x: x.abs().mean()).values
        RESID_SUM[m] = {"summary": summ.sort_values("mean_resid"),
                        "mean_rho": R.drop_duplicates(["trait"])["rho_trait"].mean(),
                        "n_matched": int(R["drug_key"].nunique())}
        save_table(RESID_SUM[m]["summary"], f"S5_residuals_{re.sub(r'[^A-Za-z0-9]+','_',m)}.csv",
                   f"Residual vs literature for {m}")
    if val_rows:
        VAL = pd.DataFrame(val_rows)
        save_table(VAL, "S5_external_validation.csv", "Concordance with the literature ordering")
        S(""); S(f"{'method':<26}{'trait':<16}{'n':>4}{'rho':>9}{'p':>12}{'sig':>6}{'95% CI':>22}")
        S(SUB)
        for _, r in VAL.iterrows():
            S(f"{r['method']:<26}{r['trait']:<16}{int(r['n_matched']):>4}{fmt(r['spearman_rho']):>9}"
              f"{fmt(r['p']):>12}{stars(r['p']):>6}"
              f"{'['+fmt(r['CI_lo'],2)+', '+fmt(r['CI_hi'],2)+']':>22}")
            if r["p"] < ALPHA:
                SIG.append(f"[S5] {r['method']} | {r['trait']}: ranking is significantly "
                           f"{'CONCORDANT' if r['spearman_rho']>0 else 'DISCORDANT'} with the "
                           f"literature weight-gain order (rho={fmt(r['spearman_rho'])}, "
                           f"p={fmt(r['p'])} {stars(r['p'])}, n={int(r['n_matched'])}).")
        S("")
        for m, grp in VAL.groupby("method"):
            S(f"  mean rho for {m}: {fmt(grp['spearman_rho'].mean())} (over {len(grp)} traits)")
        best = VAL.loc[VAL["spearman_rho"].idxmax()]
        SIG.append(f"[S5] Best external agreement: {best['method']} | {best['trait']} "
                   f"(rho={fmt(best['spearman_rho'])}).")
    for m, d in RESID_SUM.items():
        SUBSEC(f"LARGEST SYSTEMATIC DEVIATIONS — {m}")
        s_ = d["summary"]
        S("  Model ranks HIGHER RISK than literature (negative residual):")
        S(f"    {'drug':<28}{'mean resid':>12}{'sd':>8}{'n':>5}")
        for _, r in s_.head(8).iterrows():
            S(f"    {str(r['drug'])[:27]:<28}{fmt(r['mean_resid'],2):>12}{fmt(r['sd_resid'],2):>8}{int(r['n']):>5}")
        S("  Model ranks LOWER RISK than literature (positive residual):")
        for _, r in s_.tail(8).iloc[::-1].iterrows():
            S(f"    {str(r['drug'])[:27]:<28}{fmt(r['mean_resid'],2):>12}{fmt(r['sd_resid'],2):>8}{int(r['n']):>5}")
        for _, r in s_[s_["mean_resid"].abs() >= 4].iterrows():
            SIG.append(f"[S5] {m}: {r['drug']} is systematically ranked "
                       f"{'HIGHER' if r['mean_resid']<0 else 'LOWER'} risk than the literature "
                       f"(mean residual {fmt(r['mean_resid'],2)} ranks).")
    if HAVE_PAIR and WMETHOD in RESID_SUM and UMETHOD in RESID_SUM:
        a = RESID_SUM[WMETHOD]["summary"][["drug_key","drug","mean_abs_resid","mean_resid"]]
        b = RESID_SUM[UMETHOD]["summary"][["drug_key","mean_abs_resid","mean_resid"]]
        cmp_ = a.merge(b, on="drug_key", suffixes=("_weighted","_uniform"))
        pw = np.nan
        if len(cmp_) >= 6:
            try: pw = float(wilcoxon(cmp_["mean_abs_resid_weighted"], cmp_["mean_abs_resid_uniform"])[1])
            except Exception: pw = np.nan
        save_table(cmp_, "S5_calibration_method_comparison.csv", "Absolute residual: weighted vs uniform")
        SUBSEC("CALIBRATION AGAINST THE LITERATURE: WEIGHTED vs UNIFORM")
        S(f"Mean |residual| weighted : {fmt(cmp_['mean_abs_resid_weighted'].mean(),3)}")
        S(f"Mean |residual| uniform  : {fmt(cmp_['mean_abs_resid_uniform'].mean(),3)}")
        S(f"Wilcoxon signed-rank p   : {fmt(pw)} {stars(pw)}")
        better = ("clinical-proxy (weighted)" if cmp_['mean_abs_resid_weighted'].mean() <
                  cmp_['mean_abs_resid_uniform'].mean() else "genomics-influenced (uniform)")
        SIG.append(f"[S5] Calibration to the literature ordering is better under the {better} ranking "
                   f"(mean |residual| {fmt(cmp_['mean_abs_resid_weighted'].mean(),2)} vs "
                   f"{fmt(cmp_['mean_abs_resid_uniform'].mean(),2)}; Wilcoxon p={fmt(pw)}).")
else:
    S("External validation disabled or scipy unavailable — section skipped.")

# ---------------- S6 stability-weighted consensus ------------------------------
SEC("S6. STABILITY-WEIGHTED CONSENSUS RANKINGS")
S("weight = 1/(1 + SD_rank_across_traits);  weighted score = mean_rank * weight (lower = high & consistent).")
CONS = []
for m, d in STAB.items():
    t = d["tab"].copy()
    t["weight"] = 1.0/(1.0 + t["sd_rank"])
    t["stability_weighted_score"] = t["mean_rank"] * t["weight"]
    t["consensus_rank"] = t["stability_weighted_score"].rank(method="min").astype(int)
    t["plain_rank"] = t["mean_rank"].rank(method="min").astype(int)
    t["rank_shift_vs_plain"] = t["plain_rank"] - t["consensus_rank"]
    t = t.sort_values("consensus_rank"); t["method"] = m; CONS.append(t)
    save_table(t, f"S6_stability_weighted_{re.sub(r'[^A-Za-z0-9]+','_',m)}.csv", f"Consensus {m}")
    SUBSEC(f"STABILITY-WEIGHTED CONSENSUS — {m}")
    S(f"{'#':>3}  {'drug':<28}{'mean rank':>11}{'SD':>7}{'weight':>8}{'w.score':>10}{'plain #':>9}{'shift':>7}")
    for _, r in t.head(TOP_K+5).iterrows():
        S(f"{int(r['consensus_rank']):>3}  {str(r['drug'])[:27]:<28}{fmt(r['mean_rank'],2):>11}"
          f"{fmt(r['sd_rank'],2):>7}{fmt(r['weight'],3):>8}{fmt(r['stability_weighted_score'],3):>10}"
          f"{int(r['plain_rank']):>9}{int(r['rank_shift_vs_plain']):>7}")
    for _, r in t[t["rank_shift_vs_plain"].abs() >= 2].head(6).iterrows():
        SIG.append(f"[S6] {m}: {r['drug']} shifts {int(r['rank_shift_vs_plain']):+d} positions when "
                   f"cross-trait instability is penalised (SD={fmt(r['sd_rank'],2)}).")
if CONS: save_table(pd.concat(CONS, ignore_index=True), "S6_stability_weighted_all_methods.csv",
                    "Stability-weighted consensus, all methods")

# ---------------- S7 run similarity --------------------------------------------
SEC("S7. SIMILARITY STRUCTURE OF ALL RUNS (method vs trait)")
CORR = None
if COMPLETE.shape[0] >= 5 and COMPLETE.shape[1] >= 2 and HAVE_SCIPY:
    lbl = [f"{a}|{b}" for a, b in COMPLETE.columns]; M = COMPLETE.values; k = M.shape[1]
    C = np.ones((k, k))
    for i in range(k):
        for j in range(i+1, k): C[i, j] = C[j, i] = spearmanr(M[:, i], M[:, j])[0]
    CORR = pd.DataFrame(C, index=lbl, columns=lbl)
    save_table(CORR.reset_index().rename(columns={"index":"run"}),
               "S7a_run_rank_correlation_matrix.csv", "Spearman rho between runs")
    S("        " + "".join(f"{c[:14]:>16}" for c in CORR.columns))
    for i, row in CORR.iterrows():
        S(f"{i[:14]:<16}" + "".join(f"{fmt(v,3):>16}" for v in row.values))
    meth_of = [c.split("|")[0] for c in lbl]; trait_of = [c.split("|")[1] for c in lbl]
    wm, bm, wt, bt = [], [], [], []
    for i in range(k):
        for j in range(i+1, k):
            (wm if meth_of[i] == meth_of[j] else bm).append(C[i, j])
            (wt if trait_of[i] == trait_of[j] else bt).append(C[i, j])
    p_m = float(mannwhitneyu(wm, bm, alternative="two-sided").pvalue) if (wm and bm) else np.nan
    p_t = float(mannwhitneyu(wt, bt, alternative="two-sided").pvalue) if (wt and bt) else np.nan
    SUBSEC("WHAT DRIVES RUN SIMILARITY — METHOD OR DISEASE?")
    S(f"mean rho SAME method      : {fmt(np.mean(wm)) if wm else 'NA'} (n={len(wm)})")
    S(f"mean rho DIFFERENT method : {fmt(np.mean(bm)) if bm else 'NA'} (n={len(bm)})   MWU p={fmt(p_m)} {stars(p_m)}")
    S(f"mean rho SAME trait       : {fmt(np.mean(wt)) if wt else 'NA'} (n={len(wt)})")
    S(f"mean rho DIFFERENT trait  : {fmt(np.mean(bt)) if bt else 'NA'} (n={len(bt)})   MWU p={fmt(p_t)} {stars(p_t)}")
    if wm and bm:
        SIG.append(f"[S7] Runs sharing the SAME WEIGHTING SCHEME are more similar (mean rho "
                   f"{fmt(np.mean(wm))}) than runs sharing the same disease "
                   f"(within-trait {fmt(np.mean(wt)) if wt else 'NA'}; MWU p={fmt(p_m)}) "
                   f"-> the weighting choice dominates the trait choice.")
    try:
        D = 1 - CORR.values; np.fill_diagonal(D, 0.0); D = (D + D.T)/2
        Z = linkage(squareform(D, checks=False), method="average")
        coph = cophenet(Z, squareform(D, checks=False))[0]
        grp = fcluster(Z, min(2, len(lbl)-1) if len(lbl) > 2 else 2, criterion="maxclust")
        ct = pd.DataFrame({"run": lbl, "cluster": grp, "method": meth_of, "trait": trait_of})
        save_table(ct, "S7b_run_clusters.csv", "Cluster solution over runs")
        S(f"\nCophenetic correlation of the run dendrogram: {fmt(coph)}")
        for c in sorted(set(grp)):
            S(f"  Run-cluster {c}: " + ", ".join(ct[ct.cluster == c]["run"].tolist()))
        pure = all(ct.groupby("cluster")["method"].nunique() == 1)
        SIG.append(f"[S7] The cluster solution over runs is "
                   f"{'PERFECTLY separated by weighting scheme' if pure else 'NOT purely method-separated'} "
                   f"(cophenetic r={fmt(coph)}).")
    except Exception as e:
        S(f"[warn] dendrogram failed: {e}")
else:
    S("Insufficient complete data or scipy unavailable — section skipped.")

# ---------------- S8 variance decomposition ------------------------------------
SEC("S8. VARIANCE DECOMPOSITION OF THE RANKS: DRUG vs TRAIT vs METHOD")
bal = LONG.merge(pd.DataFrame({"drug_key": COMPLETE.index}), on="drug_key", how="inner").dropna(subset=["risk_rank"])
if bal["drug_key"].nunique() >= 4 and len(bal) >= 12:
    grand = bal["risk_rank"].mean(); SST = float(((bal["risk_rank"]-grand)**2).sum()); ss = {}
    for f in ["drug_key","trait","method"]:
        g = bal.groupby(f)["risk_rank"]; ss[f] = float((g.count()*(g.mean()-grand)**2).sum())
    ss["residual/interaction"] = SST - sum(ss.values())
    VD = pd.DataFrame({"source": ["drug","trait","method","residual/interaction"],
                       "SS": [ss["drug_key"], ss["trait"], ss["method"], ss["residual/interaction"]]})
    VD["pct_of_total"] = 100*VD["SS"]/SST if SST > 0 else np.nan
    save_table(VD, "S8_variance_decomposition.csv", "Rank variance decomposition")
    S(f"Complete-case design: {bal['drug_key'].nunique()} drugs x {bal['trait'].nunique()} traits "
      f"x {bal['method'].nunique()} methods (SS total = {fmt(SST,1)})")
    S(""); S(f"{'source':<26}{'sum of squares':>18}{'% of total':>13}"); S(SUB)
    for _, r in VD.iterrows(): S(f"{r['source']:<26}{fmt(r['SS'],1):>18}{fmt(r['pct_of_total'],1):>13}")
    pd_drug = float(VD.loc[VD.source == "drug","pct_of_total"].iloc[0])
    pd_res  = float(VD.loc[VD.source == "residual/interaction","pct_of_total"].iloc[0])
    SIG.append(f"[S8] Drug identity explains {fmt(pd_drug,1)}% of all rank variance; "
               f"{fmt(pd_res,1)}% is drug x trait / drug x method interaction -> the ordering is "
               f"predominantly TRANSDIAGNOSTIC with {fmt(pd_res,1)}% context-specific re-ordering.")
    S(""); S("NOTE: trait and method MAIN effects are structurally ~0 because every run's ranks are a")
    S("permutation of 1..n; only the drug effect and the interaction term are interpretable.")
else:
    S("Insufficient balanced data for variance decomposition.")

# ---------------- S9 trait specificity ------------------------------------------
SEC("S9. TRAIT-SPECIFICITY: WHICH DISEASE DRIVES EACH DRUG?")
TS_rows = []
for m, d in STAB.items():
    piv = d["piv"]
    zz = piv.sub(piv.mean(axis=1), axis=0).div(piv.std(axis=1).replace(0, np.nan), axis=0)
    for dk in piv.index:
        row = zz.loc[dk]
        if row.isna().all(): continue
        TS_rows.append({"method": m, "drug": DRUG_LABEL[dk],
                        "most_implicated_trait": row.idxmin(), "z_most": row.min(),
                        "least_implicated_trait": row.idxmax(), "z_least": row.max(),
                        "rank_range": float(piv.loc[dk].max()-piv.loc[dk].min())})
if TS_rows:
    TS = pd.DataFrame(TS_rows).sort_values(["method","rank_range"], ascending=[True, False])
    save_table(TS, "S9_trait_specificity_profiles.csv", "Trait-specificity per drug")
    for m, grp in TS.groupby("method"):
        SUBSEC(f"TRAIT-SPECIFICITY — {m}")
        S(f"{'drug':<28}{'most implicated':<16}{'z':>7}{'least implicated':<18}{'z':>7}{'rank range':>12}")
        for _, r in grp.head(15).iterrows():
            S(f"{str(r['drug'])[:27]:<28}{str(r['most_implicated_trait'])[:15]:<16}{fmt(r['z_most'],2):>7}"
              f"{str(r['least_implicated_trait'])[:17]:<18}{fmt(r['z_least'],2):>7}{fmt(r['rank_range'],0):>12}")
        for _, r in grp[grp["rank_range"] >= 3].head(6).iterrows():
            SIG.append(f"[S9] {m}: {r['drug']} is comparatively most implicated in "
                       f"{r['most_implicated_trait']} and least in {r['least_implicated_trait']} "
                       f"(rank range {int(r['rank_range'])}).")

# ---------------- S10 composite classification ----------------------------------
SEC("S10. COMPOSITE GENOMICS-SIGNAL SCORE AND FINAL CLASSIFICATION")
FINAL = None
if AMP is not None:
    base = AMP[["drug_key","drug","mean_delta","sd_delta","direction",
                "consistent_amplifier","consistent_dominant"]].copy()
    pooled_rank = (LONG[LONG.method.isin([WMETHOD, UMETHOD])].dropna(subset=["risk_rank"])
                   .groupby("drug_key")["risk_rank"]
                   .agg(pooled_mean_rank="mean", pooled_sd_rank="std", n_runs="count").reset_index())
    base = base.merge(pooled_rank, on="drug_key", how="left")
    for mm, cn in [(UMETHOD,"resid_uniform"), (WMETHOD,"resid_weighted")]:
        if mm in RESID_SUM:
            base = base.merge(RESID_SUM[mm]["summary"][["drug_key","mean_resid"]]
                              .rename(columns={"mean_resid": cn}), on="drug_key", how="left")
        else: base[cn] = np.nan
    if AMETHOD and AMETHOD in [k[0] for k in RISK]:
        ad = (LONG[LONG.method == AMETHOD].groupby("drug_key")["risk_rank"].mean()
              .rename("adapted_mean_rank").reset_index())
        base = base.merge(ad, on="drug_key", how="left")
        base["adaptation_rank_gain"] = base["pooled_mean_rank"] - base["adapted_mean_rank"]
    base["z_amplification"] = zscore(base["mean_delta"])
    base["z_resid_gain"] = zscore(-base["resid_uniform"].fillna(base["resid_uniform"].mean()))
    base["genomics_signal"] = base["z_amplification"] + base["z_resid_gain"]
    nq = min(3, max(2, base.shape[0]//3 or 2))
    base["risk_tier"] = pd.qcut(base["pooled_mean_rank"].rank(method="first"), q=nq,
                                labels=["HIGH","INTERMEDIATE","LOW"][:nq])
    def classify(r):
        hi = str(r["risk_tier"]) == "HIGH"
        if hi and r["consistent_amplifier"]: return "A. High-risk & genomically amplified (priority)"
        if hi and r["consistent_dominant"]:  return "B. High-risk but prior-dependent (verify)"
        if hi:                               return "C. High-risk, weighting-robust"
        if r["mean_delta"] > 0 and r["genomics_signal"] > 0:
            return "D. Emerging genomic signal (lower pharmacological rank)"
        return "E. Low priority"
    base["classification"] = base.apply(classify, axis=1)
    FINAL = base.sort_values(["pooled_mean_rank","genomics_signal"], ascending=[True, False]).reset_index(drop=True)
    save_table(FINAL, "S10_final_transdiagnostic_classification.csv", "Composite classification")
    S("genomics_signal = z(mean rank rise without prior) + z(-mean residual vs literature under uniform).")
    S("")
    hdr = (f"{'drug':<28}{'pooled rank':>12}{'meanΔ':>8}{'residU':>9}{'gSignal':>9}"
           + ("{:>10}".format("adaptΔ") if "adaptation_rank_gain" in FINAL.columns else "")
           + "  classification")
    S(hdr); S(SUB)
    for _, r in FINAL.iterrows():
        line = (f"{str(r['drug'])[:27]:<28}{fmt(r['pooled_mean_rank'],2):>12}{fmt(r['mean_delta'],1):>8}"
                f"{fmt(r['resid_uniform'],1):>9}{fmt(r['genomics_signal'],2):>9}")
        if "adaptation_rank_gain" in FINAL.columns:
            line += f"{fmt(r['adaptation_rank_gain'],1):>10}"
        S(line + f"  {r['classification']}")
    SUBSEC("CLASSIFICATION GROUPS")
    for c, grp in FINAL.groupby("classification"):
        S(f"  {c}: " + ", ".join(grp["drug"].tolist()))
        SIG.append(f"[S10] {c} -> " + ", ".join(grp["drug"].head(8).tolist()))
    top_sig = FINAL.sort_values("genomics_signal", ascending=False).head(5)
    SIG.append("[S10] Strongest composite genomics signal: " +
               ", ".join(f"{r['drug']} ({fmt(r['genomics_signal'],2)})" for _, r in top_sig.iterrows()) + ".")
    if HAVE_SCIPY and FINAL["resid_uniform"].notna().sum() >= 5:
        rho, p = spearmanr(FINAL["mean_delta"], FINAL["resid_uniform"], nan_policy="omit")
        S(""); S(f"Amplification index vs literature residual (uniform): rho={fmt(rho)}, p={fmt(p)} {stars(p)}")
        SIG.append(f"[S10] Genomic rank shift vs deviation from the literature ordering: rho={fmt(rho)}, "
                   f"p={fmt(p)} -> {'systematic, not random' if (not np.isnan(p) and p < ALPHA) else 'partly independent axes'}.")
else:
    S("Amplification index unavailable — classification skipped.")


# =============================================================================
# 10. FIGURES
# =============================================================================
if HAVE_MPL:
    print("Rendering figures ...")
    try:
        Rm = RANK_WIDE.dropna(how="any")
        if not Rm.empty:
            Rm = Rm.loc[Rm.mean(axis=1).sort_values().index]
            fig, ax = plt.subplots(figsize=(1.5*Rm.shape[1]+4, 0.34*len(Rm)+2))
            im = ax.imshow(Rm.values, aspect="auto", cmap="YlOrRd_r")
            ax.set_yticks(range(len(Rm))); ax.set_yticklabels([DRUG_LABEL[i] for i in Rm.index], fontsize=8)
            ax.set_xticks(range(Rm.shape[1]))
            ax.set_xticklabels([f"{a}\n{b}" for a, b in Rm.columns], fontsize=7, rotation=45, ha="right")
            for i in range(Rm.shape[0]):
                for j in range(Rm.shape[1]):
                    ax.text(j, i, int(Rm.values[i, j]), ha="center", va="center", fontsize=6)
            plt.colorbar(im, ax=ax, label="rank (1 = highest risk)")
            ax.set_title("Drug metabolic-risk rank across every method x trait run")
            save_fig(fig, "fig01_rank_heatmap.png", "Rank heatmap")

        if AMP is not None and len(AMP):
            d = AMP.sort_values("mean_delta")
            fig, ax = plt.subplots(figsize=(8, 0.32*len(d)+2))
            ax.barh(d["drug"], d["mean_delta"], xerr=d["sd_delta"].fillna(0),
                    color=["tab:red" if v > 0 else "tab:blue" for v in d["mean_delta"]],
                    alpha=.85, error_kw=dict(lw=.7))
            ax.axvline(0, color="k", lw=.8)
            ax.set_xlabel("mean Δrank (weighted − uniform; >0 = genomically amplified)")
            ax.set_title("Transdiagnostic genomic-amplification index")
            save_fig(fig, "fig02_amplification_index.png", "Amplification index")

        if WMETHOD in STAB and UMETHOD in STAB:
            c2 = STAB[WMETHOD]["tab"][["drug_key","drug","sd_rank"]].merge(
                 STAB[UMETHOD]["tab"][["drug_key","sd_rank"]], on="drug_key", suffixes=("_w","_u"))
            fig, ax = plt.subplots(figsize=(6.5, 6))
            ax.scatter(c2["sd_rank_w"], c2["sd_rank_u"], s=40, c="tab:purple", alpha=.8)
            lim = max(c2[["sd_rank_w","sd_rank_u"]].max())*1.15 + .2
            ax.plot([0, lim], [0, lim], "k--", lw=.8)
            for _, r in c2.iterrows():
                ax.annotate(str(r["drug"])[:16], (r["sd_rank_w"], r["sd_rank_u"]),
                            fontsize=7, xytext=(3, 3), textcoords="offset points")
            ax.set_xlabel("cross-trait rank SD — weighted"); ax.set_ylabel("cross-trait rank SD — uniform")
            ax.set_title("Cross-trait stability of the two rankings")
            save_fig(fig, "fig03_stability_scatter.png", "Stability scatter")

        if CORR is not None:
            fig, ax = plt.subplots(figsize=(1.0*CORR.shape[1]+4, 0.9*CORR.shape[0]+3))
            im = ax.imshow(CORR.values, cmap="viridis", vmin=np.nanmin(CORR.values), vmax=1)
            ax.set_xticks(range(CORR.shape[1])); ax.set_xticklabels(CORR.columns, rotation=45, ha="right", fontsize=7)
            ax.set_yticks(range(CORR.shape[0])); ax.set_yticklabels(CORR.index, fontsize=7)
            for i in range(CORR.shape[0]):
                for j in range(CORR.shape[1]):
                    ax.text(j, i, f"{CORR.values[i,j]:.2f}", ha="center", va="center", fontsize=6, color="w")
            plt.colorbar(im, ax=ax, label="Spearman rho"); ax.set_title("Similarity of all method x trait runs")
            save_fig(fig, "fig04_run_correlation.png", "Run correlation heatmap")

        if GENE_DELTA is not None and len(GENE_DELTA):
            d = GENE_DELTA.sort_values("mean_delta")
            fig, ax = plt.subplots(figsize=(7.5, 0.30*len(d)+2))
            ax.barh(d["gene"], d["mean_delta"],
                    color=["tab:green" if v > 0 else "tab:orange" for v in d["mean_delta"]], alpha=.85)
            ax.axvline(0, color="k", lw=.8)
            ax.set_xlabel("Δ pooled share (uniform − weighted, share points)")
            ax.set_title("Receptor drivers gained / lost without the literature prior")
            save_fig(fig, "fig05_gene_share_delta.png", "Gene share delta")

        for m, d in RESID_SUM.items():
            s_ = d["summary"].sort_values("mean_resid")
            fig, ax = plt.subplots(figsize=(7.5, 0.30*len(s_)+2))
            ax.barh(s_["drug"], s_["mean_resid"], xerr=s_["sd_resid"].fillna(0),
                    color=["tab:red" if v < 0 else "tab:blue" for v in s_["mean_resid"]],
                    alpha=.85, error_kw=dict(lw=.7))
            ax.axvline(0, color="k", lw=.8)
            ax.set_xlabel("mean residual (model rank − literature rank)")
            ax.set_title(f"Deviation from the literature ordering — {m}")
            save_fig(fig, f"fig06_residuals_{re.sub(r'[^A-Za-z0-9]+','_',m)}.png", "Residuals")
    except Exception as e:
        print(f"  [warn] figure rendering issue: {e}")


# =============================================================================
# 11. FINDINGS, SYNTHESIS, CAVEATS, FILE INDEX, WRITE MASTER SUMMARY
# =============================================================================
SEC("Z1. HEADLINE / SIGNIFICANT FINDINGS (auto-collected)")
if SIG:
    seen, ordered = set(), []
    for s_ in SIG:
        if s_ not in seen: seen.add(s_); ordered.append(s_)
    for i, s_ in enumerate(ordered, 1):
        for j, chunk in enumerate(textwrap.wrap(s_, 96)):
            S(f"{str(i)+'.':>4} {chunk}" if j == 0 else f"     {chunk}")
else:
    S("No findings met the reporting thresholds.")

SEC("Z2. SYNTHESIS — WHAT THIS DESIGN ADDS")
for line in [
 "1. TWAS is now injected PER RECEPTOR as exp(beta * -log10 p) BEFORE aggregation, so a",
 "   genomically implicated receptor amplifies the specific drug-receptor contribution it",
 "   belongs to, instead of applying a single blunt drug-level multiplier at the end.",
 "2. The -log10(p) scale is capped (MAX_NEGLOG10_P) and beta is small, so genomics MODULATES",
 "   rather than dominates the pharmacological signal; section C quantifies exactly how much.",
 "3. The dual run isolates the literature prior: 'weighted' is a clinical-proxy ranking,",
 "   'uniform' is a genomics-influenced ranking in which every receptor with Ki data counts",
 "   equally. Their difference (S1) is the genomic-amplification index.",
 "4. S3 splits the drugs into a robust dual-evidence set (top-k under BOTH), a genomically",
 "   amplified set (uniform only, hypothesis-generating) and a prior-dependent set.",
 "5. S4 shows WHICH receptor genes gain share when the prior is removed — i.e. which drivers",
 "   the literature weighting was suppressing.",
 "6. The antidepressant-specific chronic 5-HT adaptation is retained as a third method and",
 "   quantified in S1b: it captures the long-term SERT -> 5-HT2C/2A up / 5-HT1A down",
 "   remodelling that acute Ki values structurally cannot represent.",
 "7. S8 separates the transdiagnostic component (drug main effect) from context-specific",
 "   re-ordering (interaction), and S10 merges risk tier + amplification + literature",
 "   residual into an actionable taxonomy.",
]: S(line)

SEC("Z3. CAVEATS")
for line in [
 "1. Scores are normalised to amitriptyline = 100 WITHIN each run; cross-run comparisons must",
 "   be made on RANKS, not on raw scores.",
 "2. 'weighted' runs embed a literature receptor->metabolic prior and are partially circular",
 "   with clinical expectation; 'uniform' runs are non-circular but give equal a-priori weight",
 "   to every receptor with Ki data, including pharmacologically irrelevant ones.",
 "3. Ki aggregation uses the MINIMUM (highest-affinity) value per drug x receptor, biasing",
 "   toward the most potent reported measurement.",
 "4. Imputed Ki and imputed TWAS values propagate into the scores; rows flagged KiImp / TwImp",
 "   in section C are lower-confidence. Fairness-imputed genes neither gain nor lose relative",
 "   to the panel mean -log10(p).",
 "5. The four Zhou traits (CKD, ESSHP, T2D, obesity) are genetically correlated, so their",
 "   rankings are NOT independent replicates; Kendall's W and the cross-trait tests are",
 "   consistency checks, not independent validations.",
 "6. The literature weight-gain ordering (S5) is ordinal, incomplete and largely derived from",
 "   short-term adult trials; a null residual result is not evidence against the model.",
 "7. The 5-HT adaptation coefficients (HTR2C +0.18, HTR2A +0.12, HTR1A -0.05, SERT threshold",
 "   0.1) are expert-prior modelling weights, not measured fold-changes; run the suggested",
 "   sensitivity ranges before drawing strong conclusions.",
 "8. No multiplicity correction is applied across the many per-drug sign tests; prioritise by",
 "   consistency of direction rather than by nominal p-value alone.",
]: S(line)

SEC("Z4. FILE INDEX")
S(f"{'path':<86}  description"); S(SUB)
for p, note in FILES_OUT: S(f"{p:<86}  {note}")

S(""); S(BAR); S("END OF MASTER SUMMARY"); S(BAR)

with open(MASTER_TXT, "w", encoding="utf-8") as fh:
    fh.write("\n".join(SUMMARY))
FILES_OUT.append((MASTER_TXT, "MASTER detailed summary"))

payload = {"generated": str(datetime.datetime.now()), "output_root": OUTPUT_ROOT,
           "methods": METHODS, "traits": TRAITS, "n_runs": int(N_RUNS),
           "twas_scaling": TWAS_SCALING, "logp_beta": LOGP_BETA,
           "max_neglog10_p": MAX_NEGLOG10_P, "secondary_boost": SECONDARY_BOOST_FAC,
           "twas_sig_bonus": TWAS_SIG_BONUS, "reference_drug": REFERENCE_DRUG,
           "n_drugs_union": int(RANK_WIDE.shape[0]), "n_drugs_complete": int(COMPLETE.shape[0]),
           "consistent_amplifiers": (AMP[AMP.consistent_amplifier]["drug"].tolist() if AMP is not None else []),
           "consistent_prior_dominant": (AMP[AMP.consistent_dominant]["drug"].tolist() if AMP is not None else []),
           "significant_findings": SIG, "files": [p for p, _ in FILES_OUT]}
json_path = os.path.join(OUTPUT_ROOT, "results_manifest.json")
with open(json_path, "w", encoding="utf-8") as fh:
    json.dump(payload, fh, indent=2, default=str)

print("\n" + "="*100)
print("DONE.")
print(f"  MASTER summary : {MASTER_TXT}")
print(f"  JSON manifest  : {json_path}")
print(f"  Stage-3 tables : {S3_TABLES} ({len(glob.glob(os.path.join(S3_TABLES,'*.csv')))} files)")
print(f"  Figures        : {S3_FIGS} ({len(glob.glob(os.path.join(S3_FIGS,'*.png')))} files)")
print("="*100)
print("\n--- first 160 lines of MASTER_SUMMARY.txt ---\n")
print("\n".join(SUMMARY[:160]))

if COPY_TO_DRIVE:
    try:
        os.makedirs(DRIVE_DEST, exist_ok=True)
        dest = os.path.join(DRIVE_DEST, os.path.basename(OUTPUT_ROOT))
        if os.path.exists(dest): shutil.rmtree(dest)
        shutil.copytree(OUTPUT_ROOT, dest)
        print(f"\nCopied {OUTPUT_ROOT} -> {dest}")
    except Exception as e:
        print(f"\n[warn] could not copy to Drive: {e}")


####################################################################################################
# PART 1 — SCORING (JvH logp_expo TWAS integration, dual weighting)
####################################################################################################

####################################################################################################
# TRAIT: zhou_CKD   (/content/zhou/zhou_CKD)
####################################################################################################
Building TWAS lookup for: /content/zhou/zhou_CKD
  TWAS genes loaded: 17,854 from 6/6 file(s)
    HTR2C present? False
    HRH1 present? True
    DRD2 present? True
    CHRM1 present? True
    CHRM3 present? True
    HTR2A present? True
    SLC6A4 present? True
  Ki rows: 98,764
  Drugs: 68

Matching drugs and building drug x receptor table ...


100%|██████████| 68/68 [00:06<00:00,  9.91it/s]



[TWAS] 9 receptor gene(s) had no TWAS data (e.g. ADRB3, DRD3, DRD5, HRH3, HTR1A, HTR1D, HTR1E, HTR2C)

[DDD filter] kept 49/68 drug(s) with a valid DDD (19 dropped).

Imputation step ...
  [ki-impute] 'mean': nothing to fill.
  [twas-impute] 'mean': filled 129 row(s) (z=-0.245, p=0.277); kept NON-significant.
  [drop-zero] removed 0 row(s) with affinity_dose_score == 0.

--- scoring [weighted] zhou_CKD ---
  [weights] literature METABOLIC_WEIGHTS (clinical proxy)
  [note] TWAS coverage: 397/397 (100%) scored-gene rows have twas_p.
  [TWAS-logp] exp scaling on -log10(p) (beta=0.045, cap=25.0); secondary boost 0.02
  [drop-zero] removed 8 drug(s) with risk == 0.
 risk_rank             drug  metabolic_risk_score  base_score  max_abs_twas_z  n_twas_sig_metabolic  risk_vs_reference
         1        mianserin            109.587127    6.532804        2.660913                     2           9.587127
         2    amitriptyline            100.000000    5.502728        2.660913               

100%|██████████| 68/68 [00:02<00:00, 25.13it/s]



[TWAS] 9 receptor gene(s) had no TWAS data (e.g. ADRB3, DRD3, DRD5, HRH3, HTR1A, HTR1D, HTR1E, HTR2C)

[DDD filter] kept 49/68 drug(s) with a valid DDD (19 dropped).

Imputation step ...
  [ki-impute] 'mean': nothing to fill.
  [twas-impute] 'mean': filled 129 row(s) (z=0.270, p=0.284); kept NON-significant.
  [drop-zero] removed 0 row(s) with affinity_dose_score == 0.

--- scoring [weighted] zhou_ESSHP ---
  [weights] literature METABOLIC_WEIGHTS (clinical proxy)
  [note] TWAS coverage: 397/397 (100%) scored-gene rows have twas_p.
  [TWAS-logp] exp scaling on -log10(p) (beta=0.045, cap=25.0); secondary boost 0.02
  [drop-zero] removed 8 drug(s) with risk == 0.
 risk_rank             drug  metabolic_risk_score  base_score  max_abs_twas_z  n_twas_sig_metabolic  risk_vs_reference
         1        mianserin            106.800056    6.331716        2.410641                     0           6.800056
         2    amitriptyline            100.000000    5.293240        3.364833              

100%|██████████| 68/68 [00:03<00:00, 21.26it/s]



[TWAS] 9 receptor gene(s) had no TWAS data (e.g. ADRB3, DRD3, DRD5, HRH3, HTR1A, HTR1D, HTR1E, HTR2C)

[DDD filter] kept 49/68 drug(s) with a valid DDD (19 dropped).

Imputation step ...
  [ki-impute] 'mean': nothing to fill.
  [twas-impute] 'mean': filled 129 row(s) (z=0.350, p=0.286); kept NON-significant.
  [drop-zero] removed 0 row(s) with affinity_dose_score == 0.

--- scoring [weighted] zhou_T2D ---
  [weights] literature METABOLIC_WEIGHTS (clinical proxy)
  [note] TWAS coverage: 397/397 (100%) scored-gene rows have twas_p.
  [TWAS-logp] exp scaling on -log10(p) (beta=0.045, cap=25.0); secondary boost 0.02
  [drop-zero] removed 8 drug(s) with risk == 0.
 risk_rank             drug  metabolic_risk_score  base_score  max_abs_twas_z  n_twas_sig_metabolic  risk_vs_reference
         1        mianserin            111.422668    6.282333        3.109295                     1          11.422668
         2    amitriptyline            100.000000    5.205386        2.732265                

100%|██████████| 68/68 [00:02<00:00, 24.24it/s]



[TWAS] 9 receptor gene(s) had no TWAS data (e.g. ADRB3, DRD3, DRD5, HRH3, HTR1A, HTR1D, HTR1E, HTR2C)

[DDD filter] kept 49/68 drug(s) with a valid DDD (19 dropped).

Imputation step ...
  [ki-impute] 'mean': nothing to fill.
  [twas-impute] 'mean': filled 129 row(s) (z=0.112, p=0.328); kept NON-significant.
  [drop-zero] removed 0 row(s) with affinity_dose_score == 0.

--- scoring [weighted] zhou_obesity ---
  [weights] literature METABOLIC_WEIGHTS (clinical proxy)
  [note] TWAS coverage: 397/397 (100%) scored-gene rows have twas_p.
  [TWAS-logp] exp scaling on -log10(p) (beta=0.045, cap=25.0); secondary boost 0.02
  [drop-zero] removed 8 drug(s) with risk == 0.
 risk_rank             drug  metabolic_risk_score  base_score  max_abs_twas_z  n_twas_sig_metabolic  risk_vs_reference
         1        mianserin            120.605054    6.381618        2.986415                     2          20.605054
         2    amitriptyline            100.000000    5.342962        2.474424            

# Downstream

In [ ]:
# =============================================================================
# 081_DOWNSTREAM — post-processing of the 080KvD-v3 N06A Ki→DDD→TWAS pipeline
# -----------------------------------------------------------------------------
# INPUTS (results only — nothing is recomputed from the Ki DB or raw TWAS files):
#   <ROOT>/<method>/<trait>/metabolic_risk_score_per_drug.csv     [required]
#   <ROOT>/<method>/<trait>/receptor_contributions.csv            [strongly recommended]
#   <ROOT>/<method>/<trait>/n06a_ki_twas_full_results.csv         [optional]
#   <ROOT>/weighted_5ht_adapted/<trait>/metabolic_risk_5ht_adapted_vs_base.csv
#   <ROOT>/stage3_dual/tables/*.csv                               [used if present]
#
# ANALYSES
#   D1  Genomic-consideration priority table (amplification x classification)
#   D2  SSRI/SNRI rank-shift under uniform weighting (+ sign & Wilcoxon tests)
#   D3  Receptor drivers of the largest movers
#   D4  Top-k stability: clinical-proxy vs genomics-influenced (Jaccard, hypergeom)
#   D5  Chronic 5-HT adaptation effect, class-resolved (+ paired test)
#   D6  One-page decision table
#   D7  Drug-class analysis (TCA / SSRI / SNRI / MAOI / atypical) — Kruskal-Wallis
#   D8  Per-drug TWAS leverage (with vs without the exponential layer)
#   D9  Receptor-gene TWAS panel with BH-FDR (within-panel prioritisation)
#   D10 EXACT reproduction check + beta-sensitivity sweep (uses outputs only)
#   D11 Leave-one-trait-out robustness of the top-k sets
#   D12 Per-trait rank profiles & heterogeneity flags
#   -> figures + DOWNSTREAM_SUMMARY.txt (very detailed) + downstream_results.json
# Run as ONE cell.
# =============================================================================

import os, re, glob, json, math, textwrap, datetime, itertools, warnings, shutil
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

try:
    from scipy.stats import (spearmanr, kendalltau, wilcoxon, mannwhitneyu,
                             kruskal, hypergeom, chi2)
    HAVE_SCIPY = True
except Exception as _e:
    HAVE_SCIPY = False
    print(f"[warn] scipy unavailable ({_e}) — inferential statistics limited.")

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    HAVE_MPL = True
except Exception as _e:
    HAVE_MPL = False
    print(f"[warn] matplotlib unavailable ({_e}) — figures skipped.")


# =============================================================================
# CONFIG — EDIT PATHS
# =============================================================================
PIPELINE_ROOT = "/content/pipeline_output/n06a_logp_expo"
S3_TABLES_IN  = os.path.join(PIPELINE_ROOT, "stage3_dual", "tables")   # optional
OUT_DIR       = "/content/pipeline_output/n06a_downstream_v3"

COPY_TO_DRIVE = False
DRIVE_DEST    = "/content/drive/MyDrive/Dr Uccello/00_Studies/080_GHS_CVS"

WMETHOD, UMETHOD, AMETHOD = "weighted", "uniform", "weighted_5ht_adapted"

TOP_K       = 10
ALPHA       = 0.05
FDR_ALPHA   = 0.05
RANDOM_SEED = 42
MOVERS_N    = 8
np.random.seed(RANDOM_SEED)

# constants used by pipeline 080KvD-v3 (needed ONLY to reproduce / sweep beta)
LOGP_BETA_REF       = 0.045
MAX_NEGLOG10_P      = 25.0
SECONDARY_BOOST_FAC = 0.02
TWAS_SIG_BONUS      = 0.10
REFERENCE_DRUG      = "amitriptyline"
BETA_GRID           = [0.0, 0.015, 0.03, 0.045, 0.06, 0.09, 0.15]

METABOLIC_WEIGHTS = {
    "HRH1":1.00,"HTR2C":0.72,"CHRM3":0.68,"HTR2A":0.45,"ADRA1A":0.42,"ADRA1B":0.40,
    "HTR6":0.38,"ADRA2A":0.28,"ADRA2B":0.26,"ADRA2C":0.26,"CHRM1":0.25,"CHRM4":0.20,
    "CHRM5":0.18,"HTR7":0.22,"DRD3":0.15,"DRD2":0.10,"DRD4":0.08,"HTR1A":0.07,
    "ADRB1":0.07,"ADRB2":0.06,"SIGMAR1":0.04,"SLC6A4":0.08,"SLC6A2":0.05,"SLC6A3":0.04,
}

# ---- drug-class regexes (order matters: first match wins) --------------------
CLASS_PATTERNS = [
    ("SSRI",  r"sertraline|fluoxetine|paroxetine|citalopram|escitalopram|fluvoxamine|zimeldine|alaproclate"),
    ("SNRI",  r"venlafaxine|desvenlafaxine|duloxetine|milnacipran|levomilnacipran|sibutramine"),
    ("TCA",   r"amitriptyl|nortriptyl|imipramin|desipramin|clomipramin|doxepin|trimipramin|"
              r"dosulepin|dothiepin|lofepramin|protriptyl|amoxapin|butriptyl|iprindol|opipramol|"
              r"melitracen|dibenzepin|quinupramin|noxiptilin|amineptin|tianeptin|maprotilin"),
    ("MAOI",  r"phenelzin|tranylcypromin|isocarboxazid|moclobemid|iproniazid|nialamid|"
              r"toloxaton|selegilin|safinamid"),
    ("ATYP",  r"mirtazapin|mianserin|trazodon|nefazodon|bupropion|amfebutamon|agomelatin|"
              r"reboxetin|vortioxetin|vilazodon|esketamin|ketamin|setiptilin|viloxazin|"
              r"tandospiron|hyperici|oxitriptan|tryptophan|medifoxamin|minaprin|bifemelan"),
]
SSRI_SNRI_REGEX = CLASS_PATTERNS[0][1] + "|" + CLASS_PATTERNS[1][1] + "|clomipramine|vortioxetine|vilazodone"

os.makedirs(OUT_DIR, exist_ok=True)
TABLES = os.path.join(OUT_DIR, "tables"); os.makedirs(TABLES, exist_ok=True)
FIGS   = os.path.join(OUT_DIR, "figures"); os.makedirs(FIGS, exist_ok=True)

SUMMARY, SIG, FILES_OUT, NOTES = [], [], [], []
BAR, SUB = "=" * 100, "-" * 100


# =============================================================================
# HELPERS
# =============================================================================
def S(x=""):   SUMMARY.append(str(x))
def SEC(t):    S(""); S(BAR); S(str(t).upper()); S(BAR)
def SUBSEC(t): S(""); S(SUB); S(str(t)); S(SUB)

def stars(p):
    if p is None or (isinstance(p, float) and np.isnan(p)): return "n/a"
    return "****" if p < 1e-4 else "***" if p < 1e-3 else "**" if p < 1e-2 else "*" if p < .05 else "ns"

def fmt(x, nd=3):
    try:
        if x is None or (isinstance(x, float) and np.isnan(x)): return "NA"
        if isinstance(x, (bool, np.bool_)): return str(bool(x))
        if isinstance(x, (int, np.integer)): return f"{int(x):,}"
        ax = abs(float(x))
        if ax != 0 and (ax < 1e-3 or ax >= 1e6): return f"{float(x):.3e}"
        return f"{float(x):.{nd}f}"
    except Exception:
        return str(x)

def save_table(df, name, note=""):
    p = os.path.join(TABLES, name); df.to_csv(p, index=False)
    FILES_OUT.append((p, note)); return p

def save_fig(fig, name, note=""):
    if not HAVE_MPL: return None
    p = os.path.join(FIGS, name); fig.savefig(p, dpi=150, bbox_inches="tight")
    plt.close(fig); FILES_OUT.append((p, note)); return p

def read_csv_safe(path):
    if not path or not os.path.exists(path): return None
    for enc in ["utf-8", "utf-8-sig", "latin-1", "cp1252"]:
        try: return pd.read_csv(path, encoding=enc, low_memory=False)
        except Exception: continue
    return None

def norm_drug(x): return re.sub(r"\s+", " ", str(x).strip().lower())

def tobool(s):
    return pd.Series(s).map(lambda v: str(v).strip().lower() in {"true","1","1.0","yes","y"})

def drug_class(name):
    n = norm_drug(name)
    for cls, pat in CLASS_PATTERNS:
        if re.search(pat, n): return cls
    return "OTHER"

def bh_fdr(pvals):
    p = np.asarray(pvals, float); ok = ~np.isnan(p)
    q = np.full_like(p, np.nan, float); pv = p[ok]; n = pv.size
    if n == 0: return q
    o = np.argsort(pv); r = pv[o]
    adj = np.clip(np.minimum.accumulate((r*n/(np.arange(n)+1))[::-1])[::-1], 0, 1)
    out = np.empty(n); out[o] = adj; q[ok] = out
    return q

def binom_p(k, n, p=0.5):
    if n == 0: return np.nan
    try:
        from scipy.stats import binomtest; return float(binomtest(int(k), int(n), p).pvalue)
    except Exception:
        try:
            from scipy.stats import binom_test; return float(binom_test(int(k), int(n), p))
        except Exception: return np.nan

def safe(fn, label):
    """Run an analysis block, never let one failure kill the pipeline."""
    try:
        return fn()
    except Exception as e:
        msg = f"[{label}] FAILED: {type(e).__name__}: {e}"
        print("  " + msg); NOTES.append(msg)
        S(""); S(f"  [!] {label} could not be completed: {type(e).__name__}: {e}")
        return None


# =============================================================================
# LOAD — stage-3 tables if present, else rebuild from per-run CSVs
# =============================================================================
print("Loading pipeline artefacts ...")
if not os.path.isdir(PIPELINE_ROOT):
    raise FileNotFoundError(f"PIPELINE_ROOT not found: {PIPELINE_ROOT}")

def discover_runs(root):
    runs = {}
    for mdir in sorted(glob.glob(os.path.join(root, "*"))):
        if not os.path.isdir(mdir): continue
        method = os.path.basename(mdir)
        if method.startswith("stage3"): continue
        for tdir in sorted(glob.glob(os.path.join(mdir, "*"))):
            if not os.path.isdir(tdir): continue
            trait = os.path.basename(tdir)
            risk = os.path.join(tdir, "metabolic_risk_score_per_drug.csv")
            if not os.path.exists(risk): continue
            runs[(method, trait)] = {
                "risk": risk,
                "detail": (os.path.join(tdir, "receptor_contributions.csv")
                           if os.path.exists(os.path.join(tdir, "receptor_contributions.csv")) else None),
                "long": (os.path.join(tdir, "n06a_ki_twas_full_results.csv")
                         if os.path.exists(os.path.join(tdir, "n06a_ki_twas_full_results.csv")) else None),
                "adapt_cmp": (os.path.join(tdir, "metabolic_risk_5ht_adapted_vs_base.csv")
                              if os.path.exists(os.path.join(tdir, "metabolic_risk_5ht_adapted_vs_base.csv")) else None),
                "mtime": datetime.datetime.fromtimestamp(os.path.getmtime(risk)),
            }
    return runs

RUNS = discover_runs(PIPELINE_ROOT)
if not RUNS:
    raise RuntimeError(f"No <method>/<trait>/metabolic_risk_score_per_drug.csv found under {PIPELINE_ROOT}")

RISK = {}
for k, meta in RUNS.items():
    df = read_csv_safe(meta["risk"])
    if df is None or df.empty: continue
    df["drug_key"] = df["drug"].map(norm_drug)
    for c in ["metabolic_risk_score","base_score","max_abs_twas_z",
              "n_twas_sig_metabolic","risk_vs_reference","risk_rank"]:
        if c in df.columns: df[c] = pd.to_numeric(df[c], errors="coerce")
    if "risk_rank" not in df.columns:
        df["risk_rank"] = df["metabolic_risk_score"].rank(ascending=False, method="min")
    RISK[k] = df.sort_values("metabolic_risk_score", ascending=False).reset_index(drop=True)

# ---- master long table -------------------------------------------------------
LONG = read_csv_safe(os.path.join(S3_TABLES_IN, "00_master_long_risk_table.csv"))
if LONG is not None and {"method","trait","drug","risk_rank"}.issubset(LONG.columns):
    LONG_SOURCE = "stage3 master table"
    if "drug_key" not in LONG.columns: LONG["drug_key"] = LONG["drug"].map(norm_drug)
else:
    rows = []
    for (m, t), df in RISK.items():
        for _, r in df.iterrows():
            rows.append({"method": m, "trait": t, "drug": r["drug"], "drug_key": r["drug_key"],
                         "risk_rank": r.get("risk_rank"), "risk_score": r.get("metabolic_risk_score"),
                         "base_score": r.get("base_score"), "max_abs_twas_z": r.get("max_abs_twas_z"),
                         "n_twas_sig": r.get("n_twas_sig_metabolic")})
    LONG = pd.DataFrame(rows); LONG_SOURCE = "rebuilt from per-run risk CSVs"
LONG["risk_rank"]  = pd.to_numeric(LONG["risk_rank"], errors="coerce")
LONG["risk_score"] = pd.to_numeric(LONG.get("risk_score"), errors="coerce")
LONG["drug_class"] = LONG["drug"].map(drug_class)

METHODS = [m for m in [WMETHOD, UMETHOD, AMETHOD] if m in set(LONG["method"])] + \
          [m for m in sorted(set(LONG["method"])) if m not in {WMETHOD, UMETHOD, AMETHOD}]
TRAITS  = sorted(LONG["trait"].unique())
DRUG_LABEL = LONG.drop_duplicates("drug_key").set_index("drug_key")["drug"].to_dict()
HAVE_PAIR = (WMETHOD in METHODS) and (UMETHOD in METHODS)
HAVE_ADAPT = AMETHOD in METHODS

RANK_WIDE = LONG.pivot_table(index="drug_key", columns=["method","trait"], values="risk_rank")
COMPLETE  = RANK_WIDE.dropna(how="any")

# ---- receptor contributions --------------------------------------------------
DRV = read_csv_safe(os.path.join(S3_TABLES_IN, "04_receptor_contributions_all_runs.csv"))
DRV_SOURCE = "stage3 table 04"
if DRV is None or DRV.empty or "contribution" not in (DRV.columns if DRV is not None else []):
    frames = []
    for (m, t), meta in RUNS.items():
        d = read_csv_safe(meta["detail"])
        if d is None or d.empty: continue
        d["method"], d["trait"] = m, t
        frames.append(d)
    DRV = pd.concat(frames, ignore_index=True) if frames else None
    DRV_SOURCE = "rebuilt from per-run receptor_contributions.csv"
if DRV is not None and not DRV.empty:
    DRV["drug_key"] = DRV["drug"].map(norm_drug)
    for c in ["affinity_dose_score","affinity_log1p","twas_z","twas_p","receptor_weight",
              "twas_scale","contribution","contribution_no_twas","share_pct","Ki_nM","DDD_mg"]:
        if c in DRV.columns: DRV[c] = pd.to_numeric(DRV[c], errors="coerce")
    for c in ["twas_significant","strong_binder","ki_imputed","twas_imputed"]:
        if c in DRV.columns: DRV[c] = tobool(DRV[c]).values

# ---- optional stage-3 tables --------------------------------------------------
T = lambda n: read_csv_safe(os.path.join(S3_TABLES_IN, n))
AMP_IN    = T("S1b_genomic_amplification_index.csv")
CLS_IN    = T("S10_final_transdiagnostic_classification.csv")
ADAPT_SUM = T("S1d_5ht_adaptation_summary.csv")
ADAPT_LNG = T("S1c_5ht_adaptation_long.csv")
GENE_DLT  = T("S4a_gene_share_delta.csv")
VAL_IN    = T("S5_external_validation.csv")
VD_IN     = T("S8_variance_decomposition.csv")

# 5-HT comparison rebuilt from per-run files if the stage-3 long table is absent
if (ADAPT_LNG is None or ADAPT_LNG.empty) and HAVE_ADAPT:
    fr = []
    for (m, t), meta in RUNS.items():
        if m != AMETHOD or not meta["adapt_cmp"]: continue
        c = read_csv_safe(meta["adapt_cmp"])
        if c is None or c.empty: continue
        c["trait"] = t; fr.append(c)
    ADAPT_LNG = pd.concat(fr, ignore_index=True) if fr else None
if ADAPT_LNG is not None and not ADAPT_LNG.empty:
    for c in ["adapted","is_ssri_snri"]:
        if c in ADAPT_LNG.columns: ADAPT_LNG[c] = tobool(ADAPT_LNG[c]).values
    for c in ["base_risk","adapted_risk","adaptation_delta","rank_delta","sert_affinity",
              "base_rank","adapted_rank"]:
        if c in ADAPT_LNG.columns: ADAPT_LNG[c] = pd.to_numeric(ADAPT_LNG[c], errors="coerce")

print(f"   methods={METHODS}\n   traits ={TRAITS}\n   runs   ={len(RISK)}   "
      f"drugs(union)={RANK_WIDE.shape[0]}  complete={COMPLETE.shape[0]}")


# =============================================================================
# SUMMARY HEADER
# =============================================================================
S(BAR)
S("DOWNSTREAM ANALYSES OF THE N06A ANTIDEPRESSANT Ki -> DDD -> TWAS RISK PIPELINE (080KvD-v3)")
S("Operates EXCLUSIVELY on artefacts written by the parent pipeline.")
S("weighted = CLINICAL-PROXY ranking | uniform = GENOMICS-INFLUENCED ranking |")
S("weighted_5ht_adapted = chronic SERT -> 5-HT2C/2A up, 5-HT1A down remodelling")
S(BAR)
S(f"Generated              : {datetime.datetime.now():%Y-%m-%d %H:%M:%S}")
S(f"Pipeline root          : {PIPELINE_ROOT}")
S(f"Downstream output dir  : {OUT_DIR}")
S(f"Risk-table source      : {LONG_SOURCE}")
S(f"Contribution source    : {DRV_SOURCE if DRV is not None else 'UNAVAILABLE'}")
S(f"Methods                : {len(METHODS)} -> {', '.join(METHODS)}")
S(f"Traits                 : {len(TRAITS)} -> {', '.join(TRAITS)}")
S(f"Runs (method x trait)  : {len(RISK)}")
S(f"Drugs (union / complete): {RANK_WIDE.shape[0]} / {COMPLETE.shape[0]}")
S("")
S("Settings:")
for k, v in [("TOP_K",TOP_K),("ALPHA",ALPHA),("FDR_ALPHA",FDR_ALPHA),("MOVERS_N",MOVERS_N),
             ("LOGP_BETA_REF",LOGP_BETA_REF),("MAX_NEGLOG10_P",MAX_NEGLOG10_P),
             ("SECONDARY_BOOST_FAC",SECONDARY_BOOST_FAC),("TWAS_SIG_BONUS",TWAS_SIG_BONUS),
             ("REFERENCE_DRUG",REFERENCE_DRUG),("BETA_GRID",BETA_GRID),
             ("scipy",HAVE_SCIPY),("matplotlib",HAVE_MPL)]:
    S(f"  {k:<22}= {v}")

SEC("A. RUN INVENTORY / PROVENANCE")
S(f"{'method':<26}{'trait':<16}{'drugs':>7}{'detail':>8}{'long':>7}{'adaptCmp':>10}  modified")
S(SUB)
for (m, t), meta in sorted(RUNS.items()):
    S(f"{m:<26}{t:<16}{len(RISK.get((m,t),[])):>7}"
      f"{('yes' if meta['detail'] else 'no'):>8}{('yes' if meta['long'] else 'no'):>7}"
      f"{('yes' if meta['adapt_cmp'] else 'no'):>10}  {meta['mtime']:%Y-%m-%d %H:%M}")
cov = RANK_WIDE.notna().sum(axis=1); nr = RANK_WIDE.shape[1]
inc = [(DRUG_LABEL[i], int(c)) for i, c in cov.items() if c < nr]
S(""); S(f"Drugs present in ALL {nr} runs: {int((cov==nr).sum())}/{len(cov)}")
if inc:
    S("Drugs missing from >=1 run (zero score, no DDD, or name mismatch):")
    for d, c in sorted(inc, key=lambda x: x[1]): S(f"   - {d:<32} {c}/{nr} runs")
    SIG.append(f"[QC] {len(inc)} drug(s) are not present in every run; cross-run comparisons "
               f"use the intersection where required.")
S("")
S("Drug-class assignment (regex-based):")
for cls, grp in LONG.drop_duplicates("drug_key").groupby("drug_class"):
    S(f"   {cls:<7} (n={len(grp)}): " + ", ".join(sorted(grp['drug'].astype(str))[:14])
      + (" ..." if len(grp) > 14 else ""))


# =============================================================================
# D1. GENOMIC-CONSIDERATION PRIORITY TABLE
# =============================================================================
def d1():
    global AMP, PRIORITY
    AMP = None; PRIORITY = None
    if AMP_IN is not None and not AMP_IN.empty:
        AMP = AMP_IN.copy(); src = "stage3 S1b"
    elif HAVE_PAIR:
        parts = []
        for t in TRAITS:
            w = LONG[(LONG.method==WMETHOD)&(LONG.trait==t)][["drug_key","drug","risk_rank","risk_score"]]
            u = LONG[(LONG.method==UMETHOD)&(LONG.trait==t)][["drug_key","risk_rank","risk_score"]]
            mg = w.merge(u, on="drug_key", suffixes=("_w","_u"))
            if mg.empty: continue
            mg["trait"] = t; mg["rank_delta"] = mg["risk_rank_w"] - mg["risk_rank_u"]
            parts.append(mg)
        if not parts: return None
        DL = pd.concat(parts, ignore_index=True)
        g = DL.groupby(["drug_key","drug"])
        AMP = g["rank_delta"].agg(mean_delta="mean", sd_delta="std", min_delta="min",
                                  max_delta="max", n_traits="count").reset_index()
        AMP["n_pos"] = g["rank_delta"].apply(lambda x: int((x>0).sum())).values
        AMP["n_neg"] = g["rank_delta"].apply(lambda x: int((x<0).sum())).values
        AMP["sign_test_p"] = [binom_p(max(r.n_pos, r.n_neg), r.n_pos+r.n_neg) for r in AMP.itertuples()]
        AMP["consistent_amplifier"] = (AMP["min_delta"]>0) & (AMP["n_traits"]==len(TRAITS))
        AMP["consistent_dominant"]  = (AMP["max_delta"]<0) & (AMP["n_traits"]==len(TRAITS))
        src = "rebuilt from the long rank table"
    else:
        return None
    for c in ["consistent_amplifier","consistent_dominant"]:
        if c in AMP.columns: AMP[c] = tobool(AMP[c]).values
    if "drug_key" not in AMP.columns: AMP["drug_key"] = AMP["drug"].map(norm_drug)

    P = AMP[[c for c in ["drug_key","drug","mean_delta","sd_delta","min_delta","max_delta",
                         "n_traits","n_pos","n_neg","sign_test_p","consistent_amplifier",
                         "consistent_dominant"] if c in AMP.columns]].copy()
    if CLS_IN is not None and not CLS_IN.empty:
        keep = [c for c in ["drug_key","pooled_mean_rank","pooled_sd_rank","genomics_signal",
                            "risk_tier","classification","resid_uniform","resid_weighted",
                            "adaptation_rank_gain"] if c in CLS_IN.columns]
        cl = CLS_IN[keep].copy()
        if "drug_key" not in cl.columns and "drug" in CLS_IN.columns:
            cl["drug_key"] = CLS_IN["drug"].map(norm_drug)
        P = P.merge(cl, on="drug_key", how="left")
    if "pooled_mean_rank" not in P.columns:
        pr = (LONG[LONG.method.isin([WMETHOD,UMETHOD])].dropna(subset=["risk_rank"])
              .groupby("drug_key")["risk_rank"].mean().rename("pooled_mean_rank").reset_index())
        P = P.merge(pr, on="drug_key", how="left")
    if "risk_tier" not in P.columns:
        nq = min(3, max(2, len(P)//3 or 2))
        P["risk_tier"] = pd.qcut(P["pooled_mean_rank"].rank(method="first"), q=nq,
                                 labels=["HIGH","INTERMEDIATE","LOW"][:nq]).astype(str)
    P["drug_class"] = P["drug"].map(drug_class)
    P["genomic_priority"] = np.where(
        P.get("consistent_amplifier", False).fillna(False) & (P["risk_tier"].astype(str)=="HIGH"),
        "1-Highest",
        np.where(P.get("consistent_amplifier", False).fillna(False), "2-High (consistent, lower tier)",
        np.where(P["mean_delta"] > 0, "3-Elevated (amplified on average)",
        np.where(P["mean_delta"] < 0, "5-Prior-dependent", "4-Neutral"))))
    P = P.sort_values(["genomic_priority","mean_delta"], ascending=[True, False]).reset_index(drop=True)
    save_table(P, "D1_genomic_priority.csv", "Genomic-consideration priority table")

    SEC("D1. GENOMIC-CONSIDERATION PRIORITY TABLE")
    S(f"Source of the amplification index: {src}.")
    S("mean_delta = rank_weighted - rank_uniform, averaged over traits.")
    S("POSITIVE mean_delta = the drug RISES when the literature receptor prior is removed,")
    S("i.e. its clinical-proxy rank UNDER-CALLS the genomic/affinity contribution.")
    S("")
    cols_present = "genomics_signal" in P.columns
    S(f"{'priority':<34}{'drug':<26}{'class':<7}{'meanΔ':>8}{'sd':>7}{'+/-':>7}"
      f"{'signP':>10}{'pooled#':>9}" + (f"{'gSignal':>9}" if cols_present else ""))
    S(SUB)
    for _, r in P.iterrows():
        line = (f"{str(r['genomic_priority']):<34}{str(r['drug'])[:25]:<26}{r['drug_class']:<7}"
                f"{fmt(r['mean_delta'],2):>8}{fmt(r.get('sd_delta'),2):>7}"
                f"{str(int(r.get('n_pos',0)))+'/'+str(int(r.get('n_neg',0))):>7}"
                f"{fmt(r.get('sign_test_p')):>10}{fmt(r.get('pooled_mean_rank'),1):>9}")
        if cols_present: line += f"{fmt(r.get('genomics_signal'),2):>9}"
        S(line)
    hi = P[P["genomic_priority"].str.startswith("1-")]
    if len(hi):
        SIG.append("[D1] HIGHEST genomic-consideration priority (high pooled risk AND rank rises in "
                   "every trait once the literature prior is removed): "
                   + ", ".join(f"{r['drug']} (+{fmt(r['mean_delta'],1)} ranks)" for _, r in hi.iterrows()) + ".")
    el = P[P["genomic_priority"].str.startswith("3-")]
    if len(el):
        SIG.append("[D1] Elevated priority (amplified on average, not in every trait): "
                   + ", ".join(el["drug"].head(10).astype(str)) + ".")
    pdep = P[P["genomic_priority"].str.startswith("5-")]
    if len(pdep):
        SIG.append("[D1] Prior-dependent drugs (their rank is sustained by the literature weights and "
                   "falls without them): " + ", ".join(pdep["drug"].head(10).astype(str)) + ".")
    PRIORITY = P
    return P
safe(d1, "D1")


# =============================================================================
# D2. SSRI/SNRI RANK SHIFT UNDER UNIFORM WEIGHTING
# =============================================================================
def d2():
    global DELTA_LONG
    DELTA_LONG = None
    if not HAVE_PAIR:
        S(""); S("D2 skipped — both a weighted and a uniform run are required."); return None
    parts = []
    for t in TRAITS:
        w = LONG[(LONG.method==WMETHOD)&(LONG.trait==t)][["drug_key","drug","risk_rank","risk_score"]]
        u = LONG[(LONG.method==UMETHOD)&(LONG.trait==t)][["drug_key","risk_rank","risk_score"]]
        mg = w.merge(u, on="drug_key", suffixes=("_w","_u"))
        if mg.empty: continue
        mg["trait"] = t
        mg["rank_delta"] = mg["risk_rank_w"] - mg["risk_rank_u"]
        mg["log2_score_ratio"] = np.log2((mg["risk_score_u"]/mg["risk_score_w"]).replace(0, np.nan))
        parts.append(mg)
    DELTA_LONG = pd.concat(parts, ignore_index=True)
    DELTA_LONG["drug_class"] = DELTA_LONG["drug"].map(drug_class)
    save_table(DELTA_LONG, "D2a_rank_delta_long.csv", "Per drug x trait rank delta (weighted - uniform)")

    ss = DELTA_LONG[DELTA_LONG["drug"].str.contains(SSRI_SNRI_REGEX, case=False, na=False)]
    summ = (ss.groupby(["drug_key","drug"])
              .agg(mean_delta=("rank_delta","mean"), sd_delta=("rank_delta","std"),
                   min_delta=("rank_delta","min"), max_delta=("rank_delta","max"),
                   n_traits=("trait","nunique"),
                   median_log2_score_ratio=("log2_score_ratio","median")).reset_index())
    summ["consistent_up"]   = summ["min_delta"] > 0
    summ["consistent_down"] = summ["max_delta"] < 0
    summ["drug_class"] = summ["drug"].map(drug_class)
    summ = summ.sort_values("mean_delta", ascending=False).reset_index(drop=True)
    save_table(summ, "D2b_ssri_snri_rank_shift.csv", "SSRI/SNRI rank shift weighted -> uniform")

    SEC("D2. SSRI / SNRI RANK SHIFT UNDER GENOMICS-INFLUENCED (UNIFORM) WEIGHTING")
    S("Positive mean Δrank = the drug becomes RELATIVELY RISKIER once every receptor with Ki data")
    S("is given equal a-priori weight (i.e. the literature prior was suppressing it).")
    S("")
    S(f"{'drug':<26}{'class':<7}{'meanΔ':>8}{'sd':>7}{'min':>6}{'max':>6}{'n':>4}"
      f"{'log2(u/w)':>11}  flag")
    S(SUB)
    for _, r in summ.iterrows():
        flag = "UP(all traits)" if r["consistent_up"] else ("DOWN(all traits)" if r["consistent_down"] else "")
        S(f"{str(r['drug'])[:25]:<26}{r['drug_class']:<7}{fmt(r['mean_delta'],2):>8}"
          f"{fmt(r['sd_delta'],2):>7}{fmt(r['min_delta'],0):>6}{fmt(r['max_delta'],0):>6}"
          f"{int(r['n_traits']):>4}{fmt(r['median_log2_score_ratio'],2):>11}  {flag}")

    # class-level paired test: are SSRI/SNRI shifts different from the rest?
    ssd = summ["mean_delta"].dropna()
    other = (DELTA_LONG[~DELTA_LONG["drug"].str.contains(SSRI_SNRI_REGEX, case=False, na=False)]
             .groupby("drug")["rank_delta"].mean().dropna())
    p_mw = np.nan
    if HAVE_SCIPY and len(ssd) >= 3 and len(other) >= 3:
        try: p_mw = float(mannwhitneyu(ssd, other, alternative="two-sided").pvalue)
        except Exception: p_mw = np.nan
    S("")
    S(f"Mean Δrank, SSRI/SNRI class : {fmt(ssd.mean(),2)} (n={len(ssd)})")
    S(f"Mean Δrank, all other drugs : {fmt(other.mean(),2)} (n={len(other)})")
    S(f"Mann-Whitney U p            : {fmt(p_mw)} {stars(p_mw)}")
    nup = int((ssd > 0).sum()); p_sign = binom_p(max(nup, len(ssd)-nup), len(ssd))
    S(f"SSRIs/SNRIs rising under uniform weights: {nup}/{len(ssd)}  (sign-test p={fmt(p_sign)} {stars(p_sign)})")
    if not np.isnan(p_mw) and p_mw < ALPHA:
        direction = "HIGHER" if ssd.mean() > other.mean() else "LOWER"
        SIG.append(f"[D2] The SSRI/SNRI class shows a significantly {direction} genomic amplification than "
                   f"the remaining antidepressants (mean Δrank {fmt(ssd.mean(),2)} vs {fmt(other.mean(),2)}; "
                   f"Mann-Whitney p={fmt(p_mw)} {stars(p_mw)}).")
    else:
        SIG.append(f"[D2] SSRI/SNRI genomic amplification does not differ significantly from the other "
                   f"antidepressants (mean Δrank {fmt(ssd.mean(),2)} vs {fmt(other.mean(),2)}, p={fmt(p_mw)}).")
    cu = summ[summ["consistent_up"]]
    if len(cu):
        SIG.append("[D2] SSRIs/SNRIs that rise in EVERY trait without the literature prior: "
                   + ", ".join(f"{r['drug']} (+{fmt(r['mean_delta'],1)})" for _, r in cu.iterrows()) + ".")

    SUBSEC("MAGNITUDE OF THE WEIGHTED -> UNIFORM SHIFT, BY TRAIT (all drugs)")
    S(f"{'trait':<16}{'n':>5}{'mean|Δ|':>10}{'median|Δ|':>11}{'max|Δ|':>9}{'n moved':>9}")
    for t, grp in DELTA_LONG.groupby("trait"):
        S(f"{t:<16}{len(grp):>5}{fmt(grp['rank_delta'].abs().mean(),2):>10}"
          f"{fmt(grp['rank_delta'].abs().median(),2):>11}{fmt(grp['rank_delta'].abs().max(),0):>9}"
          f"{int((grp['rank_delta']!=0).sum()):>9}")
    return summ
SSRI_SHIFT = safe(d2, "D2")


# =============================================================================
# D3. RECEPTOR DRIVERS OF THE LARGEST MOVERS
# =============================================================================
def d3():
    if DRV is None or DRV.empty or not HAVE_PAIR or AMP is None:
        S(""); S("D3 skipped — receptor contributions or the method pair are unavailable."); return None
    SEC("D3. RECEPTOR DRIVERS OF THE LARGEST MOVERS")
    S("share_pct = the receptor's percentage of that drug's total weighted+TWAS-scaled contribution.")
    S("delta = uniform share - weighted share (positive = the receptor GAINS influence without the prior).")

    movers = (AMP.reindex(AMP["mean_delta"].abs().sort_values(ascending=False).index)
                 .head(MOVERS_N))
    rows = []
    for _, mv in movers.iterrows():
        dk = mv.get("drug_key", norm_drug(mv["drug"]))
        sub = DRV[(DRV.drug_key == dk) & (DRV.method.isin([WMETHOD, UMETHOD]))]
        if sub.empty: continue
        piv = sub.pivot_table(index="gene", columns="method", values="share_pct", aggfunc="mean").fillna(0)
        if not {WMETHOD, UMETHOD}.issubset(piv.columns): continue
        piv["delta"] = piv[UMETHOD] - piv[WMETHOD]
        piv = piv.sort_values("delta", ascending=False)
        kv = sub.groupby("gene")["Ki_nM"].min()
        S(""); S(f"  {mv['drug']}   mean Δrank = {fmt(mv['mean_delta'],2)}   "
                 f"({'GENOMICALLY AMPLIFIED' if mv['mean_delta']>0 else 'PRIOR-DEPENDENT'})")
        S(f"    {'gene':<10}{'lit w':>7}{'Ki nM':>10}{'weighted %':>12}{'uniform %':>11}{'delta':>9}")
        for gname, r in pd.concat([piv.head(5), piv.tail(3)]).drop_duplicates().iterrows():
            S(f"    {str(gname)[:9]:<10}{fmt(METABOLIC_WEIGHTS.get(gname),2):>7}"
              f"{fmt(kv.get(gname),2):>10}{fmt(r[WMETHOD],1):>12}{fmt(r[UMETHOD],1):>11}"
              f"{fmt(r['delta'],1):>9}")
        rows.append({"drug": mv["drug"], "mean_rank_delta": mv["mean_delta"],
                     "top_gaining_gene": piv.index[0], "gain_delta": piv.iloc[0]["delta"],
                     "top_losing_gene": piv.index[-1], "loss_delta": piv.iloc[-1]["delta"],
                     "weighted_dominant_gene": piv[WMETHOD].idxmax(),
                     "uniform_dominant_gene": piv[UMETHOD].idxmax()})
        SIG.append(f"[D3] {mv['drug']} (mean Δrank {fmt(mv['mean_delta'],1)}): influence shifts toward "
                   f"{piv.index[0]} (+{fmt(piv.iloc[0]['delta'],1)} pts) and away from {piv.index[-1]} "
                   f"({fmt(piv.iloc[-1]['delta'],1)} pts); dominant driver "
                   f"{piv[WMETHOD].idxmax()} -> {piv[UMETHOD].idxmax()}.")
    if rows:
        save_table(pd.DataFrame(rows), "D3_mover_driver_contrast.csv",
                   "Top gaining / losing receptor per major mover")

    # pooled gene-level contrast
    pooled = (DRV[DRV.method.isin([WMETHOD, UMETHOD])]
                .groupby(["method","trait","gene"])["contribution"].sum().reset_index())
    pooled["share"] = pooled.groupby(["method","trait"])["contribution"].transform(lambda x: 100*x/x.sum())
    pw = pooled.pivot_table(index=["gene","trait"], columns="method", values="share").reset_index().fillna(0)
    if {WMETHOD, UMETHOD}.issubset(pw.columns):
        pw["delta_share"] = pw[UMETHOD] - pw[WMETHOD]
        GD = (pw.groupby("gene").agg(mean_weighted=(WMETHOD,"mean"), mean_uniform=(UMETHOD,"mean"),
                                     mean_delta=("delta_share","mean"), min_delta=("delta_share","min"),
                                     max_delta=("delta_share","max")).reset_index())
        GD["metabolic_weight"] = GD["gene"].map(METABOLIC_WEIGHTS)
        GD["consistent_gain"] = GD["min_delta"] > 0
        GD["consistent_loss"] = GD["max_delta"] < 0
        GD = GD.sort_values("mean_delta", ascending=False).reset_index(drop=True)
        save_table(GD, "D3b_pooled_gene_share_delta.csv", "Pooled receptor share change")
        SUBSEC("POOLED RECEPTOR-SHARE CONTRAST (all drugs, averaged over traits)")
        S(f"{'gene':<10}{'lit w':>7}{'weighted %':>12}{'uniform %':>11}{'delta':>9}  flag")
        for _, r in GD.iterrows():
            flag = "GAIN(all)" if r["consistent_gain"] else ("LOSS(all)" if r["consistent_loss"] else "")
            S(f"{str(r['gene'])[:9]:<10}{fmt(r['metabolic_weight'],2):>7}{fmt(r['mean_weighted'],2):>12}"
              f"{fmt(r['mean_uniform'],2):>11}{fmt(r['mean_delta'],2):>9}  {flag}")
        if len(GD):
            SIG.append(f"[D3] Pooled dominant driver switches from {GD.sort_values('mean_weighted', ascending=False).iloc[0]['gene']} "
                       f"(clinical proxy) to {GD.sort_values('mean_uniform', ascending=False).iloc[0]['gene']} "
                       f"(genomics-influenced); biggest gainer {GD.iloc[0]['gene']} "
                       f"(+{fmt(GD.iloc[0]['mean_delta'],2)} share pts), biggest loser "
                       f"{GD.iloc[-1]['gene']} ({fmt(GD.iloc[-1]['mean_delta'],2)} pts).")
        return GD
    return None
GENE_CONTRAST = safe(d3, "D3")


# =============================================================================
# D4. TOP-K STABILITY: CLINICAL PROXY vs GENOMICS-INFLUENCED
# =============================================================================
def d4():
    SEC("D4. TOP-K STABILITY — CLINICAL PROXY vs GENOMICS-INFLUENCED")
    mr = {}
    for m in METHODS:
        piv = LONG[LONG.method==m].pivot_table(index="drug_key", columns="trait", values="risk_rank").dropna(how="any")
        if piv.empty: continue
        mr[m] = piv.mean(axis=1).sort_values()
    for m, s_ in mr.items():
        SUBSEC(f"TRANSDIAGNOSTIC TOP {TOP_K} — {m}")
        S(f"{'#':>3}  {'drug':<28}{'class':<7}{'mean rank':>11}")
        for i, k in enumerate(s_.index[:TOP_K], 1):
            S(f"{i:>3}  {DRUG_LABEL[k][:27]:<28}{drug_class(DRUG_LABEL[k]):<7}{fmt(s_[k],2):>11}")
    if not (WMETHOD in mr and UMETHOD in mr): return None
    A = set(mr[WMETHOD].index[:TOP_K]); B = set(mr[UMETHOD].index[:TOP_K])
    inter, only_u, only_w = A & B, B - A, A - B
    N = len(set(RANK_WIDE.index))
    p_over = float(hypergeom.sf(len(inter)-1, N, TOP_K, TOP_K)) if (HAVE_SCIPY and N >= TOP_K) else np.nan
    jac = len(inter)/len(A | B) if (A | B) else np.nan
    rho, p_rho = (spearmanr(mr[WMETHOD].reindex(sorted(A|B|set(mr[WMETHOD].index))).dropna().rank(),
                            mr[UMETHOD].reindex(sorted(A|B|set(mr[UMETHOD].index))).dropna().rank())
                  if HAVE_SCIPY else (np.nan, np.nan))
    common = mr[WMETHOD].index.intersection(mr[UMETHOD].index)
    if HAVE_SCIPY and len(common) >= 5:
        rho, p_rho = spearmanr(mr[WMETHOD].loc[common], mr[UMETHOD].loc[common])
    SUBSEC("SET ALGEBRA AND GLOBAL CONCORDANCE")
    S(f"Overlap of the two top-{TOP_K} lists : {len(inter)}/{TOP_K}   Jaccard={fmt(jac,2)}   "
      f"hypergeometric p={fmt(p_over)} {stars(p_over)}")
    S(f"Global Spearman of transdiagnostic mean ranks: rho={fmt(rho)}  p={fmt(p_rho)} {stars(p_rho)}  "
      f"(n={len(common)})")
    S("")
    S("ROBUST DUAL-EVIDENCE SET  : " + (", ".join(sorted(DRUG_LABEL[k] for k in inter)) or "none"))
    S("GENOMICALLY AMPLIFIED SET : " + (", ".join(sorted(DRUG_LABEL[k] for k in only_u)) or "none"))
    S("PRIOR-DEPENDENT SET       : " + (", ".join(sorted(DRUG_LABEL[k] for k in only_w)) or "none"))
    tab = pd.DataFrame({"drug": [DRUG_LABEL[k] for k in sorted(A|B)],
                        "drug_class": [drug_class(DRUG_LABEL[k]) for k in sorted(A|B)],
                        "mean_rank_weighted": [mr[WMETHOD].get(k, np.nan) for k in sorted(A|B)],
                        "mean_rank_uniform":  [mr[UMETHOD].get(k, np.nan) for k in sorted(A|B)],
                        "in_weighted_topk": [k in A for k in sorted(A|B)],
                        "in_uniform_topk":  [k in B for k in sorted(A|B)],
                        "category": ["robust (both)" if (k in A and k in B) else
                                     "genomically amplified (uniform only)" if k in B else
                                     "prior-dependent (weighted only)" for k in sorted(A|B)]})
    save_table(tab, "D4_topk_sets.csv", "Top-k membership, both rankings")
    SIG.append(f"[D4] {len(inter)} of the top-{TOP_K} clinical-proxy antidepressants remain in the top-{TOP_K} "
               f"under the genomics-influenced ranking (Jaccard {fmt(jac,2)}, hypergeom p={fmt(p_over)}): "
               + ", ".join(sorted(DRUG_LABEL[k] for k in inter)) + ".")
    if only_u:
        SIG.append("[D4] Enter the top-k ONLY once the literature prior is removed (hypothesis-generating): "
                   + ", ".join(sorted(DRUG_LABEL[k] for k in only_u)) + ".")
    if only_w:
        SIG.append("[D4] Top-k ONLY under the literature weights (prior-dependent, require verification): "
                   + ", ".join(sorted(DRUG_LABEL[k] for k in only_w)) + ".")
    if not np.isnan(rho):
        SIG.append(f"[D4] Global rank concordance between the clinical-proxy and genomics-influenced "
                   f"rankings: rho={fmt(rho)} (p={fmt(p_rho)} {stars(p_rho)}) — "
                   f"{'the prior largely determines the ordering' if rho > 0.9 else 'the weighting choice materially re-orders drugs'}.")
    return tab, mr
D4_OUT = safe(d4, "D4")
MEANRANK = D4_OUT[1] if isinstance(D4_OUT, tuple) else {}


# =============================================================================
# D5. CHRONIC 5-HT ADAPTATION EFFECT
# =============================================================================
def d5():
    if ADAPT_LNG is None or ADAPT_LNG.empty:
        S(""); S("D5 skipped — no 5-HT adaptation comparison tables found."); return None
    SEC("D5. CHRONIC 5-HT ADAPTATION (SERT -> 5-HT2C/2A UP, 5-HT1A DOWN)")
    S("adaptation_delta = adapted_risk - base_risk (normalised score points, amitriptyline = 100).")
    S("rank_delta       = base_rank - adapted_rank (POSITIVE = the drug becomes RELATIVELY RISKIER).")
    A = ADAPT_LNG.copy()
    A["drug_class"] = A["drug"].map(drug_class)
    agg = (A.groupby(["drug","drug_class"])
             .agg(mean_delta=("adaptation_delta","mean"), sd_delta=("adaptation_delta","std"),
                  mean_rank_gain=("rank_delta","mean"), sd_rank_gain=("rank_delta","std"),
                  sert_affinity=("sert_affinity","max"), adapted=("adapted","max"),
                  n_traits=("trait","nunique")).reset_index()
             .sort_values("mean_delta", ascending=False))
    save_table(agg, "D5a_5ht_adaptation_per_drug.csv", "Mean 5-HT adaptation effect per drug")
    S("")
    S(f"{'drug':<26}{'class':<7}{'SERT':>9}{'adapted':>9}{'meanΔrisk':>11}{'sd':>7}"
      f"{'mean rank gain':>16}")
    S(SUB)
    for _, r in agg.iterrows():
        S(f"{str(r['drug'])[:25]:<26}{r['drug_class']:<7}{fmt(r['sert_affinity'],3):>9}"
          f"{str(bool(r['adapted'])):>9}{fmt(r['mean_delta'],2):>11}{fmt(r['sd_delta'],2):>7}"
          f"{fmt(r['mean_rank_gain'],2):>16}")

    adp = agg[agg["adapted"] == True]
    non = agg[agg["adapted"] != True]
    S("")
    S(f"Drugs triggering the adaptation : {len(adp)}/{len(agg)}")
    S(f"Mean Δrisk among adapted drugs  : {fmt(adp['mean_delta'].mean(),3)}")
    S(f"Mean Δrisk among non-adapted    : {fmt(non['mean_delta'].mean(),3)}   "
      f"(non-zero only through re-normalisation to the reference drug)")
    p_w = np.nan
    if HAVE_SCIPY and len(A.dropna(subset=["base_risk","adapted_risk"])) >= 6:
        try:
            aa = A.dropna(subset=["base_risk","adapted_risk"])
            p_w = float(wilcoxon(aa["base_risk"], aa["adapted_risk"])[1])
        except Exception: p_w = np.nan
    S(f"Wilcoxon (base vs adapted risk, all drug x trait rows): p={fmt(p_w)} {stars(p_w)}")
    up = adp[adp["mean_rank_gain"] > 0]
    if len(up):
        SIG.append("[D5] Chronic 5-HT adaptation raises the metabolic rank of: "
                   + ", ".join(f"{r['drug']} (+{fmt(r['mean_rank_gain'],1)} ranks, "
                               f"Δrisk {fmt(r['mean_delta'],1)})" for _, r in up.head(8).iterrows()) + ".")
    ss = agg[agg["drug"].str.contains(SSRI_SNRI_REGEX, case=False, na=False)]
    if len(ss):
        S("")
        S(f"SSRI/SNRI subset: n={len(ss)}, mean Δrisk={fmt(ss['mean_delta'].mean(),3)}, "
          f"mean rank gain={fmt(ss['mean_rank_gain'].mean(),2)}")
        SIG.append(f"[D5] Among SSRIs/SNRIs the chronic 5-HT re-weighting changes the risk score by "
                   f"{fmt(ss['mean_delta'].mean(),2)} points on average (mean rank gain "
                   f"{fmt(ss['mean_rank_gain'].mean(),2)}), i.e. acute Ki-only scoring "
                   f"{'UNDER-' if ss['mean_delta'].mean() > 0 else 'OVER-'}estimates their long-term liability.")
        save_table(ss, "D5b_5ht_adaptation_ssri_snri.csv", "5-HT adaptation, SSRI/SNRI subset")
    # class-level
    SUBSEC("5-HT ADAPTATION EFFECT BY DRUG CLASS")
    S(f"{'class':<8}{'n':>5}{'mean Δrisk':>12}{'median':>10}{'mean rank gain':>16}{'n adapted':>11}")
    for cls, grp in agg.groupby("drug_class"):
        S(f"{cls:<8}{len(grp):>5}{fmt(grp['mean_delta'].mean(),2):>12}"
          f"{fmt(grp['mean_delta'].median(),2):>10}{fmt(grp['mean_rank_gain'].mean(),2):>16}"
          f"{int(grp['adapted'].sum()):>11}")
    return agg
ADAPT_AGG = safe(d5, "D5")


# =============================================================================
# D6. ONE-PAGE DECISION TABLE
# =============================================================================
def d6():
    if PRIORITY is None:
        S(""); S("D6 skipped — priority table unavailable."); return None
    SEC("D6. ONE-PAGE DECISION TABLE")
    D = PRIORITY.copy()
    if ADAPT_AGG is not None:
        D = D.merge(ADAPT_AGG[["drug","mean_rank_gain","adapted"]]
                    .rename(columns={"mean_rank_gain":"adapt_rank_gain","adapted":"adapt_fired"}),
                    on="drug", how="left")
    tier_order = {"HIGH":0, "INTERMEDIATE":1, "LOW":2}
    D["_t"] = D["risk_tier"].astype(str).map(tier_order).fillna(9)
    D = D.sort_values(["_t","mean_delta"], ascending=[True, False]).drop(columns="_t")
    cols = [c for c in ["drug","drug_class","risk_tier","classification","genomic_priority",
                        "pooled_mean_rank","mean_delta","sign_test_p","genomics_signal",
                        "resid_uniform","adapt_rank_gain","consistent_amplifier"] if c in D.columns]
    save_table(D[cols], "D6_decision_table.csv", "One-page decision table")
    S(f"{'drug':<24}{'cls':<6}{'tier':<14}{'pooled#':>8}{'meanΔ':>8}"
      + (f"{'gSig':>7}" if "genomics_signal" in D.columns else "")
      + (f"{'adaptΔ':>8}" if "adapt_rank_gain" in D.columns else "")
      + "  priority / classification")
    S(SUB)
    for _, r in D.iterrows():
        line = (f"{str(r['drug'])[:23]:<24}{r['drug_class']:<6}{str(r.get('risk_tier'))[:13]:<14}"
                f"{fmt(r.get('pooled_mean_rank'),1):>8}{fmt(r.get('mean_delta'),1):>8}")
        if "genomics_signal" in D.columns: line += f"{fmt(r.get('genomics_signal'),2):>7}"
        if "adapt_rank_gain" in D.columns: line += f"{fmt(r.get('adapt_rank_gain'),1):>8}"
        line += f"  {r.get('genomic_priority','')}"
        if "classification" in D.columns and pd.notna(r.get("classification")):
            line += f" | {r['classification']}"
        S(line)
    return D
DECISION = safe(d6, "D6")


# =============================================================================
# D7. DRUG-CLASS ANALYSIS
# =============================================================================
def d7():
    SEC("D7. DRUG-CLASS ANALYSIS OF THE METABOLIC-RISK RANKING")
    out = []
    for m in METHODS:
        sub = LONG[LONG.method == m].dropna(subset=["risk_score"])
        if sub.empty: continue
        per = (sub.groupby(["drug_key","drug","drug_class"])
                  .agg(mean_score=("risk_score","mean"), mean_rank=("risk_rank","mean")).reset_index())
        SUBSEC(f"CLASS SUMMARY — {m}")
        S(f"{'class':<8}{'n':>5}{'median score':>14}{'IQR':>10}{'mean rank':>11}{'best drug':<22}")
        groups = []
        for cls, grp in per.groupby("drug_class"):
            best = grp.loc[grp["mean_rank"].idxmin(), "drug"] if len(grp) else "NA"
            S(f"{cls:<8}{len(grp):>5}{fmt(grp['mean_score'].median(),2):>14}"
              f"{fmt(grp['mean_score'].quantile(.75)-grp['mean_score'].quantile(.25),2):>10}"
              f"{fmt(grp['mean_rank'].mean(),2):>11}  {str(best)[:20]:<22}")
            groups.append((cls, grp["mean_rank"].values))
            out.append({"method": m, "drug_class": cls, "n": len(grp),
                        "median_score": grp["mean_score"].median(),
                        "mean_rank": grp["mean_rank"].mean(), "best_drug": best})
        big = [g for _, g in groups if len(g) >= 3]
        if HAVE_SCIPY and len(big) >= 2:
            try:
                H, p = kruskal(*big)
                S(f"  Kruskal-Wallis across classes (n>=3): H={fmt(H,2)}, p={fmt(p)} {stars(p)}")
                if p < ALPHA:
                    ranked = sorted([(c, np.mean(g)) for c, g in groups if len(g) >= 3], key=lambda x: x[1])
                    SIG.append(f"[D7] {m}: antidepressant CLASS significantly stratifies metabolic-risk rank "
                               f"(Kruskal-Wallis p={fmt(p)} {stars(p)}); riskiest class = {ranked[0][0]} "
                               f"(mean rank {fmt(ranked[0][1],1)}), safest = {ranked[-1][0]} "
                               f"(mean rank {fmt(ranked[-1][1],1)}).")
            except Exception: pass
    if out: save_table(pd.DataFrame(out), "D7_drug_class_summary.csv", "Risk by drug class and method")
    return out
safe(d7, "D7")


# =============================================================================
# D8. PER-DRUG TWAS LEVERAGE (with vs without the exponential layer)
# =============================================================================
def d8():
    if DRV is None or DRV.empty or "contribution_no_twas" not in DRV.columns:
        S(""); S("D8 skipped — contribution_no_twas not available."); return None
    SEC("D8. PER-DRUG TWAS LEVERAGE (WITH vs WITHOUT THE EXPONENTIAL LAYER)")
    S("amplification = sum(contribution with TWAS scale) / sum(contribution without it).")
    S("A value > 1 means the genomic layer inflates that drug relative to a pure affinity x weight score.")
    rows = []
    for (m, t), grp in DRV.groupby(["method","trait"]):
        agg = grp.groupby("drug").agg(w=("contribution","sum"), wo=("contribution_no_twas","sum")).reset_index()
        agg = agg[agg["wo"] > 0]
        if len(agg) < 4: continue
        agg["amp"] = agg["w"]/agg["wo"]
        agg["rank_with"]    = agg["w"].rank(ascending=False, method="min")
        agg["rank_without"] = agg["wo"].rank(ascending=False, method="min")
        agg["rank_gain"]    = agg["rank_without"] - agg["rank_with"]
        agg["method"], agg["trait"] = m, t
        rows.append(agg)
    if not rows: return None
    LEV = pd.concat(rows, ignore_index=True)
    save_table(LEV, "D8a_twas_leverage_long.csv", "Per drug x run TWAS amplification")
    per = (LEV.groupby("drug").agg(mean_amp=("amp","mean"), sd_amp=("amp","std"),
                                   mean_rank_gain=("rank_gain","mean"),
                                   max_rank_gain=("rank_gain","max"),
                                   min_rank_gain=("rank_gain","min"), n=("trait","count"))
             .reset_index().sort_values("mean_amp", ascending=False))
    per["drug_class"] = per["drug"].map(drug_class)
    save_table(per, "D8b_twas_leverage_per_drug.csv", "Mean TWAS amplification per drug")
    S("")
    S(f"{'drug':<26}{'class':<7}{'mean amp':>10}{'sd':>8}{'mean rank gain':>16}{'range':>14}")
    S(SUB)
    for _, r in per.iterrows():
        S(f"{str(r['drug'])[:25]:<26}{r['drug_class']:<7}{fmt(r['mean_amp'],4):>10}"
          f"{fmt(r['sd_amp'],4):>8}{fmt(r['mean_rank_gain'],2):>16}"
          f"{(str(int(r['min_rank_gain']))+'..'+str(int(r['max_rank_gain']))):>14}")
    top = per.head(3); bot = per.tail(3)
    SIG.append("[D8] Most TWAS-amplified drugs (largest score inflation from the exponential genomic "
               "layer): " + ", ".join(f"{r['drug']} (x{fmt(r['mean_amp'],4)})" for _, r in top.iterrows())
               + "; least amplified: " + ", ".join(f"{r['drug']} (x{fmt(r['mean_amp'],4)})" for _, r in bot.iterrows()) + ".")
    movers = per[per["mean_rank_gain"].abs() >= 1]
    if len(movers):
        SIG.append("[D8] Drugs whose rank moves by >=1 position purely because of the TWAS layer: "
                   + ", ".join(f"{r['drug']} ({fmt(r['mean_rank_gain'],1):+})" for _, r in movers.iterrows()) + ".")
    else:
        SIG.append("[D8] The TWAS exponential layer alone does not move any drug by a full rank position on "
                   "average — with beta=0.045 it modulates rather than drives the ordering.")
    return per
LEVERAGE = safe(d8, "D8")


# =============================================================================
# D9. RECEPTOR-GENE TWAS PANEL WITH BH-FDR
# =============================================================================
def d9():
    if DRV is None or DRV.empty or "twas_p" not in DRV.columns:
        S(""); S("D9 skipped — no TWAS p-values in the contribution table."); return None
    SEC("D9. RECEPTOR-GENE TWAS PANEL (BH-FDR WITHIN THE RECEPTOR PANEL)")
    S("NOTE: p-values here are those actually used by the scorer, i.e. AFTER the parent pipeline's")
    S("fairness imputation for panel genes with no TWAS record. Rows flagged TwImp are imputed and")
    S("are deliberately NOT counted as significant.")
    frames = []
    for t, grp in DRV[DRV.method == (WMETHOD if WMETHOD in METHODS else METHODS[0])].groupby("trait"):
        g = (grp.groupby("gene")
                .agg(twas_z=("twas_z","first"), twas_p=("twas_p","first"),
                     n_drugs=("drug","nunique"), any_imputed=("twas_imputed","max"),
                     any_sig=("twas_significant","max")).reset_index())
        g["neglog10_p"] = -np.log10(pd.to_numeric(g["twas_p"], errors="coerce").clip(lower=1e-300))
        g["twas_q_BH"] = bh_fdr(g["twas_p"].values)
        g["twas_scale_ref"] = np.exp(LOGP_BETA_REF * g["neglog10_p"].clip(upper=MAX_NEGLOG10_P))
        g["metabolic_weight"] = g["gene"].map(METABOLIC_WEIGHTS)
        g["trait"] = t
        frames.append(g)
    if not frames: return None
    G = pd.concat(frames, ignore_index=True)
    save_table(G, "D9a_receptor_gene_twas_panel.csv", "Receptor-gene TWAS z/p/q per trait")
    for t, g in G.groupby("trait"):
        gg = g.sort_values("twas_p")
        n_nom = int(((gg["twas_p"] < 0.05) & (~gg["any_imputed"].astype(bool))).sum())
        n_fdr = int(((gg["twas_q_BH"] < FDR_ALPHA) & (~gg["any_imputed"].astype(bool))).sum())
        SUBSEC(f"{t}   genes={len(gg)}   nominal p<0.05: {n_nom}   BH-FDR q<{FDR_ALPHA}: {n_fdr}")
        S(f"{'gene':<10}{'z':>9}{'p':>12}{'q(BH)':>12}{'-log10p':>10}{'scale':>9}{'lit w':>8}"
          f"{'drugs':>7}{'imp':>5}")
        for _, r in gg.head(12).iterrows():
            S(f"{str(r['gene'])[:9]:<10}{fmt(r['twas_z']):>9}{fmt(r['twas_p']):>12}"
              f"{fmt(r['twas_q_BH']):>12}{fmt(r['neglog10_p'],2):>10}{fmt(r['twas_scale_ref'],4):>9}"
              f"{fmt(r['metabolic_weight'],2):>8}{int(r['n_drugs']):>7}"
              f"{('Y' if bool(r['any_imputed']) else '-'):>5}")
        for _, r in gg[(gg["twas_q_BH"] < FDR_ALPHA) & (~gg["any_imputed"].astype(bool))].iterrows():
            SIG.append(f"[D9][GENE] {t}: {r['gene']} survives BH-FDR within the receptor panel "
                       f"(z={fmt(r['twas_z'])}, p={fmt(r['twas_p'])}, q={fmt(r['twas_q_BH'])}); "
                       f"literature weight {fmt(r['metabolic_weight'],2)}; affects {int(r['n_drugs'])} drug(s).")
    piv = G.pivot_table(index="gene", columns="trait", values="neglog10_p", aggfunc="max")
    piv["mean_neglog10p"] = piv.mean(axis=1)
    piv = piv.sort_values("mean_neglog10p", ascending=False)
    save_table(piv.reset_index(), "D9b_gene_neglog10p_across_traits.csv", "Panel -log10 p across traits")
    SUBSEC("RECEPTOR GENES RANKED BY MEAN -log10(p) ACROSS TRAITS")
    S("  " + f"{'gene':<10}" + "".join(f"{str(c)[:12]:>13}" for c in piv.columns))
    for gn, row in piv.head(15).iterrows():
        S("  " + f"{str(gn)[:9]:<10}" + "".join(f"{fmt(row[c],2):>13}" for c in piv.columns))
    if len(piv):
        SIG.append(f"[D9] Receptor gene with the strongest average TWAS evidence across the four traits: "
                   f"{piv.index[0]} (mean -log10 p = {fmt(piv.iloc[0]['mean_neglog10p'],2)}).")
    return G
safe(d9, "D9")


# =============================================================================
# D10. EXACT REPRODUCTION CHECK + BETA SENSITIVITY SWEEP
# =============================================================================
def rescore_from_detail(det, beta, cap=MAX_NEGLOG10_P, sec=SECONDARY_BOOST_FAC,
                        sigb=TWAS_SIG_BONUS, ref=REFERENCE_DRUG):
    """Reproduce the parent scorer from the stored contribution table alone."""
    d = det.copy()
    p = pd.to_numeric(d["twas_p"], errors="coerce").clip(lower=1e-300)
    nlp = (-np.log10(p)).clip(upper=cap).fillna(0.0)
    d["_c"] = d["affinity_log1p"] * d["receptor_weight"] * np.exp(beta * nlp)
    base = d.groupby("drug")["_c"].sum().reset_index().rename(columns={"_c":"base_score"})
    z = (d.groupby("drug")["twas_z"].apply(lambda x: x.abs().max() if x.notna().any() else 0.0)
           .reset_index().rename(columns={"twas_z":"maxz"}))
    base = base.merge(z, on="drug", how="left"); base["maxz"] = base["maxz"].fillna(0.0)
    sg = d[d["twas_significant"].astype(bool) & (d["receptor_weight"] > 0)] \
            .groupby("drug").size().reset_index(name="nsig")
    base = base.merge(sg, on="drug", how="left"); base["nsig"] = base["nsig"].fillna(0)
    base["raw"] = base["base_score"] * (1 + sec*base["maxz"]) * (1 + sigb*base["nsig"])
    r = base.loc[base["drug"].astype(str).str.lower() == ref.lower(), "raw"]
    den = r.values[0] if (len(r) and r.values[0] > 0) else base["raw"].max()
    base["score"] = base["raw"]/den*100 if (den and den > 0) else base["raw"]
    base = base[base["score"] != 0].copy()
    base["rank"] = base["score"].rank(ascending=False, method="min")
    return base[["drug","base_score","score","rank"]]

def d10():
    if DRV is None or DRV.empty or not {"affinity_log1p","receptor_weight","twas_p"}.issubset(DRV.columns):
        S(""); S("D10 skipped — the contribution table lacks the columns needed to re-score."); return None
    SEC("D10. REPRODUCTION CHECK AND BETA-SENSITIVITY SWEEP")
    S("The stored receptor_contributions.csv contains affinity_log1p, receptor_weight, twas_p (post-")
    S("fairness-imputation), twas_z and twas_significant, so the parent scorer can be reproduced")
    S("EXACTLY from outputs alone — and re-run under alternative beta values without touching any")
    S("upstream data. First the reproduction is verified against the stored scores (regression test).")

    # ---- reproduction check --------------------------------------------------
    chk = []
    for (m, t), det in [(k, DRV[(DRV.method==k[0])&(DRV.trait==k[1])]) for k in RISK.keys()]:
        if det.empty: continue
        rep = rescore_from_detail(det, LOGP_BETA_REF)
        obs = RISK[(m, t)][["drug","metabolic_risk_score","risk_rank"]]
        mg = rep.merge(obs, on="drug", how="inner")
        if mg.empty: continue
        dev = (mg["score"] - mg["metabolic_risk_score"]).abs()
        rk  = (mg["rank"] - mg["risk_rank"]).abs()
        chk.append({"method": m, "trait": t, "n": len(mg),
                    "max_abs_score_dev": float(dev.max()), "mean_abs_score_dev": float(dev.mean()),
                    "n_rank_mismatch": int((rk > 0).sum())})
    CHK = pd.DataFrame(chk)
    if not CHK.empty:
        save_table(CHK, "D10a_reproduction_check.csv", "Reproduction of stored scores from contributions")
        SUBSEC("REGRESSION TEST — reproduced (beta=0.045) vs stored scores")
        S(f"{'method':<26}{'trait':<16}{'n':>5}{'max |dev|':>12}{'mean |dev|':>12}{'rank mismatches':>18}")
        for _, r in CHK.iterrows():
            S(f"{r['method']:<26}{r['trait']:<16}{int(r['n']):>5}{fmt(r['max_abs_score_dev'],6):>12}"
              f"{fmt(r['mean_abs_score_dev'],6):>12}{int(r['n_rank_mismatch']):>18}")
        worst = CHK["max_abs_score_dev"].max()
        if worst < 1e-6:
            SIG.append("[D10] Reproduction check PASSED exactly (max deviation < 1e-6 score points): the "
                       "downstream re-scoring is numerically identical to the parent pipeline, so the "
                       "beta sweep below is valid.")
        elif worst < 0.05:
            SIG.append(f"[D10] Reproduction check passed within tolerance (max deviation "
                       f"{fmt(worst,6)} score points), attributable to drug-level max|z| rows outside the "
                       f"scored mask; the beta sweep remains valid.")
        else:
            SIG.append(f"[D10][WARN] Reproduction deviates by up to {fmt(worst,4)} score points — inspect "
                       f"before interpreting the beta sweep.")

    # ---- beta sweep ----------------------------------------------------------
    rows, corr_rows = [], []
    for (m, t) in RISK.keys():
        det = DRV[(DRV.method==m)&(DRV.trait==t)]
        if det.empty: continue
        refr = rescore_from_detail(det, LOGP_BETA_REF).set_index("drug")
        for b in BETA_GRID:
            r = rescore_from_detail(det, b).set_index("drug")
            common = refr.index.intersection(r.index)
            rho = (spearmanr(refr.loc[common,"rank"], r.loc[common,"rank"])[0]
                   if (HAVE_SCIPY and len(common) >= 5) else np.nan)
            nmov = int((refr.loc[common,"rank"] != r.loc[common,"rank"]).sum())
            mx = int((refr.loc[common,"rank"] - r.loc[common,"rank"]).abs().max()) if len(common) else 0
            corr_rows.append({"method": m, "trait": t, "beta": b, "rho_vs_ref": rho,
                              "n_rank_changed": nmov, "max_abs_rank_change": mx,
                              "top1": r.sort_values("rank").index[0] if len(r) else "NA"})
            for d_, rr in r.iterrows():
                rows.append({"method": m, "trait": t, "beta": b, "drug": d_,
                             "score": rr["score"], "rank": rr["rank"]})
    if not corr_rows: return None
    SW = pd.DataFrame(corr_rows); SWL = pd.DataFrame(rows)
    save_table(SW, "D10b_beta_sensitivity_summary.csv", "Rank stability across beta")
    save_table(SWL, "D10c_beta_sensitivity_long.csv", "Score/rank per drug at each beta")
    SUBSEC(f"BETA SENSITIVITY — rank concordance against the reference beta={LOGP_BETA_REF}")
    S(f"{'beta':>7}{'mean rho':>11}{'min rho':>10}{'mean n changed':>16}{'max |Δrank|':>13}"
      f"  top-1 drug (modal)")
    for b, grp in SW.groupby("beta"):
        modal = grp["top1"].mode().iloc[0] if len(grp["top1"].mode()) else "NA"
        S(f"{b:>7}{fmt(grp['rho_vs_ref'].mean()):>11}{fmt(grp['rho_vs_ref'].min()):>10}"
          f"{fmt(grp['n_rank_changed'].mean(),2):>16}{int(grp['max_abs_rank_change'].max()):>13}  {modal}")
    lo = SW[SW["beta"] == 0.0]
    if len(lo):
        SIG.append(f"[D10] Turning the genomic layer OFF entirely (beta=0) changes the ranking by "
                   f"rho={fmt(lo['rho_vs_ref'].mean())} vs the reference beta={LOGP_BETA_REF} "
                   f"(mean {fmt(lo['n_rank_changed'].mean(),1)} drugs re-ranked, max |Δrank|="
                   f"{int(lo['max_abs_rank_change'].max())}) — this bounds the total influence of TWAS "
                   f"on the ordering.")
    hi = SW[SW["beta"] == max(BETA_GRID)]
    if len(hi):
        SIG.append(f"[D10] Even at beta={max(BETA_GRID)} (>3x the reference) the ranking stays at "
                   f"rho={fmt(hi['rho_vs_ref'].mean())} vs the reference — the conclusions are "
                   f"{'robust to' if hi['rho_vs_ref'].mean() > 0.95 else 'sensitive to'} the choice of beta.")
    return SW
BETA_SWEEP = safe(d10, "D10")


# =============================================================================
# D11. LEAVE-ONE-TRAIT-OUT ROBUSTNESS OF THE TOP-K
# =============================================================================
def d11():
    if len(TRAITS) < 3:
        S(""); S("D11 skipped — needs >=3 traits."); return None
    SEC("D11. LEAVE-ONE-TRAIT-OUT ROBUSTNESS OF THE TRANSDIAGNOSTIC TOP-K")
    S("For each method the transdiagnostic mean rank is recomputed with one trait removed at a time;")
    S("a drug's membership frequency in the top-k measures how much any single disease drives it.")
    rows = []
    for m in METHODS:
        piv = LONG[LONG.method==m].pivot_table(index="drug_key", columns="trait", values="risk_rank").dropna(how="any")
        if piv.shape[1] < 3 or piv.shape[0] < TOP_K: continue
        full = set(piv.mean(axis=1).sort_values().index[:TOP_K])
        cnt = {k: 0 for k in piv.index}
        for t in piv.columns:
            sub = piv.drop(columns=[t]).mean(axis=1).sort_values()
            for k in sub.index[:TOP_K]: cnt[k] += 1
        for k, c in cnt.items():
            if c == 0 and k not in full: continue
            rows.append({"method": m, "drug": DRUG_LABEL[k], "in_full_topk": k in full,
                         "loo_topk_count": c, "n_folds": piv.shape[1],
                         "stability": c/piv.shape[1]})
        SUBSEC(f"LEAVE-ONE-TRAIT-OUT — {m}")
        S(f"{'drug':<28}{'in full top-k':>15}{'LOO top-k hits':>16}{'stability':>11}")
        for r in sorted([x for x in rows if x["method"] == m],
                        key=lambda x: (-x["loo_topk_count"], x["drug"])):
            S(f"{str(r['drug'])[:27]:<28}{str(r['in_full_topk']):>15}"
              f"{str(r['loo_topk_count'])+'/'+str(r['n_folds']):>16}{fmt(r['stability'],2):>11}")
        frag = [r for r in rows if r["method"] == m and r["in_full_topk"] and r["stability"] < 1.0]
        if frag:
            SIG.append(f"[D11] {m}: top-{TOP_K} membership is trait-dependent for "
                       + ", ".join(f"{r['drug']} ({r['loo_topk_count']}/{r['n_folds']} folds)" for r in frag)
                       + " — driven by a single disease rather than transdiagnostically.")
        else:
            SIG.append(f"[D11] {m}: the transdiagnostic top-{TOP_K} is completely robust to removing any "
                       f"single trait.")
    if rows:
        save_table(pd.DataFrame(rows), "D11_leave_one_trait_out.csv", "LOO top-k stability")
    return rows
safe(d11, "D11")


# =============================================================================
# D12. PER-TRAIT RANK PROFILES AND HETEROGENEITY
# =============================================================================
def d12():
    SEC("D12. PER-TRAIT RANK PROFILES AND HETEROGENEITY FLAGS")
    out = []
    for m in METHODS:
        piv = LONG[LONG.method==m].pivot_table(index="drug_key", columns="trait", values="risk_rank").dropna(how="any")
        if piv.empty: continue
        zz = piv.sub(piv.mean(axis=1), axis=0).div(piv.std(axis=1).replace(0, np.nan), axis=0)
        SUBSEC(f"RANK PROFILE — {m}   (1 = highest predicted metabolic risk)")
        S(f"{'drug':<26}" + "".join(f"{str(c)[:11]:>12}" for c in piv.columns)
          + f"{'mean':>8}{'SD':>7}{'range':>7}  most implicated")
        for dk in piv.mean(axis=1).sort_values().index:
            row = piv.loc[dk]
            S(f"{DRUG_LABEL[dk][:25]:<26}" + "".join(f"{int(row[c]):>12}" for c in piv.columns)
              + f"{fmt(row.mean(),1):>8}{fmt(row.std(),2):>7}"
              + f"{int(row.max()-row.min()):>7}  {row.idxmin()}")
            out.append({"method": m, "drug": DRUG_LABEL[dk], "mean_rank": row.mean(),
                        "sd_rank": row.std(), "rank_range": int(row.max()-row.min()),
                        "most_implicated_trait": row.idxmin(), "least_implicated_trait": row.idxmax()})
        het = [o for o in out if o["method"] == m and o["rank_range"] >= 3]
        for o in sorted(het, key=lambda x: -x["rank_range"])[:5]:
            SIG.append(f"[D12] {m}: {o['drug']} is strongly context-dependent — most implicated in "
                       f"{o['most_implicated_trait']}, least in {o['least_implicated_trait']} "
                       f"(rank range {o['rank_range']} positions).")
    if out:
        save_table(pd.DataFrame(out), "D12_rank_profiles.csv", "Per-trait rank profile per drug")
    return out
safe(d12, "D12")


# =============================================================================
# FIGURES
# =============================================================================
if HAVE_MPL:
    print("Rendering figures ...")
    try:
        if PRIORITY is not None and len(PRIORITY):
            d = PRIORITY.sort_values("mean_delta")
            fig, ax = plt.subplots(figsize=(8.5, 0.32*len(d)+2))
            ax.barh(d["drug"], d["mean_delta"], xerr=d.get("sd_delta", pd.Series(0, index=d.index)).fillna(0),
                    color=["tab:red" if v > 0 else "tab:blue" for v in d["mean_delta"]],
                    alpha=.85, error_kw=dict(lw=.7))
            ax.axvline(0, color="k", lw=.8)
            ax.set_xlabel("mean Δrank (weighted − uniform;  >0 = warrants genomic consideration)")
            ax.set_title("Genomic-consideration priority")
            save_fig(fig, "figD1_genomic_priority.png", "Priority bar chart")

        if MEANRANK and WMETHOD in MEANRANK and UMETHOD in MEANRANK:
            a, b = MEANRANK[WMETHOD], MEANRANK[UMETHOD]
            common = a.index.intersection(b.index)
            fig, ax = plt.subplots(figsize=(6.8, 6.4))
            ax.scatter(a.loc[common], b.loc[common], s=44, c="tab:purple", alpha=.85)
            lim = max(a.loc[common].max(), b.loc[common].max())*1.08 + .5
            ax.plot([0, lim], [0, lim], "k--", lw=.8)
            for k in common:
                ax.annotate(DRUG_LABEL[k][:16], (a[k], b[k]), fontsize=7,
                            xytext=(3, 3), textcoords="offset points")
            ax.set_xlabel("transdiagnostic mean rank — weighted (clinical proxy)")
            ax.set_ylabel("transdiagnostic mean rank — uniform (genomics-influenced)")
            ax.invert_xaxis(); ax.invert_yaxis()
            ax.set_title("Clinical-proxy vs genomics-influenced ranking")
            save_fig(fig, "figD4_rank_rank_scatter.png", "Rank-rank scatter")

        if SSRI_SHIFT is not None and len(SSRI_SHIFT):
            d = SSRI_SHIFT.sort_values("mean_delta")
            fig, ax = plt.subplots(figsize=(7.5, 0.34*len(d)+2))
            ax.barh(d["drug"], d["mean_delta"], xerr=d["sd_delta"].fillna(0),
                    color="tab:orange", alpha=.85, error_kw=dict(lw=.7))
            ax.axvline(0, color="k", lw=.8)
            ax.set_xlabel("mean Δrank under uniform weighting")
            ax.set_title("SSRI / SNRI rank shift when the literature prior is removed")
            save_fig(fig, "figD2_ssri_shift.png", "SSRI/SNRI shift")

        if GENE_CONTRAST is not None and len(GENE_CONTRAST):
            d = GENE_CONTRAST.sort_values("mean_delta")
            fig, ax = plt.subplots(figsize=(7.5, 0.30*len(d)+2))
            ax.barh(d["gene"], d["mean_delta"],
                    color=["tab:green" if v > 0 else "tab:orange" for v in d["mean_delta"]], alpha=.85)
            ax.axvline(0, color="k", lw=.8)
            ax.set_xlabel("Δ pooled share (uniform − weighted, share points)")
            ax.set_title("Receptor drivers gained / lost without the literature prior")
            save_fig(fig, "figD3_gene_share_delta.png", "Gene share delta")

        if BETA_SWEEP is not None and len(BETA_SWEEP):
            g = BETA_SWEEP.groupby("beta")["rho_vs_ref"].agg(["mean","min","max"]).reset_index()
            fig, ax = plt.subplots(figsize=(7, 4.5))
            ax.plot(g["beta"], g["mean"], "o-", color="tab:blue")
            ax.fill_between(g["beta"], g["min"], g["max"], alpha=.2, color="tab:blue")
            ax.axvline(LOGP_BETA_REF, color="tab:red", ls="--", lw=.9, label=f"reference β={LOGP_BETA_REF}")
            ax.set_xlabel("β  (exponential rate on −log10 p)")
            ax.set_ylabel("Spearman rho vs the reference ranking")
            ax.set_title("Sensitivity of the ranking to the TWAS scaling rate")
            ax.legend(fontsize=8)
            save_fig(fig, "figD10_beta_sensitivity.png", "Beta sensitivity")

        if ADAPT_AGG is not None and len(ADAPT_AGG):
            d = ADAPT_AGG.sort_values("mean_delta")
            fig, ax = plt.subplots(figsize=(7.5, 0.30*len(d)+2))
            ax.barh(d["drug"], d["mean_delta"], xerr=d["sd_delta"].fillna(0),
                    color=["tab:red" if bool(v) else "tab:grey" for v in d["adapted"]],
                    alpha=.85, error_kw=dict(lw=.7))
            ax.axvline(0, color="k", lw=.8)
            ax.set_xlabel("mean Δrisk from the chronic 5-HT adaptation (score points)")
            ax.set_title("Chronic 5-HT re-weighting (red = adaptation fired)")
            save_fig(fig, "figD5_5ht_adaptation.png", "5-HT adaptation")

        Rm = RANK_WIDE.dropna(how="any")
        if not Rm.empty:
            Rm = Rm.loc[Rm.mean(axis=1).sort_values().index]
            fig, ax = plt.subplots(figsize=(1.45*Rm.shape[1]+4, 0.34*len(Rm)+2))
            im = ax.imshow(Rm.values, aspect="auto", cmap="YlOrRd_r")
            ax.set_yticks(range(len(Rm))); ax.set_yticklabels([DRUG_LABEL[i] for i in Rm.index], fontsize=8)
            ax.set_xticks(range(Rm.shape[1]))
            ax.set_xticklabels([f"{a}\n{b}" for a, b in Rm.columns], fontsize=7, rotation=45, ha="right")
            for i in range(Rm.shape[0]):
                for j in range(Rm.shape[1]):
                    ax.text(j, i, int(Rm.values[i, j]), ha="center", va="center", fontsize=6)
            plt.colorbar(im, ax=ax, label="rank (1 = highest risk)")
            ax.set_title("Rank across every method x trait run")
            save_fig(fig, "figD0_rank_heatmap.png", "Rank heatmap")
    except Exception as e:
        print(f"  [warn] figure rendering issue: {e}")


# =============================================================================
# FINDINGS, INTERPRETATION, CAVEATS, FILE INDEX, WRITE SUMMARY
# =============================================================================
SEC("Z1. HEADLINE / SIGNIFICANT FINDINGS (auto-collected)")
if SIG:
    seen, ordered = set(), []
    for s_ in SIG:
        if s_ not in seen: seen.add(s_); ordered.append(s_)
    for i, s_ in enumerate(ordered, 1):
        for j, chunk in enumerate(textwrap.wrap(s_, 96)):
            S(f"{str(i)+'.':>4} {chunk}" if j == 0 else f"     {chunk}")
else:
    S("No findings met the reporting thresholds.")

SEC("Z2. HOW TO READ THIS REPORT")
for line in [
 "1. D1 is the operational output: drugs at the top are those whose CLINICAL-PROXY rank under-calls",
 "   the genomic/affinity contribution, i.e. the primary candidates for polygenic / TWAS follow-up.",
 "2. D2 and D5 answer the antidepressant-specific question from two independent directions:",
 "   removing the literature prior (D2) and modelling chronic SERT-driven 5-HT remodelling (D5).",
 "   A drug flagged by BOTH is the strongest case that acute Ki-based scoring is insufficient.",
 "3. D3 supplies the mechanism behind every mover: which receptor gains and which loses share.",
 "4. D4 is the robustness headline for a paper abstract ('X of the top-10 clinical-proxy drugs",
 "   remain top-10 when genomics is allowed to dominate').",
 "5. D8 and D10 together bound the influence of the genomic layer: D8 per drug, D10 globally by",
 "   sweeping beta from 0 (no TWAS at all) upward. If rho stays > 0.95 the conclusions do not",
 "   depend on the exact scaling constant.",
 "6. D9 prioritises receptor genes WITHIN the panel; it is not a genome-wide correction.",
 "7. D11 and D12 separate transdiagnostic signal from single-disease artefacts.",
]: S(line)

SEC("Z3. CAVEATS")
for line in [
 "1. Every score is normalised to amitriptyline = 100 WITHIN its own run; cross-run comparisons in",
 "   this report are therefore made on RANKS, never on raw score differences.",
 "2. 'weighted' embeds a literature receptor->metabolic prior and is partially circular with clinical",
 "   expectation; 'uniform' is non-circular but grants equal a-priori weight to every receptor with Ki",
 "   data, including pharmacologically peripheral ones. Neither is a ground truth.",
 "3. Rank deltas are bounded by the number of drugs, so the amplification index is relative. Sign",
 "   consistency across traits (D1/D2) is a more robust statement than the magnitude.",
 "4. TWAS p-values in D9/D10 are those actually used by the scorer, i.e. after the parent pipeline's",
 "   fairness imputation. Imputed rows are excluded from the significance counts but still enter the",
 "   scaling, by design (they neither gain nor lose relative to the panel mean).",
 "5. Ki values were aggregated with the MINIMUM (highest-affinity) measurement upstream, biasing",
 "   toward the most potent reported binding; imputed Ki rows propagate into every analysis here.",
 "6. The four Zhou traits are genetically correlated, so cross-trait agreement is a consistency check,",
 "   not independent replication; leave-one-trait-out (D11) is the honest robustness statement.",
 "7. The 5-HT adaptation coefficients are expert-prior modelling weights, not measured fold-changes.",
 "   D5 quantifies their consequence, it does not validate them.",
 "8. Drug-class assignment (D7) is regex-based on ATC substance names and may mis-file rare agents.",
 "9. No multiplicity correction is applied across the many per-drug tests; prioritise by consistency",
 "   of direction and effect size rather than by nominal p-value alone.",
]: S(line)

if NOTES:
    SEC("Z4. RUNTIME NOTES / NON-FATAL ERRORS")
    for n in NOTES: S(f"  - {n}")

SEC("Z5. FILE INDEX")
S(f"{'path':<84}  description"); S(SUB)
for p, note in FILES_OUT: S(f"{p:<84}  {note}")

S(""); S(BAR); S("END OF DOWNSTREAM SUMMARY"); S(BAR)

summary_path = os.path.join(OUT_DIR, "DOWNSTREAM_SUMMARY.txt")
with open(summary_path, "w", encoding="utf-8") as fh:
    fh.write("\n".join(SUMMARY))
FILES_OUT.append((summary_path, "MASTER detailed downstream summary"))

payload = {"generated": str(datetime.datetime.now()), "pipeline_root": PIPELINE_ROOT,
           "output_dir": OUT_DIR, "risk_source": LONG_SOURCE, "contribution_source": DRV_SOURCE,
           "methods": METHODS, "traits": TRAITS, "n_runs": len(RISK),
           "n_drugs_union": int(RANK_WIDE.shape[0]), "n_drugs_complete": int(COMPLETE.shape[0]),
           "beta_grid": BETA_GRID, "significant_findings": SIG, "runtime_notes": NOTES,
           "files": [p for p, _ in FILES_OUT]}
json_path = os.path.join(OUT_DIR, "downstream_results.json")
with open(json_path, "w", encoding="utf-8") as fh:
    json.dump(payload, fh, indent=2, default=str)

print("\n" + "="*100)
print("DOWNSTREAM DONE.")
print(f"  Summary : {summary_path}")
print(f"  JSON    : {json_path}")
print(f"  Tables  : {TABLES}  ({len(glob.glob(os.path.join(TABLES,'*.csv')))} files)")
print(f"  Figures : {FIGS}   ({len(glob.glob(os.path.join(FIGS,'*.png')))} files)")
print("="*100)
print("\n--- first 150 lines of DOWNSTREAM_SUMMARY.txt ---\n")
print("\n".join(SUMMARY[:150]))

if COPY_TO_DRIVE:
    try:
        os.makedirs(DRIVE_DEST, exist_ok=True)
        dest = os.path.join(DRIVE_DEST, os.path.basename(OUT_DIR))
        if os.path.exists(dest): shutil.rmtree(dest)
        shutil.copytree(OUT_DIR, dest)
        print(f"\nCopied {OUT_DIR} -> {dest}")
    except Exception as e:
        print(f"\n[warn] could not copy to Drive: {e}")

Loading pipeline artefacts ...
   methods=['weighted', 'uniform', 'weighted_5ht_adapted']
   traits =['zhou_CKD', 'zhou_ESSHP', 'zhou_T2D', 'zhou_obesity']
   runs   =12   drugs(union)=41  complete=41
Rendering figures ...

DOWNSTREAM DONE.
  Summary : /content/pipeline_output/n06a_downstream_v3/DOWNSTREAM_SUMMARY.txt
  JSON    : /content/pipeline_output/n06a_downstream_v3/downstream_results.json
  Tables  : /content/pipeline_output/n06a_downstream_v3/tables  (19 files)
  Figures : /content/pipeline_output/n06a_downstream_v3/figures   (7 files)

--- first 150 lines of DOWNSTREAM_SUMMARY.txt ---

DOWNSTREAM ANALYSES OF THE N06A ANTIDEPRESSANT Ki -> DDD -> TWAS RISK PIPELINE (080KvD-v3)
Operates EXCLUSIVELY on artefacts written by the parent pipeline.
weighted = CLINICAL-PROXY ranking | uniform = GENOMICS-INFLUENCED ranking |
weighted_5ht_adapted = chronic SERT -> 5-HT2C/2A up, 5-HT1A down remodelling
Generated              : 2026-08-09 15:05:50
Pipeline root          : /content/pipeline

# Stage 3

In [ ]:
# =============================================================================
# 082_STAGE3_SYNTHESIS — Stage-3 analyses on 080KvD-v3 OUTPUTS ONLY
# -----------------------------------------------------------------------------
# INPUTS (results only; nothing recomputed from Ki DB / raw TWAS):
#   <ROOT>/stage3_dual/tables/00_master_long_risk_table.csv        [preferred]
#   <ROOT>/stage3_dual/tables/S1a_rank_delta_long.csv              [optional]
#   <ROOT>/stage3_dual/tables/S1b_genomic_amplification_index.csv  [optional]
#   <ROOT>/stage3_dual/tables/S1c/S1d 5-HT adaptation tables       [optional]
#   <ROOT>/stage3_dual/tables/04_receptor_contributions_all_runs.csv
#   <ROOT>/stage3_dual/tables/S5_residuals_*.csv                   [optional]
#   <ROOT>/stage3_dual/tables/S10_final_transdiagnostic_classification.csv
#   <ROOT>/<method>/<trait>/*.csv                                  [fallback]
#   <DOWNSTREAM>/tables/*.csv                                      [optional]
#
# ANALYSES
#   T1  Refined genomic-amplification PRIORITY SCORE (magnitude x consistency x tier)
#   T2  Class-stratified amplification test (SSRI/SNRI vs rest; MWU, Fisher, perm, Cliff's d)
#   T3  Receptor-mechanism test: which gene's share predicts amplification (all genes, BH-FDR)
#   T4  Top-k retention / genomic entrants / prior-dependent exits (k = 5,10,15)
#   T5  Amplification x cross-trait stability interaction
#   T6  Convergent evidence: amplification x chronic 5-HT adaptation
#   T7  Literature triangulation: does genomics move drugs TOWARD the clinical ordering?
#   T8  Trait heterogeneity of the amplification signal (Friedman, Kendall's W)
#   T9  Integrated Stage-3 tiering (final actionable table)
#   T10 Manuscript-ready statements (auto-generated with numbers)
#   -> figures + STAGE3_SUMMARY.txt (very detailed) + stage3_results.json
# Run as ONE cell.
# =============================================================================

import os, re, glob, json, math, textwrap, datetime, itertools, warnings, shutil
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

try:
    from scipy.stats import (spearmanr, kendalltau, mannwhitneyu, wilcoxon, kruskal,
                             fisher_exact, hypergeom, friedmanchisquare, chi2)
    HAVE_SCIPY = True
except Exception as _e:
    HAVE_SCIPY = False
    print(f"[warn] scipy unavailable ({_e}) — inferential statistics limited.")

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    HAVE_MPL = True
except Exception as _e:
    HAVE_MPL = False
    print(f"[warn] matplotlib unavailable ({_e}) — figures skipped.")


# =============================================================================
# CONFIG — EDIT PATHS
# =============================================================================
PIPELINE_ROOT   = "/content/pipeline_output/n06a_logp_expo"
S3_IN           = os.path.join(PIPELINE_ROOT, "stage3_dual", "tables")
DOWNSTREAM_IN   = "/content/pipeline_output/n06a_downstream_v3/tables"   # optional
OUT_DIR         = "/content/pipeline_output/n06a_stage3_synthesis"

COPY_TO_DRIVE = False
DRIVE_DEST    = "/content/drive/MyDrive/Dr Uccello/00_Studies/080_GHS_CVS"

WMETHOD, UMETHOD, AMETHOD = "weighted", "uniform", "weighted_5ht_adapted"

TOP_K        = 10
K_GRID       = [5, 10, 15]
ALPHA        = 0.05
FDR_ALPHA    = 0.05
N_PERM       = 20000
RANDOM_SEED  = 42
np.random.seed(RANDOM_SEED)

# ---- priority-score weights (must sum to 1.0) -------------------------------
W_MAGNITUDE  = 0.45     # percentile of mean Δrank (weighted - uniform)
W_CONSISTENCY= 0.30     # fraction of traits in which the drug rises
W_RISKTIER   = 0.25     # inverse percentile of pooled mean rank (higher risk -> higher)

MECH_GENE_OF_INTEREST = "SLC6A2"    # NET — the gene D3 flagged as the dominant gainer

METABOLIC_WEIGHTS = {
    "HRH1":1.00,"HTR2C":0.72,"CHRM3":0.68,"HTR2A":0.45,"ADRA1A":0.42,"ADRA1B":0.40,
    "HTR6":0.38,"ADRA2A":0.28,"ADRA2B":0.26,"ADRA2C":0.26,"CHRM1":0.25,"CHRM4":0.20,
    "CHRM5":0.18,"HTR7":0.22,"DRD3":0.15,"DRD2":0.10,"DRD4":0.08,"HTR1A":0.07,
    "ADRB1":0.07,"ADRB2":0.06,"SIGMAR1":0.04,"SLC6A4":0.08,"SLC6A2":0.05,"SLC6A3":0.04,
}

CLASS_PATTERNS = [
    ("SSRI",  r"sertraline|fluoxetine|paroxetine|citalopram|escitalopram|fluvoxamine|zimeldine|alaproclate"),
    ("SNRI",  r"venlafaxine|desvenlafaxine|duloxetine|milnacipran|levomilnacipran|sibutramine"),
    ("TCA",   r"amitriptyl|nortriptyl|imipramin|desipramin|clomipramin|doxepin|trimipramin|"
              r"dosulepin|dothiepin|lofepramin|protriptyl|amoxapin|butriptyl|iprindol|opipramol|"
              r"melitracen|dibenzepin|quinupramin|noxiptilin|amineptin|tianeptin|maprotilin"),
    ("MAOI",  r"phenelzin|tranylcypromin|isocarboxazid|moclobemid|iproniazid|nialamid|"
              r"toloxaton|selegilin|safinamid"),
    ("ATYP",  r"mirtazapin|mianserin|trazodon|nefazodon|bupropion|amfebutamon|agomelatin|"
              r"reboxetin|vortioxetin|vilazodon|esketamin|ketamin|setiptilin|viloxazin|"
              r"tandospiron|hyperici|oxitriptan|tryptophan|medifoxamin|minaprin|bifemelan"),
]
SSRI_SNRI_REGEX = CLASS_PATTERNS[0][1] + "|" + CLASS_PATTERNS[1][1] + "|vortioxetine|vilazodone"

# Fallback ordinal literature ranking (1 = highest weight-gain liability); used ONLY
# if the parent S5 residual tables are absent. Orthogonal reference, not part of the model.
LITERATURE_WEIGHT_GAIN_RANK = {
    "amitriptyline":1,"mirtazapine":2,"doxepin":3,"imipramine":4,"trimipramine":5,
    "nortriptyline":6,"clomipramine":7,"paroxetine":8,"mianserin":9,"amoxapine":10,
    "maprotiline":11,"dosulepin":5,"desipramine":9,"phenelzine":8,"tranylcypromine":14,
    "isocarboxazid":12,"citalopram":12,"escitalopram":13,"sertraline":14,"fluvoxamine":15,
    "venlafaxine":16,"duloxetine":17,"milnacipran":18,"vortioxetine":18,"fluoxetine":19,
    "bupropion":21,"amfebutamone":21,"agomelatine":20,"reboxetine":22,"moclobemide":20,
    "trazodone":11,"nefazodone":15,"tianeptine":16,"vilazodone":17,"lofepramine":8,
    "opipramol":7,"protriptyline":10,"melitracen":6,
}

os.makedirs(OUT_DIR, exist_ok=True)
TABLES = os.path.join(OUT_DIR, "tables"); os.makedirs(TABLES, exist_ok=True)
FIGS   = os.path.join(OUT_DIR, "figures"); os.makedirs(FIGS, exist_ok=True)

SUMMARY, SIG, MS, FILES_OUT, NOTES, PROV = [], [], [], [], [], []
BAR, SUB = "=" * 100, "-" * 100


# =============================================================================
# HELPERS
# =============================================================================
def S(x=""):   SUMMARY.append(str(x))
def SEC(t):    S(""); S(BAR); S(str(t).upper()); S(BAR)
def SUBSEC(t): S(""); S(SUB); S(str(t)); S(SUB)

def stars(p):
    if p is None or (isinstance(p, float) and np.isnan(p)): return "n/a"
    return "****" if p < 1e-4 else "***" if p < 1e-3 else "**" if p < 1e-2 else "*" if p < .05 else "ns"

def fmt(x, nd=3):
    try:
        if x is None or (isinstance(x, float) and np.isnan(x)): return "NA"
        if isinstance(x, (bool, np.bool_)): return str(bool(x))
        if isinstance(x, (int, np.integer)): return f"{int(x):,}"
        ax = abs(float(x))
        if ax != 0 and (ax < 1e-3 or ax >= 1e6): return f"{float(x):.3e}"
        return f"{float(x):.{nd}f}"
    except Exception:
        return str(x)

def save_table(df, name, note=""):
    p = os.path.join(TABLES, name); df.to_csv(p, index=False)
    FILES_OUT.append((p, note)); return p

def save_fig(fig, name, note=""):
    if not HAVE_MPL: return None
    p = os.path.join(FIGS, name); fig.savefig(p, dpi=150, bbox_inches="tight")
    plt.close(fig); FILES_OUT.append((p, note)); return p

def read_csv_safe(path):
    if not path or not os.path.exists(path): return None
    for enc in ["utf-8", "utf-8-sig", "latin-1", "cp1252"]:
        try:
            df = pd.read_csv(path, encoding=enc, low_memory=False)
            return df if df is not None and not df.empty else None
        except Exception: continue
    return None

def norm_drug(x): return re.sub(r"\s+", " ", str(x).strip().lower())

def tobool(s):
    return pd.Series(s).map(lambda v: str(v).strip().lower() in {"true","1","1.0","yes","y"})

def drug_class(name):
    n = norm_drug(name)
    for cls, pat in CLASS_PATTERNS:
        if re.search(pat, n): return cls
    return "OTHER"

def bh_fdr(pvals):
    p = np.asarray(pvals, float); ok = ~np.isnan(p)
    q = np.full_like(p, np.nan, float); pv = p[ok]; n = pv.size
    if n == 0: return q
    o = np.argsort(pv); r = pv[o]
    adj = np.clip(np.minimum.accumulate((r*n/(np.arange(n)+1))[::-1])[::-1], 0, 1)
    out = np.empty(n); out[o] = adj; q[ok] = out
    return q

def cliffs_delta(a, b):
    a = np.asarray(a, float); b = np.asarray(b, float)
    if len(a) == 0 or len(b) == 0: return np.nan
    gt = sum((x > b).sum() for x in a); lt = sum((x < b).sum() for x in a)
    return (gt - lt) / (len(a)*len(b))

def cliff_label(d):
    if d is None or (isinstance(d, float) and np.isnan(d)): return "NA"
    a = abs(d)
    return "negligible" if a < .147 else "small" if a < .33 else "medium" if a < .474 else "large"

def perm_mean_diff(a, b, n=N_PERM):
    a = np.asarray(a, float); b = np.asarray(b, float)
    a = a[~np.isnan(a)]; b = b[~np.isnan(b)]
    if len(a) < 2 or len(b) < 2: return np.nan, np.nan
    obs = a.mean() - b.mean(); pool = np.concatenate([a, b]); n1 = len(a)
    cnt = 0
    for _ in range(n):
        np.random.shuffle(pool)
        if abs(pool[:n1].mean() - pool[n1:].mean()) >= abs(obs) - 1e-12: cnt += 1
    return float(obs), float((cnt + 1) / (n + 1))

def kendalls_w(R):
    R = np.asarray(R, float); n, m = R.shape
    if n < 3 or m < 2: return (np.nan,)*4
    Ri = R.sum(axis=1); Sd = ((Ri - Ri.mean())**2).sum()
    W = 12*Sd/(m**2*(n**3-n)); chi = m*(n-1)*W; df = n-1
    p = float(1-chi2.cdf(chi, df)) if HAVE_SCIPY else np.nan
    return float(W), float(chi), int(df), p

def binom_p(k, n, p=0.5):
    if n == 0: return np.nan
    try:
        from scipy.stats import binomtest; return float(binomtest(int(k), int(n), p).pvalue)
    except Exception:
        try:
            from scipy.stats import binom_test; return float(binom_test(int(k), int(n), p))
        except Exception: return np.nan

def safe(fn, label):
    try:
        return fn()
    except Exception as e:
        msg = f"[{label}] FAILED: {type(e).__name__}: {e}"
        print("  " + msg); NOTES.append(msg)
        S(""); S(f"  [!] {label} could not be completed: {type(e).__name__}: {e}")
        return None


# =============================================================================
# LOAD — parent Stage-3 tables preferred, per-run CSVs as fallback
# =============================================================================
print("Loading 080KvD-v3 artefacts ...")
if not os.path.isdir(PIPELINE_ROOT):
    raise FileNotFoundError(f"PIPELINE_ROOT not found: {PIPELINE_ROOT}")

def T(name, root=S3_IN): return read_csv_safe(os.path.join(root, name))

# ---- per-run discovery (fallback + provenance) ------------------------------
def discover_runs(root):
    runs = {}
    for mdir in sorted(glob.glob(os.path.join(root, "*"))):
        if not os.path.isdir(mdir): continue
        method = os.path.basename(mdir)
        if method.startswith("stage3"): continue
        for tdir in sorted(glob.glob(os.path.join(mdir, "*"))):
            if not os.path.isdir(tdir): continue
            trait = os.path.basename(tdir)
            risk = os.path.join(tdir, "metabolic_risk_score_per_drug.csv")
            if not os.path.exists(risk): continue
            runs[(method, trait)] = {
                "risk": risk,
                "detail": (os.path.join(tdir, "receptor_contributions.csv")
                           if os.path.exists(os.path.join(tdir, "receptor_contributions.csv")) else None),
                "adapt": (os.path.join(tdir, "metabolic_risk_5ht_adapted_vs_base.csv")
                          if os.path.exists(os.path.join(tdir, "metabolic_risk_5ht_adapted_vs_base.csv")) else None),
                "mtime": datetime.datetime.fromtimestamp(os.path.getmtime(risk))}
    return runs

RUNS = discover_runs(PIPELINE_ROOT)
if not RUNS:
    raise RuntimeError(f"No <method>/<trait>/metabolic_risk_score_per_drug.csv under {PIPELINE_ROOT}")

# ---- LONG master rank table --------------------------------------------------
LONG = T("00_master_long_risk_table.csv")
if LONG is not None and {"method","trait","drug","risk_rank"}.issubset(LONG.columns):
    LONG_SRC = "stage3 00_master_long_risk_table.csv"
else:
    rows = []
    for (m, t), meta in RUNS.items():
        df = read_csv_safe(meta["risk"])
        if df is None: continue
        for _, r in df.iterrows():
            rows.append({"method": m, "trait": t, "drug": r["drug"],
                         "risk_rank": r.get("risk_rank"),
                         "risk_score": r.get("metabolic_risk_score"),
                         "base_score": r.get("base_score"),
                         "max_abs_twas_z": r.get("max_abs_twas_z"),
                         "n_twas_sig": r.get("n_twas_sig_metabolic")})
    LONG = pd.DataFrame(rows); LONG_SRC = "rebuilt from per-run risk CSVs"
PROV.append(("master long rank table", LONG_SRC))
if "drug_key" not in LONG.columns: LONG["drug_key"] = LONG["drug"].map(norm_drug)
LONG["risk_rank"]  = pd.to_numeric(LONG["risk_rank"], errors="coerce")
LONG["risk_score"] = pd.to_numeric(LONG.get("risk_score"), errors="coerce")
LONG["drug_class"] = LONG["drug"].map(drug_class)

METHODS = [m for m in [WMETHOD, UMETHOD, AMETHOD] if m in set(LONG["method"])] + \
          [m for m in sorted(set(LONG["method"])) if m not in {WMETHOD, UMETHOD, AMETHOD}]
TRAITS  = sorted(LONG["trait"].unique())
DRUG_LABEL = LONG.drop_duplicates("drug_key").set_index("drug_key")["drug"].to_dict()
HAVE_PAIR  = (WMETHOD in METHODS) and (UMETHOD in METHODS)
HAVE_ADAPT = AMETHOD in METHODS

RANK_WIDE = LONG.pivot_table(index="drug_key", columns=["method","trait"], values="risk_rank")
COMPLETE  = RANK_WIDE.dropna(how="any")

def mean_rank_by_method(m):
    piv = LONG[LONG.method == m].pivot_table(index="drug_key", columns="trait",
                                             values="risk_rank").dropna(how="any")
    return piv

PIV = {m: mean_rank_by_method(m) for m in METHODS}
PIV = {m: p for m, p in PIV.items() if not p.empty}

# ---- rank-delta long ---------------------------------------------------------
DELTA = T("S1a_rank_delta_long.csv")
if DELTA is not None:
    dcol = next((c for c in ["rank_delta","delta","Δrank","rank_diff"] if c in DELTA.columns), None)
    if dcol is None: DELTA = None
    else:
        DELTA = DELTA.rename(columns={dcol: "rank_delta"})
        DELTA_SRC = "stage3 S1a_rank_delta_long.csv"
if DELTA is None and HAVE_PAIR:
    parts = []
    for t in TRAITS:
        w = LONG[(LONG.method==WMETHOD)&(LONG.trait==t)][["drug_key","drug","risk_rank","risk_score"]]
        u = LONG[(LONG.method==UMETHOD)&(LONG.trait==t)][["drug_key","risk_rank","risk_score"]]
        mg = w.merge(u, on="drug_key", suffixes=("_w","_u"))
        if mg.empty: continue
        mg["trait"] = t
        mg["rank_delta"] = mg["risk_rank_w"] - mg["risk_rank_u"]
        mg["log2_score_ratio"] = np.log2((mg["risk_score_u"]/mg["risk_score_w"]).replace(0, np.nan))
        parts.append(mg)
    DELTA = pd.concat(parts, ignore_index=True) if parts else None
    DELTA_SRC = "recomputed from the long rank table"
if DELTA is not None:
    if "drug_key" not in DELTA.columns: DELTA["drug_key"] = DELTA["drug"].map(norm_drug)
    DELTA["rank_delta"] = pd.to_numeric(DELTA["rank_delta"], errors="coerce")
    PROV.append(("rank-delta long", DELTA_SRC))

# ---- amplification index -----------------------------------------------------
AMP = T("S1b_genomic_amplification_index.csv")
if AMP is not None and "mean_delta" in AMP.columns:
    AMP_SRC = "stage3 S1b_genomic_amplification_index.csv"
elif DELTA is not None:
    g = DELTA.groupby(["drug_key","drug"])
    AMP = g["rank_delta"].agg(mean_delta="mean", sd_delta="std", min_delta="min",
                              max_delta="max", n_traits="count").reset_index()
    AMP["n_pos"] = g["rank_delta"].apply(lambda x: int((x>0).sum())).values
    AMP["n_neg"] = g["rank_delta"].apply(lambda x: int((x<0).sum())).values
    AMP["sign_test_p"] = [binom_p(max(r.n_pos, r.n_neg), r.n_pos+r.n_neg) for r in AMP.itertuples()]
    AMP["consistent_amplifier"] = (AMP["min_delta"]>0) & (AMP["n_traits"]==len(TRAITS))
    AMP["consistent_dominant"]  = (AMP["max_delta"]<0) & (AMP["n_traits"]==len(TRAITS))
    AMP_SRC = "recomputed from the rank-delta table"
else:
    AMP = None; AMP_SRC = "UNAVAILABLE"
if AMP is not None:
    if "drug_key" not in AMP.columns: AMP["drug_key"] = AMP["drug"].map(norm_drug)
    for c in ["consistent_amplifier","consistent_dominant"]:
        if c in AMP.columns: AMP[c] = tobool(AMP[c]).values
    for c in ["mean_delta","sd_delta","min_delta","max_delta","sign_test_p"]:
        if c in AMP.columns: AMP[c] = pd.to_numeric(AMP[c], errors="coerce")
    AMP["drug_class"] = AMP["drug"].map(drug_class)
PROV.append(("amplification index", AMP_SRC))

# ---- receptor contributions --------------------------------------------------
DRV = T("04_receptor_contributions_all_runs.csv")
DRV_SRC = "stage3 04_receptor_contributions_all_runs.csv"
if DRV is None or "contribution" not in (DRV.columns if DRV is not None else []):
    fr = []
    for (m, t), meta in RUNS.items():
        d = read_csv_safe(meta["detail"])
        if d is None: continue
        d["method"], d["trait"] = m, t; fr.append(d)
    DRV = pd.concat(fr, ignore_index=True) if fr else None
    DRV_SRC = "rebuilt from per-run receptor_contributions.csv"
if DRV is not None:
    DRV["drug_key"] = DRV["drug"].map(norm_drug)
    for c in ["contribution","contribution_no_twas","share_pct","twas_p","twas_z",
              "receptor_weight","twas_scale","affinity_log1p","Ki_nM"]:
        if c in DRV.columns: DRV[c] = pd.to_numeric(DRV[c], errors="coerce")
PROV.append(("receptor contributions", DRV_SRC if DRV is not None else "UNAVAILABLE"))

# ---- 5-HT adaptation ---------------------------------------------------------
ADAPT = T("S1c_5ht_adaptation_long.csv")
ADAPT_SRC = "stage3 S1c_5ht_adaptation_long.csv"
if ADAPT is None:
    fr = []
    for (m, t), meta in RUNS.items():
        if m != AMETHOD or not meta["adapt"]: continue
        c = read_csv_safe(meta["adapt"])
        if c is None: continue
        c["trait"] = t; fr.append(c)
    ADAPT = pd.concat(fr, ignore_index=True) if fr else None
    ADAPT_SRC = "rebuilt from per-run metabolic_risk_5ht_adapted_vs_base.csv"
if ADAPT is not None:
    for c in ["adapted","is_ssri_snri"]:
        if c in ADAPT.columns: ADAPT[c] = tobool(ADAPT[c]).values
    for c in ["base_risk","adapted_risk","adaptation_delta","rank_delta","sert_affinity"]:
        if c in ADAPT.columns: ADAPT[c] = pd.to_numeric(ADAPT[c], errors="coerce")
    ADAPT["drug_key"] = ADAPT["drug"].map(norm_drug)
PROV.append(("5-HT adaptation", ADAPT_SRC if ADAPT is not None else "UNAVAILABLE"))

# ---- residuals vs literature -------------------------------------------------
RESID = {}
for m in METHODS:
    f = T(f"S5_residuals_{re.sub(r'[^A-Za-z0-9]+','_',m)}.csv")
    if f is not None and "mean_resid" in f.columns:
        if "drug_key" not in f.columns: f["drug_key"] = f["drug"].map(norm_drug)
        RESID[m] = f
RESID_SRC = "stage3 S5 residual tables" if RESID else "recomputed from the embedded literature ordering"
if not RESID and HAVE_SCIPY:
    for m in METHODS:
        pts = []
        for t in TRAITS:
            d = LONG[(LONG.method==m)&(LONG.trait==t)].copy()
            d["lit"] = d["drug_key"].map(LITERATURE_WEIGHT_GAIN_RANK)
            d = d.dropna(subset=["lit","risk_rank"])
            if len(d) < 5: continue
            d["residual"] = d["risk_rank"].rank(method="average") - d["lit"].rank(method="average")
            pts.append(d[["drug_key","drug","residual"]])
        if pts:
            R = pd.concat(pts, ignore_index=True)
            RESID[m] = (R.groupby(["drug_key","drug"])["residual"]
                          .agg(mean_resid="mean", sd_resid="std", n="count").reset_index())
PROV.append(("literature residuals", RESID_SRC if RESID else "UNAVAILABLE"))

CLS_IN = T("S10_final_transdiagnostic_classification.csv")
if CLS_IN is not None and "drug_key" not in CLS_IN.columns and "drug" in CLS_IN.columns:
    CLS_IN["drug_key"] = CLS_IN["drug"].map(norm_drug)
PROV.append(("S10 classification", "stage3 S10 table" if CLS_IN is not None else "absent"))

print(f"   methods={METHODS}\n   traits ={TRAITS}\n   drugs(union)={RANK_WIDE.shape[0]} "
      f"complete={COMPLETE.shape[0]}")


# =============================================================================
# SUMMARY HEADER + PROVENANCE
# =============================================================================
S(BAR)
S("STAGE-3 SYNTHESIS — N06A ANTIDEPRESSANT Ki -> DDD -> TWAS METABOLIC-RISK PIPELINE")
S("Operates EXCLUSIVELY on artefacts written by 080KvD-v3. Nothing is re-scored upstream.")
S("weighted = CLINICAL-PROXY ranking | uniform = GENOMICS-INFLUENCED ranking |")
S("weighted_5ht_adapted = chronic SERT -> 5-HT2C/2A up, 5-HT1A down remodelling")
S("POSITIVE Δrank (weighted - uniform) = the drug WARRANTS GREATER GENOMIC CONSIDERATION.")
S(BAR)
S(f"Generated              : {datetime.datetime.now():%Y-%m-%d %H:%M:%S}")
S(f"Pipeline root          : {PIPELINE_ROOT}")
S(f"Stage-3 input tables   : {S3_IN}")
S(f"Output dir             : {OUT_DIR}")
S(f"Methods                : {len(METHODS)} -> {', '.join(METHODS)}")
S(f"Traits                 : {len(TRAITS)} -> {', '.join(TRAITS)}")
S(f"Drugs (union/complete) : {RANK_WIDE.shape[0]} / {COMPLETE.shape[0]}")
S("")
S("Settings:")
for k, v in [("TOP_K",TOP_K),("K_GRID",K_GRID),("ALPHA",ALPHA),("FDR_ALPHA",FDR_ALPHA),
             ("N_PERM",N_PERM),("W_MAGNITUDE",W_MAGNITUDE),("W_CONSISTENCY",W_CONSISTENCY),
             ("W_RISKTIER",W_RISKTIER),("MECH_GENE_OF_INTEREST",MECH_GENE_OF_INTEREST),
             ("scipy",HAVE_SCIPY),("matplotlib",HAVE_MPL)]:
    S(f"  {k:<24}= {v}")

SEC("A. PROVENANCE OF EVERY INPUT USED")
S(f"{'artefact':<32}  source")
S(SUB)
for a, s_ in PROV: S(f"{a:<32}  {s_}")
S("")
S(f"{'method':<26}{'trait':<16}{'detail':>8}{'adaptCmp':>10}  modified")
S(SUB)
for (m, t), meta in sorted(RUNS.items()):
    S(f"{m:<26}{t:<16}{('yes' if meta['detail'] else 'no'):>8}"
      f"{('yes' if meta['adapt'] else 'no'):>10}  {meta['mtime']:%Y-%m-%d %H:%M}")
S("")
S("Drug-class assignment (regex on ATC substance names):")
for cls, grp in LONG.drop_duplicates("drug_key").groupby("drug_class"):
    S(f"   {cls:<7} (n={len(grp)}): " + ", ".join(sorted(grp['drug'].astype(str))[:14])
      + (" ..." if len(grp) > 14 else ""))


# =============================================================================
# T1. REFINED GENOMIC-AMPLIFICATION PRIORITY SCORE
# =============================================================================
def t1():
    if AMP is None or DELTA is None:
        S(""); S("T1 skipped — amplification / delta tables unavailable."); return None
    SEC("T1. REFINED GENOMIC-AMPLIFICATION PRIORITY SCORE")
    S("priority_score = "
      f"{W_MAGNITUDE}*pct(mean Δrank) + {W_CONSISTENCY}*frac_traits_rising + "
      f"{W_RISKTIER}*(1 - pct(pooled mean rank))")
    S("All three components are in [0,1]; higher = the drug more urgently warrants genomic follow-up.")
    S("Component 1 rewards MAGNITUDE, component 2 rewards CONSISTENCY across the four diseases,")
    S("component 3 rewards ABSOLUTE predicted risk (so a large shift in a trivial drug is discounted).")

    cons = (DELTA.groupby(["drug_key","drug"])["rank_delta"]
              .agg(n_up=lambda x: int((x > 0).sum()),
                   n_dn=lambda x: int((x < 0).sum()),
                   n_traits="count").reset_index())
    cons["frac_up"] = cons["n_up"]/cons["n_traits"]

    pooled = (LONG[LONG.method.isin([WMETHOD, UMETHOD])].dropna(subset=["risk_rank"])
                .groupby("drug_key")["risk_rank"]
                .agg(pooled_mean_rank="mean", pooled_sd_rank="std").reset_index())

    P = (AMP[[c for c in ["drug_key","drug","mean_delta","sd_delta","min_delta","max_delta",
                          "sign_test_p","consistent_amplifier","consistent_dominant"]
              if c in AMP.columns]]
         .merge(cons[["drug_key","frac_up","n_up","n_dn","n_traits"]], on="drug_key", how="left")
         .merge(pooled, on="drug_key", how="left"))
    if CLS_IN is not None:
        keep = [c for c in ["drug_key","genomics_signal","risk_tier","classification"]
                if c in CLS_IN.columns]
        P = P.merge(CLS_IN[keep], on="drug_key", how="left")
    P["drug_class"] = P["drug"].map(drug_class)

    P["pct_magnitude"] = P["mean_delta"].rank(pct=True)
    P["pct_risk"]      = 1 - P["pooled_mean_rank"].rank(pct=True)
    P["priority_score"] = (W_MAGNITUDE*P["pct_magnitude"].fillna(0)
                           + W_CONSISTENCY*P["frac_up"].fillna(0)
                           + W_RISKTIER*P["pct_risk"].fillna(0))
    if "risk_tier" not in P.columns or P["risk_tier"].isna().all():
        nq = min(3, max(2, len(P)//3 or 2))
        P["risk_tier"] = pd.qcut(P["pooled_mean_rank"].rank(method="first"), q=nq,
                                 labels=["HIGH","INTERMEDIATE","LOW"][:nq]).astype(str)
    q = P["priority_score"].quantile([1/3, 2/3]).values
    P["priority_band"] = np.where(P["priority_score"] >= q[1], "A (top tertile)",
                          np.where(P["priority_score"] >= q[0], "B (middle)", "C (low)"))
    P = P.sort_values("priority_score", ascending=False).reset_index(drop=True)
    P.insert(0, "priority_rank", np.arange(1, len(P)+1))
    save_table(P, "T1_refined_priority_score.csv", "Refined genomic-amplification priority")

    S("")
    S(f"{'#':>3}  {'drug':<26}{'cls':<6}{'score':>8}{'meanΔ':>8}{'frac up':>9}{'pooled#':>9}"
      f"{'tier':<14}{'band':<16}{'signP':>9}")
    S(SUB)
    for _, r in P.iterrows():
        S(f"{int(r['priority_rank']):>3}  {str(r['drug'])[:25]:<26}{r['drug_class']:<6}"
          f"{fmt(r['priority_score'],3):>8}{fmt(r['mean_delta'],2):>8}"
          f"{fmt(r['frac_up'],2):>9}{fmt(r['pooled_mean_rank'],1):>9}"
          f"{str(r.get('risk_tier'))[:13]:<14}{str(r['priority_band'])[:15]:<16}"
          f"{fmt(r.get('sign_test_p')):>9}")

    topA = P[P["priority_band"].str.startswith("A")]
    SIG.append("[T1] Top-tertile genomic-consideration priority (band A): "
               + ", ".join(f"{r['drug']} ({fmt(r['priority_score'],2)})" for _, r in topA.head(10).iterrows())
               + ".")
    hard = P[(P["frac_up"] == 1.0) & (P["mean_delta"] > 0)]
    if len(hard):
        SIG.append("[T1] Rise in EVERY trait when the literature prior is removed (unanimous amplifiers): "
                   + ", ".join(f"{r['drug']} (+{fmt(r['mean_delta'],1)} ranks)" for _, r in hard.iterrows())
                   + ".")
    dep = P[(P["frac_up"] == 0.0) & (P["mean_delta"] < 0)]
    if len(dep):
        SIG.append("[T1] Fall in EVERY trait without the prior (fully prior-dependent): "
                   + ", ".join(f"{r['drug']} ({fmt(r['mean_delta'],1)})" for _, r in dep.iterrows()) + ".")
    if len(P):
        MS.append(f"A composite priority score combining the magnitude of the genomic rank shift "
                  f"({int(W_MAGNITUDE*100)}%), its consistency across the four cardiometabolic traits "
                  f"({int(W_CONSISTENCY*100)}%) and absolute predicted risk ({int(W_RISKTIER*100)}%) "
                  f"identified {P.iloc[0]['drug']} as the highest-priority antidepressant for genomic "
                  f"follow-up (score {fmt(P.iloc[0]['priority_score'],2)}; mean Δrank "
                  f"{fmt(P.iloc[0]['mean_delta'],1)}; rising in {int(P.iloc[0]['n_up'])}/"
                  f"{int(P.iloc[0]['n_traits'])} traits).")
    return P
PRIORITY = safe(t1, "T1")


# =============================================================================
# T2. CLASS-STRATIFIED AMPLIFICATION TEST
# =============================================================================
def t2():
    if AMP is None:
        S(""); S("T2 skipped — amplification index unavailable."); return None
    SEC("T2. CLASS-STRATIFIED AMPLIFICATION TEST (SSRI/SNRI vs REST, AND ALL CLASSES)")
    A = AMP.copy()
    A["is_ssri_snri"] = A["drug"].str.contains(SSRI_SNRI_REGEX, case=False, na=False)
    g1 = A.loc[A["is_ssri_snri"], "mean_delta"].dropna()
    g0 = A.loc[~A["is_ssri_snri"], "mean_delta"].dropna()

    u = p_mw = np.nan
    if HAVE_SCIPY and len(g1) >= 2 and len(g0) >= 2:
        try: u, p_mw = mannwhitneyu(g1, g0, alternative="two-sided"); u, p_mw = float(u), float(p_mw)
        except Exception: pass
    d_cl = cliffs_delta(g1.values, g0.values)
    obs, p_perm = perm_mean_diff(g1.values, g0.values)

    tab = pd.crosstab(A["is_ssri_snri"], A["mean_delta"] > 0)
    odds = p_fish = np.nan
    if HAVE_SCIPY and tab.shape == (2, 2):
        try: odds, p_fish = fisher_exact(tab.values); odds, p_fish = float(odds), float(p_fish)
        except Exception: pass

    SUBSEC("SSRI/SNRI vs ALL OTHER ANTIDEPRESSANTS")
    S(f"SSRI/SNRI   : n={len(g1)}  mean Δrank={fmt(g1.mean(),3)}  median={fmt(g1.median(),3)}")
    S(f"All others  : n={len(g0)}  mean Δrank={fmt(g0.mean(),3)}  median={fmt(g0.median(),3)}")
    S(f"Mann-Whitney U = {fmt(u,1)}, p = {fmt(p_mw)} {stars(p_mw)}")
    S(f"Cliff's delta  = {fmt(d_cl,3)} ({cliff_label(d_cl)} effect)")
    S(f"Permutation test on the mean difference ({N_PERM:,} reps): obs Δ = {fmt(obs,3)}, "
      f"p = {fmt(p_perm)} {stars(p_perm)}")
    if tab.shape == (2, 2):
        S(f"Fisher exact on 'rises vs does not rise': OR = {fmt(odds,2)}, p = {fmt(p_fish)} {stars(p_fish)}")
        S(f"   contingency (rows: is_ssri_snri False/True; cols: Δ<=0 / Δ>0) = {tab.values.tolist()}")

    if not np.isnan(p_mw):
        direction = "GREATER" if g1.mean() > g0.mean() else "SMALLER"
        verdict = ("a significantly" if p_mw < ALPHA else "no significantly")
        SIG.append(f"[T2] The SSRI/SNRI class shows {verdict} {direction.lower()} genomic amplification "
                   f"than the remaining antidepressants (mean Δrank {fmt(g1.mean(),2)} vs "
                   f"{fmt(g0.mean(),2)}; Mann-Whitney p={fmt(p_mw)} {stars(p_mw)}; "
                   f"Cliff's d={fmt(d_cl,2)}, {cliff_label(d_cl)}; permutation p={fmt(p_perm)}).")
        MS.append(f"Antidepressants of the SSRI/SNRI class were {'preferentially' if g1.mean()>g0.mean() else 'not preferentially'} "
                  f"re-ranked upward once the literature receptor prior was removed "
                  f"(mean Δrank {fmt(g1.mean(),2)} vs {fmt(g0.mean(),2)} for all other agents; "
                  f"Mann-Whitney p = {fmt(p_mw)}, Cliff's delta = {fmt(d_cl,2)}, "
                  f"permutation p = {fmt(p_perm)}).")

    SUBSEC("ALL DRUG CLASSES")
    S(f"{'class':<8}{'n':>4}{'mean Δ':>10}{'median Δ':>11}{'sd':>8}{'n rising':>10}{'sign p':>10}")
    rows, groups = [], []
    for cls, grp in A.groupby("drug_class"):
        v = grp["mean_delta"].dropna()
        nup = int((v > 0).sum()); ps = binom_p(max(nup, len(v)-nup), len(v))
        S(f"{cls:<8}{len(v):>4}{fmt(v.mean(),3):>10}{fmt(v.median(),3):>11}{fmt(v.std(),3):>8}"
          f"{str(nup)+'/'+str(len(v)):>10}{fmt(ps):>10}")
        rows.append({"drug_class": cls, "n": len(v), "mean_delta": v.mean(),
                     "median_delta": v.median(), "sd_delta": v.std(),
                     "n_rising": nup, "sign_test_p": ps})
        if len(v) >= 3: groups.append((cls, v.values))
    if HAVE_SCIPY and len(groups) >= 2:
        try:
            H, pk = kruskal(*[g for _, g in groups])
            S(f"  Kruskal-Wallis across classes with n>=3: H={fmt(H,2)}, p={fmt(pk)} {stars(pk)}")
            if pk < ALPHA:
                ranked = sorted([(c, np.mean(g)) for c, g in groups], key=lambda x: -x[1])
                SIG.append(f"[T2] Drug CLASS significantly stratifies the genomic-amplification index "
                           f"(Kruskal-Wallis p={fmt(pk)} {stars(pk)}); most amplified class = {ranked[0][0]} "
                           f"(mean Δ {fmt(ranked[0][1],2)}), least = {ranked[-1][0]} "
                           f"(mean Δ {fmt(ranked[-1][1],2)}).")
            else:
                SIG.append(f"[T2] Drug class does not significantly stratify the amplification index "
                           f"(Kruskal-Wallis p={fmt(pk)}).")
        except Exception: pass
    CT = pd.DataFrame(rows)
    save_table(CT, "T2a_class_amplification_summary.csv", "Amplification by drug class")
    save_table(A[["drug","drug_class","is_ssri_snri","mean_delta","sd_delta",
                  "sign_test_p"]].sort_values("mean_delta", ascending=False),
               "T2b_amplification_with_class.csv", "Per-drug amplification with class label")
    return {"class_table": CT, "p_mw": p_mw, "cliffs": d_cl, "p_perm": p_perm,
            "p_fisher": p_fish, "mean_ssri": g1.mean(), "mean_other": g0.mean()}
T2_OUT = safe(t2, "T2")


# =============================================================================
# T3. RECEPTOR-MECHANISM TEST — WHICH GENE'S SHARE PREDICTS AMPLIFICATION?
# =============================================================================
def t3():
    if DRV is None or AMP is None or "share_pct" not in DRV.columns:
        S(""); S("T3 skipped — contribution shares or amplification unavailable."); return None
    SEC("T3. RECEPTOR-MECHANISM TEST — WHICH RECEPTOR EXPLAINS THE AMPLIFICATION?")
    S("For every receptor gene, its mean % share of a drug's total contribution under the")
    S("GENOMICS-INFLUENCED (uniform) ranking is correlated with that drug's amplification index.")
    S("A positive rho means: the more a drug's profile is dominated by that receptor, the more it")
    S("rises once the literature prior is removed -> that receptor is the mechanistic driver.")

    uni = DRV[DRV.method == UMETHOD]
    wgt = DRV[DRV.method == WMETHOD]
    if uni.empty:
        S("No uniform-method contributions found."); return None
    share_u = uni.groupby(["drug_key","gene"])["share_pct"].mean().unstack(fill_value=0.0)
    share_w = (wgt.groupby(["drug_key","gene"])["share_pct"].mean().unstack(fill_value=0.0)
               if not wgt.empty else None)

    a = AMP.set_index("drug_key")["mean_delta"]
    common = share_u.index.intersection(a.index)
    share_u = share_u.loc[common]; y = a.loc[common]

    rows = []
    for g in share_u.columns:
        x = share_u[g]
        if x.std(ddof=0) == 0 or len(x) < 5: continue
        rho, p = (spearmanr(x, y) if HAVE_SCIPY else (np.nan, np.nan))
        dsh = np.nan
        if share_w is not None and g in share_w.columns:
            idx = share_w.index.intersection(common)
            dsh = float(share_u.loc[idx, g].mean() - share_w.loc[idx, g].mean())
        rows.append({"gene": g, "mean_share_uniform": float(x.mean()),
                     "mean_share_weighted": (float(share_w.loc[share_w.index.intersection(common), g].mean())
                                             if (share_w is not None and g in share_w.columns) else np.nan),
                     "delta_share": dsh, "literature_weight": METABOLIC_WEIGHTS.get(g, np.nan),
                     "spearman_rho": rho, "p": p, "n": len(x)})
    if not rows: return None
    M = pd.DataFrame(rows)
    M["q_BH"] = bh_fdr(M["p"].values)
    M = M.sort_values("spearman_rho", ascending=False).reset_index(drop=True)
    save_table(M, "T3a_gene_share_vs_amplification.csv",
               "Correlation of each receptor's uniform share with the amplification index")

    S("")
    S(f"{'gene':<10}{'lit w':>7}{'share u%':>10}{'share w%':>10}{'Δshare':>9}"
      f"{'rho':>9}{'p':>11}{'q(BH)':>11}{'sig':>6}")
    S(SUB)
    for _, r in M.iterrows():
        S(f"{str(r['gene'])[:9]:<10}{fmt(r['literature_weight'],2):>7}"
          f"{fmt(r['mean_share_uniform'],2):>10}{fmt(r['mean_share_weighted'],2):>10}"
          f"{fmt(r['delta_share'],2):>9}{fmt(r['spearman_rho']):>9}{fmt(r['p']):>11}"
          f"{fmt(r['q_BH']):>11}{stars(r['q_BH']):>6}")

    hits = M[(M["q_BH"] < FDR_ALPHA)]
    for _, r in hits.iterrows():
        SIG.append(f"[T3] {r['gene']} share under the genomics-influenced ranking predicts the "
                   f"amplification index (rho={fmt(r['spearman_rho'])}, p={fmt(r['p'])}, "
                   f"q={fmt(r['q_BH'])}; literature weight {fmt(r['literature_weight'],2)}) -> "
                   f"{'mechanistic DRIVER of the upward shift' if r['spearman_rho']>0 else 'associated with prior-dependence'}.")

    # focused test on the gene of interest
    g0 = MECH_GENE_OF_INTEREST
    if g0 in share_u.columns:
        SUBSEC(f"FOCUSED TEST — {g0} (the receptor flagged as the dominant gainer)")
        x = share_u[g0]
        rho, p = (spearmanr(x, y) if HAVE_SCIPY else (np.nan, np.nan))
        S(f"Spearman({g0} uniform share, amplification index): rho={fmt(rho)}, p={fmt(p)} {stars(p)} "
          f"(n={len(x)})")
        tt = pd.DataFrame({"drug": [DRUG_LABEL[i] for i in x.index],
                           "drug_class": [drug_class(DRUG_LABEL[i]) for i in x.index],
                           f"{g0}_share_uniform": x.values,
                           "mean_delta": y.values}).sort_values(f"{g0}_share_uniform", ascending=False)
        save_table(tt, f"T3b_{g0}_share_vs_amplification.csv", f"{g0} share vs amplification")
        S("")
        S(f"{'drug':<26}{'class':<7}{g0+' share %':>14}{'mean Δrank':>12}")
        for _, r in tt.head(12).iterrows():
            S(f"{str(r['drug'])[:25]:<26}{r['drug_class']:<7}"
              f"{fmt(r[f'{g0}_share_uniform'],2):>14}{fmt(r['mean_delta'],2):>12}")
        if not np.isnan(p):
            SIG.append(f"[T3][{g0}] Drugs whose genomics-influenced profile is dominated by {g0} are "
                       f"{'preferentially amplified' if (rho or 0) > 0 else 'not preferentially amplified'} "
                       f"(rho={fmt(rho)}, p={fmt(p)} {stars(p)}, n={len(x)}).")
            MS.append(f"The receptor share attributable to {g0} under the genomics-influenced ranking "
                      f"correlated with the genomic-amplification index (Spearman rho = {fmt(rho)}, "
                      f"p = {fmt(p)}), identifying {g0} as the principal mechanistic driver of the "
                      f"re-ranking.")
    return M
MECH = safe(t3, "T3")


# =============================================================================
# T4. TOP-K RETENTION / GENOMIC ENTRANTS / PRIOR-DEPENDENT EXITS
# =============================================================================
def t4():
    if not (WMETHOD in PIV and UMETHOD in PIV):
        S(""); S("T4 skipped — need both weighted and uniform runs."); return None
    SEC("T4. TOP-K RETENTION: CLINICAL PROXY vs GENOMICS-INFLUENCED")
    mw = PIV[WMETHOD].mean(axis=1).sort_values()
    mu = PIV[UMETHOD].mean(axis=1).sort_values()
    N = len(set(mw.index) | set(mu.index))
    rows = []
    for k in K_GRID:
        A_ = set(mw.index[:k]); B_ = set(mu.index[:k])
        inter, ent, ex = A_ & B_, B_ - A_, A_ - B_
        jac = len(inter)/len(A_ | B_) if (A_ | B_) else np.nan
        ph = float(hypergeom.sf(len(inter)-1, N, k, k)) if (HAVE_SCIPY and N >= k) else np.nan
        rows.append({"k": k, "overlap": len(inter), "jaccard": jac, "hypergeom_p": ph,
                     "retained": "; ".join(sorted(DRUG_LABEL[i] for i in inter)),
                     "genomic_entrants": "; ".join(sorted(DRUG_LABEL[i] for i in ent)),
                     "prior_dependent_exits": "; ".join(sorted(DRUG_LABEL[i] for i in ex))})
        SUBSEC(f"k = {k}")
        S(f"Overlap {len(inter)}/{k}   Jaccard = {fmt(jac,2)}   hypergeometric p = {fmt(ph)} {stars(ph)}")
        S("RETAINED (both)          : " + (", ".join(sorted(DRUG_LABEL[i] for i in inter)) or "none"))
        S("GENOMIC ENTRANTS (uniform only) : " + (", ".join(sorted(DRUG_LABEL[i] for i in ent)) or "none"))
        S("PRIOR-DEPENDENT EXITS (weighted only): " + (", ".join(sorted(DRUG_LABEL[i] for i in ex)) or "none"))
        if k == TOP_K:
            SIG.append(f"[T4] {len(inter)}/{k} of the clinical-proxy top-{k} antidepressants are retained "
                       f"in the genomics-influenced top-{k} (Jaccard {fmt(jac,2)}, hypergeometric "
                       f"p={fmt(ph)}). Genomic entrants: "
                       + (", ".join(sorted(DRUG_LABEL[i] for i in ent)) or "none")
                       + "; prior-dependent exits: "
                       + (", ".join(sorted(DRUG_LABEL[i] for i in ex)) or "none") + ".")
            MS.append(f"{len(inter)} of the top-{k} antidepressants ranked by the clinical-proxy model were "
                      f"retained when receptor weights were made uniform and the genomic layer was allowed "
                      f"to dominate (Jaccard index {fmt(jac,2)}; hypergeometric p = {fmt(ph)}), with "
                      f"{len(ent)} genomic entrant(s) "
                      + (f"({', '.join(sorted(DRUG_LABEL[i] for i in ent))}) " if ent else "")
                      + f"and {len(ex)} prior-dependent exit(s)"
                      + (f" ({', '.join(sorted(DRUG_LABEL[i] for i in ex))})" if ex else "") + ".")
    K = pd.DataFrame(rows)
    save_table(K, "T4a_topk_retention.csv", "Top-k retention across k")

    common = mw.index.intersection(mu.index)
    if HAVE_SCIPY and len(common) >= 5:
        rho, p = spearmanr(mw.loc[common], mu.loc[common])
        tau, pt = kendalltau(mw.loc[common], mu.loc[common])
        S(""); S(f"Global concordance of the transdiagnostic mean ranks (n={len(common)}): "
                 f"Spearman rho={fmt(rho)} (p={fmt(p)} {stars(p)}), Kendall tau={fmt(tau)} "
                 f"(p={fmt(pt)} {stars(pt)})")
        SIG.append(f"[T4] Global rank concordance between the clinical-proxy and genomics-influenced "
                   f"rankings: rho={fmt(rho)}, tau={fmt(tau)} -> "
                   f"{'the prior largely fixes the ordering' if (rho or 0) > 0.9 else 'the weighting choice materially re-orders drugs'}.")
    RR = pd.DataFrame({"drug": [DRUG_LABEL[i] for i in common],
                       "drug_class": [drug_class(DRUG_LABEL[i]) for i in common],
                       "mean_rank_weighted": mw.loc[common].values,
                       "mean_rank_uniform": mu.loc[common].values})
    RR["rank_delta"] = RR["mean_rank_weighted"] - RR["mean_rank_uniform"]
    save_table(RR.sort_values("mean_rank_weighted"), "T4b_mean_rank_both_methods.csv",
               "Transdiagnostic mean rank under both methods")
    return {"k_table": K, "mw": mw, "mu": mu}
T4_OUT = safe(t4, "T4")


# =============================================================================
# T5. AMPLIFICATION x CROSS-TRAIT STABILITY
# =============================================================================
def t5():
    if AMP is None or not (WMETHOD in PIV and UMETHOD in PIV):
        S(""); S("T5 skipped."); return None
    SEC("T5. AMPLIFICATION x CROSS-TRAIT STABILITY INTERACTION")
    S("Does the genomic re-ranking buy amplification at the cost of cross-disease consistency?")
    sw = PIV[WMETHOD].std(axis=1).rename("sd_weighted")
    su = PIV[UMETHOD].std(axis=1).rename("sd_uniform")
    st = pd.concat([sw, su], axis=1).dropna()
    st["sd_diff"] = st["sd_uniform"] - st["sd_weighted"]   # >0 = more volatile without the prior
    st = st.reset_index().merge(AMP[["drug_key","drug","mean_delta","drug_class"]],
                                on="drug_key", how="inner")
    save_table(st.sort_values("mean_delta", ascending=False),
               "T5_amplification_vs_stability.csv", "Amplification vs change in cross-trait volatility")

    rho = p = np.nan
    if HAVE_SCIPY and len(st) >= 5:
        rho, p = spearmanr(st["mean_delta"], st["sd_diff"])
    pw = np.nan
    if HAVE_SCIPY and len(st) >= 6 and st["sd_diff"].abs().sum() > 0:
        try: pw = float(wilcoxon(st["sd_weighted"], st["sd_uniform"])[1])
        except Exception: pass
    S("")
    S(f"Mean cross-trait rank SD — weighted (clinical proxy) : {fmt(st['sd_weighted'].mean(),3)}")
    S(f"Mean cross-trait rank SD — uniform (genomics-driven) : {fmt(st['sd_uniform'].mean(),3)}")
    S(f"Wilcoxon signed-rank (SD_w vs SD_u)                  : p={fmt(pw)} {stars(pw)}")
    S(f"Spearman(amplification, change in volatility)        : rho={fmt(rho)}, p={fmt(p)} {stars(p)}")
    S("")
    S(f"{'drug':<26}{'class':<7}{'meanΔ':>8}{'SD w':>8}{'SD u':>8}{'ΔSD':>8}  more stable under")
    S(SUB)
    for _, r in st.sort_values("mean_delta", ascending=False).iterrows():
        ms = "weighted" if r["sd_diff"] > 0 else ("uniform" if r["sd_diff"] < 0 else "tie")
        S(f"{str(r['drug'])[:25]:<26}{r['drug_class']:<7}{fmt(r['mean_delta'],2):>8}"
          f"{fmt(r['sd_weighted'],2):>8}{fmt(r['sd_uniform'],2):>8}{fmt(r['sd_diff'],2):>8}  {ms}")
    if not np.isnan(pw):
        which = ("the LITERATURE PRIOR stabilises ranks across diseases"
                 if st["sd_weighted"].mean() < st["sd_uniform"].mean()
                 else "the GENOMICS-INFLUENCED ranking is the more stable one")
        SIG.append(f"[T5] Cross-trait rank variability: mean SD {fmt(st['sd_weighted'].mean(),2)} "
                   f"(weighted) vs {fmt(st['sd_uniform'].mean(),2)} (uniform), Wilcoxon p={fmt(pw)} "
                   f"{stars(pw)} -> {which}.")
    if not np.isnan(rho):
        SIG.append(f"[T5] The size of the genomic amplification is "
                   f"{'positively' if (rho or 0) > 0 else 'negatively'} associated with the change in "
                   f"cross-trait volatility (rho={fmt(rho)}, p={fmt(p)} {stars(p)}) -> "
                   f"{'amplified drugs become less consistent across diseases' if (rho or 0) > 0 and (p or 1) < ALPHA else 'amplification does not systematically cost consistency'}.")
    return st
STAB_INT = safe(t5, "T5")


# =============================================================================
# T6. CONVERGENT EVIDENCE — AMPLIFICATION x CHRONIC 5-HT ADAPTATION
# =============================================================================
def t6():
    if AMP is None or ADAPT is None or ADAPT.empty:
        S(""); S("T6 skipped — 5-HT adaptation tables unavailable."); return None
    SEC("T6. CONVERGENT EVIDENCE — GENOMIC AMPLIFICATION x CHRONIC 5-HT ADAPTATION")
    S("Two independent perturbations of the same scoring machinery:")
    S("  (i)  removing the literature receptor prior      -> amplification index (mean Δrank)")
    S("  (ii) modelling chronic SERT-driven remodelling   -> adaptation rank gain")
    S("Drugs flagged by BOTH are the strongest case that acute Ki-only scoring is insufficient.")
    ad = (ADAPT.groupby(["drug_key","drug"])
            .agg(adapt_delta_risk=("adaptation_delta","mean"),
                 adapt_rank_gain=("rank_delta","mean"),
                 sert=("sert_affinity","max"),
                 fired=("adapted","max"), n=("trait","nunique")).reset_index())
    M = AMP[["drug_key","drug","mean_delta","drug_class"]].merge(
            ad.drop(columns=["drug"]), on="drug_key", how="inner")
    M["convergent"] = (M["mean_delta"] > 0) & (M["adapt_rank_gain"] > 0)
    M["divergent"]  = (M["mean_delta"] > 0) != (M["adapt_rank_gain"] > 0)
    M = M.sort_values(["convergent","mean_delta"], ascending=[False, False])
    save_table(M, "T6_convergent_evidence.csv", "Amplification x 5-HT adaptation")

    rho = p = np.nan
    if HAVE_SCIPY and len(M) >= 5:
        rho, p = spearmanr(M["mean_delta"], M["adapt_rank_gain"], nan_policy="omit")
    S("")
    S(f"Spearman(amplification, 5-HT adaptation rank gain): rho={fmt(rho)}, p={fmt(p)} {stars(p)} "
      f"(n={len(M)})")
    S("")
    S(f"{'drug':<26}{'class':<7}{'meanΔ(amp)':>12}{'adaptΔrank':>12}{'Δrisk':>9}{'SERT':>8}"
      f"{'fired':>7}  flag")
    S(SUB)
    for _, r in M.iterrows():
        flag = "CONVERGENT" if r["convergent"] else ("divergent" if r["divergent"] else "")
        S(f"{str(r['drug'])[:25]:<26}{r['drug_class']:<7}{fmt(r['mean_delta'],2):>12}"
          f"{fmt(r['adapt_rank_gain'],2):>12}{fmt(r['adapt_delta_risk'],2):>9}"
          f"{fmt(r['sert'],3):>8}{str(bool(r['fired'])):>7}  {flag}")
    conv = M[M["convergent"]]
    if len(conv):
        SIG.append("[T6] DUAL-EVIDENCE set — drugs whose rank rises BOTH when the literature prior is "
                   "removed AND when chronic 5-HT remodelling is modelled: "
                   + ", ".join(f"{r['drug']} (amp {fmt(r['mean_delta'],1)}, adapt "
                               f"{fmt(r['adapt_rank_gain'],1)})" for _, r in conv.head(10).iterrows()) + ".")
        MS.append(f"{len(conv)} antidepressant(s) were flagged by two independent perturbations of the "
                  f"model — removal of the literature receptor prior and modelling of chronic "
                  f"SERT-driven 5-HT receptor remodelling — providing convergent evidence that their "
                  f"acute Ki-based metabolic liability is under-estimated ("
                  + ", ".join(conv["drug"].head(6).astype(str)) + ").")
    if not np.isnan(p):
        SIG.append(f"[T6] The two perturbations are {'concordant' if (rho or 0) > 0 else 'discordant'} "
                   f"overall (rho={fmt(rho)}, p={fmt(p)} {stars(p)}) -> they capture "
                   f"{'a shared' if (p or 1) < ALPHA else 'largely independent'} axis of under-estimation.")
    return M
CONV = safe(t6, "T6")


# =============================================================================
# T7. LITERATURE TRIANGULATION
# =============================================================================
def t7():
    if AMP is None or not RESID:
        S(""); S("T7 skipped — residual tables unavailable."); return None
    SEC("T7. LITERATURE TRIANGULATION — DOES GENOMICS MOVE DRUGS TOWARD CLINICAL EXPECTATION?")
    S("residual = model rank - ordinal literature weight-gain rank (both re-ranked in the matched set).")
    S("NEGATIVE residual = the model calls the drug HIGHER risk than the literature does.")
    S("|residual| shrinking from weighted to uniform = the genomic layer improves calibration.")
    if WMETHOD not in RESID or UMETHOD not in RESID:
        S("Need residuals for both methods."); return None
    a = RESID[WMETHOD][["drug_key","drug","mean_resid"]].rename(columns={"mean_resid":"resid_w"})
    b = RESID[UMETHOD][["drug_key","mean_resid"]].rename(columns={"mean_resid":"resid_u"})
    R = a.merge(b, on="drug_key", how="inner").merge(
            AMP[["drug_key","mean_delta"]], on="drug_key", how="left")
    R["drug_class"] = R["drug"].map(drug_class)
    R["abs_w"] = R["resid_w"].abs(); R["abs_u"] = R["resid_u"].abs()
    R["calibration_gain"] = R["abs_w"] - R["abs_u"]   # >0 = uniform is better calibrated
    R["verdict"] = np.where(R["calibration_gain"] > 0, "genomics improves calibration",
                     np.where(R["calibration_gain"] < 0, "prior better calibrated", "tie"))
    R = R.sort_values("calibration_gain", ascending=False)
    save_table(R, "T7_literature_triangulation.csv", "Residual calibration, weighted vs uniform")

    pw = np.nan
    if HAVE_SCIPY and len(R) >= 6:
        try: pw = float(wilcoxon(R["abs_w"], R["abs_u"])[1])
        except Exception: pass
    rho = p = np.nan
    if HAVE_SCIPY and R["mean_delta"].notna().sum() >= 5:
        rho, p = spearmanr(R["mean_delta"], R["calibration_gain"], nan_policy="omit")
    S("")
    S(f"Mean |residual| weighted : {fmt(R['abs_w'].mean(),3)}")
    S(f"Mean |residual| uniform  : {fmt(R['abs_u'].mean(),3)}")
    S(f"Wilcoxon signed-rank p   : {fmt(pw)} {stars(pw)}")
    S(f"Spearman(amplification, calibration gain): rho={fmt(rho)}, p={fmt(p)} {stars(p)}")
    S("")
    S(f"{'drug':<26}{'class':<7}{'resid w':>10}{'resid u':>10}{'|w|-|u|':>10}{'meanΔ':>8}  verdict")
    S(SUB)
    for _, r in R.iterrows():
        S(f"{str(r['drug'])[:25]:<26}{r['drug_class']:<7}{fmt(r['resid_w'],2):>10}"
          f"{fmt(r['resid_u'],2):>10}{fmt(r['calibration_gain'],2):>10}"
          f"{fmt(r['mean_delta'],2):>8}  {r['verdict']}")
    better = "genomics-influenced (uniform)" if R["abs_u"].mean() < R["abs_w"].mean() else "clinical-proxy (weighted)"
    SIG.append(f"[T7] Calibration against the ordinal literature weight-gain ordering is better under the "
               f"{better} ranking (mean |residual| {fmt(R['abs_w'].mean(),2)} weighted vs "
               f"{fmt(R['abs_u'].mean(),2)} uniform; Wilcoxon p={fmt(pw)} {stars(pw)}).")
    recall = R[(R["calibration_gain"] > 0) & (R["mean_delta"] > 0)]
    if len(recall):
        SIG.append("[T7] GENOMIC RE-CALLS — drugs that are both amplified AND better calibrated to the "
                   "literature once the prior is removed: "
                   + ", ".join(f"{r['drug']} (gain {fmt(r['calibration_gain'],1)})"
                               for _, r in recall.head(8).iterrows()) + ".")
        MS.append(f"For {len(recall)} agent(s) the genomics-influenced ranking simultaneously raised the "
                  f"predicted metabolic rank and reduced the absolute deviation from the published "
                  f"weight-gain ordering, indicating that the re-ranking moves these drugs toward, not "
                  f"away from, clinical expectation.")
    return R
TRIANG = safe(t7, "T7")


# =============================================================================
# T8. TRAIT HETEROGENEITY OF THE AMPLIFICATION SIGNAL
# =============================================================================
def t8():
    if DELTA is None:
        S(""); S("T8 skipped — rank-delta table unavailable."); return None
    SEC("T8. TRAIT HETEROGENEITY OF THE AMPLIFICATION SIGNAL")
    piv = DELTA.pivot_table(index="drug_key", columns="trait", values="rank_delta").dropna(how="any")
    if piv.shape[0] < 3 or piv.shape[1] < 2:
        S("Insufficient complete data."); return None
    S("Per-trait magnitude of the weighted -> uniform shift, and whether the four diseases agree")
    S("on WHICH drugs move (Friedman on the delta matrix; Kendall's W on the delta ranks).")
    S("")
    S(f"{'trait':<16}{'n':>5}{'mean Δ':>10}{'mean |Δ|':>11}{'max |Δ|':>10}{'n moved':>9}{'n rising':>10}")
    rows = []
    for t in piv.columns:
        v = piv[t]
        S(f"{t:<16}{len(v):>5}{fmt(v.mean(),2):>10}{fmt(v.abs().mean(),2):>11}"
          f"{fmt(v.abs().max(),0):>10}{int((v != 0).sum()):>9}{int((v > 0).sum()):>10}")
        rows.append({"trait": t, "n": len(v), "mean_delta": v.mean(),
                     "mean_abs_delta": v.abs().mean(), "max_abs_delta": v.abs().max(),
                     "n_moved": int((v != 0).sum()), "n_rising": int((v > 0).sum())})
    TT = pd.DataFrame(rows).sort_values("mean_abs_delta")
    save_table(TT, "T8a_trait_heterogeneity.csv", "Amplification magnitude per trait")

    fp = np.nan
    if HAVE_SCIPY and piv.shape[1] >= 3:
        try: fp = float(friedmanchisquare(*[piv[c].values for c in piv.columns])[1])
        except Exception: pass
    Rk = piv.rank(axis=0)
    W, chi, dfree, pW = kendalls_w(Rk.values)
    S("")
    S(f"Friedman test across traits on the delta matrix : p={fmt(fp)} {stars(fp)}")
    S(f"Kendall's W on the per-trait delta rankings     : W={fmt(W)}, chi2={fmt(chi,1)}, "
      f"df={dfree}, p={fmt(pW)} {stars(pW)}")
    if not np.isnan(W):
        lab = "very high" if W > .9 else "high" if W > .75 else "moderate" if W > .5 else "low"
        S(f"  -> agreement between the four diseases about WHICH drugs move is {lab.upper()}.")
        SIG.append(f"[T8] The four cardiometabolic traits show {lab} agreement about which drugs are "
                   f"genomically amplified (Kendall's W={fmt(W)}, p={fmt(pW)} {stars(pW)}) -> the "
                   f"amplification signal is {'largely transdiagnostic' if W > .6 else 'substantially trait-specific'}.")
    if len(TT):
        SIG.append(f"[T8] The weighting scheme matters LEAST for {TT.iloc[0]['trait']} "
                   f"(mean |Δrank| {fmt(TT.iloc[0]['mean_abs_delta'],2)}) and MOST for "
                   f"{TT.iloc[-1]['trait']} ({fmt(TT.iloc[-1]['mean_abs_delta'],2)}).")

    # per-drug heterogeneity
    het = pd.DataFrame({"drug": [DRUG_LABEL[i] for i in piv.index],
                        "mean_delta": piv.mean(axis=1).values,
                        "sd_delta": piv.std(axis=1).values,
                        "range": (piv.max(axis=1)-piv.min(axis=1)).values,
                        "max_trait": piv.idxmax(axis=1).values,
                        "min_trait": piv.idxmin(axis=1).values}).sort_values("sd_delta", ascending=False)
    save_table(het, "T8b_per_drug_delta_heterogeneity.csv", "Per-drug heterogeneity of the shift")
    SUBSEC("PER-DRUG HETEROGENEITY OF THE SHIFT (highest SD first)")
    S(f"{'drug':<26}{'mean Δ':>9}{'SD':>8}{'range':>8}  most amplified in / least in")
    for _, r in het.head(12).iterrows():
        S(f"{str(r['drug'])[:25]:<26}{fmt(r['mean_delta'],2):>9}{fmt(r['sd_delta'],2):>8}"
          f"{fmt(r['range'],0):>8}  {r['max_trait']} / {r['min_trait']}")
    for _, r in het[het["sd_delta"] >= 2].head(5).iterrows():
        SIG.append(f"[T8][HETEROGENEOUS] {r['drug']} is amplified in a trait-dependent way "
                   f"(mean Δ {fmt(r['mean_delta'],1)}, SD {fmt(r['sd_delta'],1)}; strongest in "
                   f"{r['max_trait']}, weakest in {r['min_trait']}).")
    return TT
TRAITHET = safe(t8, "T8")


# =============================================================================
# T9. INTEGRATED STAGE-3 TIERING
# =============================================================================
def t9():
    if PRIORITY is None:
        S(""); S("T9 skipped — priority table unavailable."); return None
    SEC("T9. INTEGRATED STAGE-3 TIERING (FINAL ACTIONABLE TABLE)")
    D = PRIORITY.copy()
    if CONV is not None:
        D = D.merge(CONV[["drug_key","adapt_rank_gain","convergent"]], on="drug_key", how="left")
    if TRIANG is not None:
        D = D.merge(TRIANG[["drug_key","calibration_gain"]], on="drug_key", how="left")
    if STAB_INT is not None:
        D = D.merge(STAB_INT[["drug_key","sd_diff"]], on="drug_key", how="left")
    if T4_OUT is not None:
        A_ = set(T4_OUT["mw"].index[:TOP_K]); B_ = set(T4_OUT["mu"].index[:TOP_K])
        D["in_topk_weighted"] = D["drug_key"].isin(A_)
        D["in_topk_uniform"]  = D["drug_key"].isin(B_)
    else:
        D["in_topk_weighted"] = D["in_topk_uniform"] = False

    def tier(r):
        amp_up  = (r.get("mean_delta") or 0) > 0
        unan    = (r.get("frac_up") == 1.0)
        both_tk = bool(r.get("in_topk_weighted")) and bool(r.get("in_topk_uniform"))
        conv    = bool(r.get("convergent")) if pd.notna(r.get("convergent")) else False
        cal     = (r.get("calibration_gain") or 0) > 0
        if both_tk and (unan or conv):
            return "I. CONFIRMED PRIORITY (high risk, amplified, multi-evidence)"
        if both_tk:
            return "II. ROBUST HIGH RISK (weighting-insensitive)"
        if bool(r.get("in_topk_uniform")) and not bool(r.get("in_topk_weighted")):
            return "III. GENOMIC ENTRANT (hypothesis-generating)"
        if bool(r.get("in_topk_weighted")) and not bool(r.get("in_topk_uniform")):
            return "IV. PRIOR-DEPENDENT (verify before acting)"
        if amp_up and (conv or cal):
            return "V. EMERGING SIGNAL (lower absolute rank)"
        return "VI. LOW PRIORITY"
    D["stage3_tier"] = D.apply(tier, axis=1)
    order = {"I":0,"II":1,"III":2,"IV":3,"V":4,"VI":5}
    D["_o"] = D["stage3_tier"].str.split(".").str[0].map(order).fillna(9)
    D = D.sort_values(["_o","priority_score"], ascending=[True, False]).drop(columns="_o")
    cols = [c for c in ["stage3_tier","priority_rank","drug","drug_class","priority_score",
                        "mean_delta","frac_up","sign_test_p","pooled_mean_rank","risk_tier",
                        "adapt_rank_gain","convergent","calibration_gain","sd_diff",
                        "in_topk_weighted","in_topk_uniform","classification"] if c in D.columns]
    save_table(D[cols], "T9_integrated_stage3_tiering.csv", "Final integrated Stage-3 tier table")

    S("Tier logic: I = in the top-k under BOTH rankings AND (unanimous amplification OR convergent")
    S("5-HT evidence); II = top-k under both; III = enters the top-k only without the prior;")
    S("IV = top-k only with the prior; V = amplified with supporting evidence but lower absolute rank.")
    S("")
    S(f"{'drug':<24}{'cls':<6}{'score':>7}{'meanΔ':>8}{'frac↑':>7}"
      + ("{:>9}".format("adaptΔ") if "adapt_rank_gain" in D.columns else "")
      + ("{:>8}".format("calGain") if "calibration_gain" in D.columns else "")
      + "  tier")
    S(SUB)
    for _, r in D.iterrows():
        line = (f"{str(r['drug'])[:23]:<24}{r['drug_class']:<6}{fmt(r['priority_score'],2):>7}"
                f"{fmt(r['mean_delta'],1):>8}{fmt(r['frac_up'],2):>7}")
        if "adapt_rank_gain" in D.columns: line += f"{fmt(r.get('adapt_rank_gain'),1):>9}"
        if "calibration_gain" in D.columns: line += f"{fmt(r.get('calibration_gain'),1):>8}"
        S(line + f"  {r['stage3_tier']}")
    SUBSEC("TIER MEMBERSHIP")
    for tname, grp in D.groupby("stage3_tier", sort=False):
        S(f"  {tname}")
        S("      " + ", ".join(grp["drug"].astype(str).tolist()))
        SIG.append(f"[T9] {tname} -> " + ", ".join(grp["drug"].astype(str).head(8).tolist()) + ".")
    return D
TIERS = safe(t9, "T9")


# =============================================================================
# T10. MANUSCRIPT-READY STATEMENTS
# =============================================================================
def t10():
    SEC("T10. MANUSCRIPT-READY STATEMENTS (auto-generated, numbers filled in)")
    if TIERS is not None:
        t1_ = TIERS[TIERS["stage3_tier"].str.startswith("I.")]
        if len(t1_):
            MS.append("Integrating rank magnitude, cross-trait consistency, absolute predicted risk, "
                      "chronic 5-HT remodelling and literature calibration, "
                      + ", ".join(t1_["drug"].astype(str).tolist())
                      + " were assigned to the highest Stage-3 tier (confirmed priority for genomic "
                        "follow-up).")
        t3_ = TIERS[TIERS["stage3_tier"].str.startswith("III.")]
        if len(t3_):
            MS.append("Conversely, " + ", ".join(t3_["drug"].astype(str).tolist())
                      + " entered the transdiagnostic top-10 only when the literature receptor prior was "
                        "removed, and should be regarded as hypothesis-generating rather than confirmed.")
    if TRAITHET is not None and len(TRAITHET):
        MS.append(f"The influence of the weighting scheme was smallest for {TRAITHET.iloc[0]['trait']} "
                  f"(mean |Δrank| = {fmt(TRAITHET.iloc[0]['mean_abs_delta'],2)}) and largest for "
                  f"{TRAITHET.iloc[-1]['trait']} ({fmt(TRAITHET.iloc[-1]['mean_abs_delta'],2)}), "
                  f"indicating that genomic re-ranking is partly disease-specific.")
    if not MS:
        S("No statements could be generated (upstream sections unavailable)."); return None
    seen, out = set(), []
    for s_ in MS:
        if s_ not in seen: seen.add(s_); out.append(s_)
    for i, s_ in enumerate(out, 1):
        S("")
        for j, chunk in enumerate(textwrap.wrap(s_, 96)):
            S(f"{str(i)+'.':>4} {chunk}" if j == 0 else f"     {chunk}")
    save_table(pd.DataFrame({"n": range(1, len(out)+1), "statement": out}),
               "T10_manuscript_statements.csv", "Auto-generated manuscript sentences")
    return out
safe(t10, "T10")


# =============================================================================
# FIGURES
# =============================================================================
if HAVE_MPL:
    print("Rendering figures ...")
    try:
        if PRIORITY is not None and len(PRIORITY):
            d = PRIORITY.sort_values("priority_score")
            colmap = {"A (top tertile)":"tab:red","B (middle)":"tab:orange","C (low)":"tab:blue"}
            fig, ax = plt.subplots(figsize=(8.5, 0.32*len(d)+2))
            ax.barh(d["drug"], d["priority_score"],
                    color=[colmap.get(b, "grey") for b in d["priority_band"]], alpha=.88)
            ax.set_xlabel("refined genomic-consideration priority score")
            ax.set_title("Stage-3 priority for genomic follow-up")
            save_fig(fig, "figT1_priority_score.png", "Priority score")

        if AMP is not None and len(AMP):
            A = AMP.copy(); A["cls"] = A["drug"].map(drug_class)
            groups = [(c, g["mean_delta"].dropna().values) for c, g in A.groupby("cls") if len(g) >= 1]
            fig, ax = plt.subplots(figsize=(7.5, 5))
            ax.boxplot([g for _, g in groups], labels=[c for c, _ in groups], showfliers=False)
            for i, (_, g) in enumerate(groups, start=1):
                ax.scatter(np.random.normal(i, .06, len(g)), g, s=22, c="k", alpha=.6)
            ax.axhline(0, color="tab:red", ls="--", lw=.8)
            ax.set_ylabel("mean Δrank (weighted − uniform)")
            ax.set_title("Genomic amplification by antidepressant class")
            save_fig(fig, "figT2_class_amplification.png", "Class amplification")

        if MECH is not None and DRV is not None and MECH_GENE_OF_INTEREST in set(DRV["gene"].dropna()):
            g0 = MECH_GENE_OF_INTEREST
            sh = (DRV[(DRV.method == UMETHOD) & (DRV.gene == g0)]
                    .groupby("drug_key")["share_pct"].mean())
            mm = AMP.set_index("drug_key")["mean_delta"]
            ix = sh.index.intersection(mm.index)
            fig, ax = plt.subplots(figsize=(7, 6))
            ax.scatter(sh.loc[ix], mm.loc[ix], s=45, c="tab:green", alpha=.85)
            for i in ix:
                ax.annotate(DRUG_LABEL[i][:15], (sh[i], mm[i]), fontsize=7,
                            xytext=(3, 3), textcoords="offset points")
            ax.axhline(0, color="grey", lw=.6)
            ax.set_xlabel(f"{g0} share of contribution under uniform weights (%)")
            ax.set_ylabel("mean Δrank (amplification index)")
            ax.set_title(f"Mechanistic driver test — {g0}")
            save_fig(fig, f"figT3_{g0}_mechanism.png", "Mechanistic scatter")

        if T4_OUT is not None:
            mw, mu = T4_OUT["mw"], T4_OUT["mu"]
            common = mw.index.intersection(mu.index)
            sel = sorted(common, key=lambda i: mw[i])[:15]
            fig, ax = plt.subplots(figsize=(7.5, 7))
            for i in sel:
                col = ("tab:red" if mu[i] < mw[i] else "tab:blue" if mu[i] > mw[i] else "grey")
                ax.plot([0, 1], [mw[i], mu[i]], "o-", color=col, alpha=.8)
                ax.text(-0.03, mw[i], DRUG_LABEL[i][:20], ha="right", va="center", fontsize=8)
                ax.text(1.03, mu[i], DRUG_LABEL[i][:20], ha="left", va="center", fontsize=8)
            ax.set_xticks([0, 1]); ax.set_xticklabels(["clinical proxy\n(weighted)",
                                                        "genomics-influenced\n(uniform)"], fontsize=9)
            ax.set_xlim(-.6, 1.6); ax.invert_yaxis()
            ax.set_ylabel("transdiagnostic mean rank (1 = highest risk)")
            ax.set_title("Rank migration of the top-15 clinical-proxy antidepressants")
            save_fig(fig, "figT4_rank_migration.png", "Slope chart")

        if STAB_INT is not None and len(STAB_INT):
            fig, ax = plt.subplots(figsize=(7, 6))
            ax.scatter(STAB_INT["mean_delta"], STAB_INT["sd_diff"], s=45, c="tab:purple", alpha=.85)
            for _, r in STAB_INT.iterrows():
                ax.annotate(str(r["drug"])[:15], (r["mean_delta"], r["sd_diff"]), fontsize=7,
                            xytext=(3, 3), textcoords="offset points")
            ax.axhline(0, color="grey", lw=.6); ax.axvline(0, color="grey", lw=.6)
            ax.set_xlabel("mean Δrank (amplification)")
            ax.set_ylabel("Δ cross-trait rank SD (uniform − weighted)")
            ax.set_title("Does amplification cost cross-disease consistency?")
            save_fig(fig, "figT5_amplification_vs_stability.png", "Stability interaction")

        if CONV is not None and len(CONV):
            fig, ax = plt.subplots(figsize=(7, 6))
            ax.scatter(CONV["mean_delta"], CONV["adapt_rank_gain"],
                       s=50, c=["tab:red" if c else "tab:grey" for c in CONV["convergent"]], alpha=.85)
            for _, r in CONV.iterrows():
                ax.annotate(str(r["drug"])[:15], (r["mean_delta"], r["adapt_rank_gain"]),
                            fontsize=7, xytext=(3, 3), textcoords="offset points")
            ax.axhline(0, color="grey", lw=.6); ax.axvline(0, color="grey", lw=.6)
            ax.set_xlabel("genomic amplification (mean Δrank)")
            ax.set_ylabel("chronic 5-HT adaptation rank gain")
            ax.set_title("Convergent evidence quadrant (red = both positive)")
            save_fig(fig, "figT6_convergence.png", "Convergence quadrant")

        if TRAITHET is not None and len(TRAITHET):
            fig, ax = plt.subplots(figsize=(6.5, 4))
            ax.bar(TRAITHET["trait"], TRAITHET["mean_abs_delta"], color="tab:cyan", alpha=.9)
            ax.set_ylabel("mean |Δrank| (weighted − uniform)")
            ax.set_title("How much the weighting scheme matters, by disease")
            plt.setp(ax.get_xticklabels(), rotation=30, ha="right", fontsize=8)
            save_fig(fig, "figT8_trait_heterogeneity.png", "Trait heterogeneity")
    except Exception as e:
        print(f"  [warn] figure rendering issue: {e}")


# =============================================================================
# FINDINGS, GUIDE, CAVEATS, FILE INDEX, WRITE SUMMARY
# =============================================================================
SEC("Z1. HEADLINE / SIGNIFICANT FINDINGS (auto-collected)")
if SIG:
    seen, ordered = set(), []
    for s_ in SIG:
        if s_ not in seen: seen.add(s_); ordered.append(s_)
    for i, s_ in enumerate(ordered, 1):
        for j, chunk in enumerate(textwrap.wrap(s_, 96)):
            S(f"{str(i)+'.':>4} {chunk}" if j == 0 else f"     {chunk}")
else:
    S("No findings met the reporting thresholds.")

SEC("Z2. HOW TO READ THIS REPORT")
for line in [
 "1. T1 is the operational deliverable: a single ordered list of antidepressants by how strongly",
 "   the analysis argues for genomic follow-up. It deliberately blends magnitude, consistency and",
 "   absolute risk so that a large rank shift in a low-risk agent does not dominate.",
 "2. T2 asks whether the SSRI/SNRI class as a whole behaves differently. Three tests are reported",
 "   (Mann-Whitney, permutation on the mean difference, Fisher on the direction) plus Cliff's delta",
 "   as a distribution-free effect size, because n per class is small.",
 "3. T3 is the mechanism: it identifies which receptor's share actually predicts the amplification,",
 "   correcting across the whole receptor panel with BH-FDR rather than testing one gene post hoc.",
 "4. T4 is the abstract-ready robustness statement, evaluated at k = 5, 10 and 15 so the claim does",
 "   not depend on an arbitrary cut-off.",
 "5. T5 checks the price of amplification: if the genomics-influenced ranking is markedly less",
 "   stable across the four diseases, its entrants are weaker candidates.",
 "6. T6 is the strongest antidepressant-specific argument: two independent perturbations of the same",
 "   model (removing the prior; modelling chronic SERT-driven 5-HT remodelling) flagging the same",
 "   drug is much harder to dismiss than either alone.",
 "7. T7 triangulates against an external ordinal literature ordering and asks the sharper question:",
 "   does the genomic layer move drugs TOWARD or AWAY FROM clinical expectation?",
 "8. T8 separates transdiagnostic signal from disease-specific artefact.",
 "9. T9 fuses everything into six actionable tiers; T10 writes the sentences with the numbers in.",
]: S(line)

SEC("Z3. CAVEATS")
for line in [
 "1. All comparisons are made on RANKS. Scores are normalised to amitriptyline = 100 WITHIN each",
 "   run and are not comparable across runs.",
 "2. 'weighted' embeds a literature receptor->metabolic prior and is partially circular with clinical",
 "   expectation; 'uniform' is non-circular but gives equal a-priori weight to every receptor with Ki",
 "   data, including pharmacologically peripheral ones. Neither is ground truth; their DIFFERENCE is",
 "   the quantity of interest.",
 "3. Rank deltas are bounded by the number of drugs, so the amplification index is relative.",
 "   Sign consistency across traits (frac_up in T1) is more robust than the raw magnitude.",
 "4. The priority score in T1 uses fixed weights (0.45 / 0.30 / 0.25). These are an editorial choice;",
 "   re-run with alternative weights before making strong claims about the exact ordering.",
 "5. Class sizes are small (T2). The permutation test and Cliff's delta are reported precisely",
 "   because the asymptotic Mann-Whitney p-value is unreliable at this n.",
 "6. T3 is correlational across drugs, not causal: receptor shares are collinear (a drug dominated by",
 "   one transporter is necessarily not dominated by another), so several genes may appear",
 "   significant as mirror images of the same contrast.",
 "7. The four Zhou traits are genetically correlated; cross-trait agreement (T8) is a consistency",
 "   check, not independent replication.",
 "8. The literature ordering used in T7 is ordinal, incomplete and largely from short-term adult",
 "   trials. A null calibration result is not evidence against the model.",
 "9. The 5-HT adaptation coefficients underlying T6 are expert-prior modelling weights, not measured",
 "   fold-changes; T6 quantifies their consequence, it does not validate them.",
 "10. No multiplicity correction is applied across the many per-drug tests; prioritise by consistency",
 "    of direction and effect size rather than nominal p-values alone.",
]: S(line)

if NOTES:
    SEC("Z4. RUNTIME NOTES / NON-FATAL ERRORS")
    for n in NOTES: S(f"  - {n}")

SEC("Z5. FILE INDEX")
S(f"{'path':<84}  description"); S(SUB)
for p, note in FILES_OUT: S(f"{p:<84}  {note}")

S(""); S(BAR); S("END OF STAGE-3 SUMMARY"); S(BAR)

summary_path = os.path.join(OUT_DIR, "STAGE3_SUMMARY.txt")
with open(summary_path, "w", encoding="utf-8") as fh:
    fh.write("\n".join(SUMMARY))
FILES_OUT.append((summary_path, "MASTER detailed Stage-3 summary"))

payload = {"generated": str(datetime.datetime.now()), "pipeline_root": PIPELINE_ROOT,
           "output_dir": OUT_DIR, "provenance": {a: s_ for a, s_ in PROV},
           "methods": METHODS, "traits": TRAITS,
           "n_drugs_union": int(RANK_WIDE.shape[0]), "n_drugs_complete": int(COMPLETE.shape[0]),
           "priority_weights": {"magnitude": W_MAGNITUDE, "consistency": W_CONSISTENCY,
                                "risk_tier": W_RISKTIER},
           "significant_findings": SIG, "manuscript_statements": MS,
           "runtime_notes": NOTES, "files": [p for p, _ in FILES_OUT]}
json_path = os.path.join(OUT_DIR, "stage3_results.json")
with open(json_path, "w", encoding="utf-8") as fh:
    json.dump(payload, fh, indent=2, default=str)

print("\n" + "="*100)
print("STAGE 3 DONE.")
print(f"  Summary : {summary_path}")
print(f"  JSON    : {json_path}")
print(f"  Tables  : {TABLES}  ({len(glob.glob(os.path.join(TABLES,'*.csv')))} files)")
print(f"  Figures : {FIGS}   ({len(glob.glob(os.path.join(FIGS,'*.png')))} files)")
print("="*100)
print("\n--- first 150 lines of STAGE3_SUMMARY.txt ---\n")
print("\n".join(SUMMARY[:150]))

if COPY_TO_DRIVE:
    try:
        os.makedirs(DRIVE_DEST, exist_ok=True)
        dest = os.path.join(DRIVE_DEST, os.path.basename(OUT_DIR))
        if os.path.exists(dest): shutil.rmtree(dest)
        shutil.copytree(OUT_DIR, dest)
        print(f"\nCopied {OUT_DIR} -> {dest}")
    except Exception as e:
        print(f"\n[warn] could not copy to Drive: {e}")

Loading 080KvD-v3 artefacts ...
   methods=['weighted', 'uniform', 'weighted_5ht_adapted']
   traits =['zhou_CKD', 'zhou_ESSHP', 'zhou_T2D', 'zhou_obesity']
   drugs(union)=41 complete=41
Rendering figures ...

STAGE 3 DONE.
  Summary : /content/pipeline_output/n06a_stage3_synthesis/STAGE3_SUMMARY.txt
  JSON    : /content/pipeline_output/n06a_stage3_synthesis/stage3_results.json
  Tables  : /content/pipeline_output/n06a_stage3_synthesis/tables  (14 files)
  Figures : /content/pipeline_output/n06a_stage3_synthesis/figures   (7 files)

--- first 150 lines of STAGE3_SUMMARY.txt ---

STAGE-3 SYNTHESIS — N06A ANTIDEPRESSANT Ki -> DDD -> TWAS METABOLIC-RISK PIPELINE
Operates EXCLUSIVELY on artefacts written by 080KvD-v3. Nothing is re-scored upstream.
weighted = CLINICAL-PROXY ranking | uniform = GENOMICS-INFLUENCED ranking |
weighted_5ht_adapted = chronic SERT -> 5-HT2C/2A up, 5-HT1A down remodelling
POSITIVE Δrank (weighted - uniform) = the drug WARRANTS GREATER GENOMIC CONSIDERATION.
Gene

In [ ]:
!cp "/content/pipeline_output/n06a_downstream_analyses" "/content/drive/MyDrive/Dr Uccello/00_Studies/080_GHS_CVS" -r

# The End